# I-GDP-PA: MNIST experiment with a centered isometric anchor

This notebook implements the versioned MNIST anchor-design experiment. It is self-contained and intended for Google Colab. The completed Gaussian-anchor experiment remains unchanged in its original Drive folder.

It evaluates all six methods:

1. C-GDP
2. C-GDP-AN
3. I-GDP
4. AA-I-GDP
5. AA-I-GDP-AN
6. PA-I-GDP

The common data reducer remains the public Gaussian random projection at $h=100$. Anchor-based methods use one zero-mean, unit-scale isometric anchor $A=Q$, where $\mathbf 1_r^\top Q=0$ and $Q^\top Q=I_h$. Hence $A^\top A=I_h$, every singular value is one, and the anchor condition number is one. For anchor noise, the reported level is the raw Gaussian standard deviation, $\lambda_a=v=v/\sigma_{\min}(A)$. The notebook defaults to a new validation-only pilot. A full run requires both `RUN_MODE="full"` and `CONFIRM_FULL_RUN=True`.

All images are reduced through one public linear map

$$\widetilde X_i=X_iF,$$

where $F\in\mathbb R^{784\times h}$ is sampled independently of all data with iid standard-normal entries $F_{ab}\sim\mathcal N(0,1)$, matching the supplied reference paper. No participant supplies a mean or fits the projection. Reconstructed reduced rows are decoded by

$$\widehat X_i=\widehat{\widetilde X}_iF^\dagger.$$

Here $h=100$, so every reduced row is reshaped to $10\times10$. Before the unchanged valid-convolution stack, it is symmetrically zero-padded to $16\times16$; padding adds no projected coordinates. The decoder reaches the Gaussian-projection floor $X_iFF^\dagger$. The pilot uses validation data only, calibrates separate private-data and anchor-noise upper bounds, and never loads the official MNIST test split.


## Protocol implementation map

For participant $i$, let $O_i\in\mathcal O(h)$, $\Psi_i\in\mathbb R^h$, and let $A\in\mathbb R^{r\times h}$ be the anchor.

- **C-GDP:** $Y_i=\widetilde X_iO+\mathbf1\Psi^\top$.
- **C-GDP-AN:** $Y_i=\widetilde X_iO+\mathbf1\Psi^\top+E_i$.
- **I-GDP:** $Y_i=\widetilde X_iO_i+\mathbf1\Psi_i^\top$ (no cross-participant alignment).
- **AA-I-GDP:** upload $Y_i$ and $B_i=AO_i+\mathbf1\Psi_i^\top$; solve $R_i$ by centered anchor Procrustes and align into participant 0's coordinates.
- **AA-I-GDP-AN:** add private-data noise before the same exact anchor alignment.
- **PA-I-GDP:** upload the noiseless private representation and a noisy anchor. The generalized orthogonal Procrustes solver estimates the rotations jointly; the aligned private data omit AA's reference-anchor translation term, matching the draft.

Private-data noise is parameterized as $\sigma=\lambda_x s_X$, where $s_X$ is the frozen reduced-data RMS. Anchor noise is parameterized directly by its Gaussian standard deviation $v$; because $\sigma_{\min}(A)=1$, this is also the scale-free ratio $v/\sigma_{\min}(A)$. In the main design, each iteration independently draws $\lambda_x\sim\operatorname{Unif}[0,\lambda_{x,U}]$ and $v\sim\operatorname{Unif}[0,v_U]$. The retained code field `anchor_lambda` stores $v$ exactly.

The full run fixes one outer deployment. The dataset split, participant assignments, public $F$, anchor, participant GDP parameters, auditor, linkage galleries, model initialization, minibatch order, and optimizer seed are frozen. Its 100 draws independently regenerate both scalar noise levels and Gaussian tensors. The analysis is a conditional noise-response study with an inner-draw Monte Carlo bootstrap; outer-seed TOST is not used.

For I-GDP, an unrelated colluder secret is displayed only as a negative control. The primary empirical sensitivity is an unpaired colluder-reference distribution-moment attack; target-secret inversion is shown separately as an oracle outside the threat model.


In [ ]:
#@title 1. Run controls { display-mode: "form" }
RUN_MODE = "pilot" #@param ["smoke", "pilot", "full"]
CONFIRM_FULL_RUN = False #@param {type:"boolean"}
PREFER_A100 = True #@param {type:"boolean"}
REQUIRE_A100 = False #@param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/I-GDP-PA" #@param {type:"string"}
RESULTS_SUBFOLDER = "MNIST_GRP_h100_unit_isometric_anchor_results_v2" #@param {type:"string"}
PILOT_MANIFEST_PATH = "/content/drive/MyDrive/I-GDP-PA/MNIST_GRP_h100_unit_isometric_anchor_results_v2/pilot/frozen_pilot_manifest.json" #@param {type:"string"}

if RUN_MODE == "full" and not CONFIRM_FULL_RUN:
    raise RuntimeError(
        "Full mode is intentionally guarded. Set CONFIRM_FULL_RUN=True only "
        "after inspecting the smoke and pilot artifacts."
    )
print(f"Selected run mode: {RUN_MODE!r}")


In [ ]:
# 2. Environment and persistent output location
from __future__ import annotations

import importlib.util
import os
import platform
import subprocess
import sys
from pathlib import Path

OPTIONAL_PACKAGES = {
    "sklearn": "scikit-learn>=1.4",
    "skimage": "scikit-image>=0.22",
    "scipy": "scipy>=1.11",
    "pandas": "pandas>=2.0",
    "statsmodels": "statsmodels>=0.14",
    "yaml": "PyYAML>=6",
    "tqdm": "tqdm>=4.66",
    "psutil": "psutil>=5.9",
    "matplotlib": "matplotlib>=3.7",
}
missing = [pip_name for module, pip_name in OPTIONAL_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
local_root = Path("/content/igdp_pa_mnist") if IN_COLAB else Path.cwd() / "igdp_pa_mnist"
project_root = Path(DRIVE_PROJECT_PATH)
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    if not project_root.is_dir():
        raise FileNotFoundError(
            f"The verified Drive project folder is absent: {project_root}. "
            "Add the linked I-GDP-PA folder to My Drive (or correct "
            "DRIVE_PROJECT_PATH) rather than creating a lookalike folder."
        )
    OUTPUT_DIR = project_root / RESULTS_SUBFOLDER
else:
    OUTPUT_DIR = local_root / RESULTS_SUBFOLDER
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "in_colab": IN_COLAB,
    "output_dir": str(OUTPUT_DIR),
})


## 3. Embedded implementation

The next three cells materialize the exact implementation bundled with this notebook. Their SHA-256 hashes are recorded in every run manifest.

In [ ]:
%%writefile core_protocols.py
"""Float64 PyTorch implementations of the I-GDP paper protocols and attacks.

Records are row vectors.  Every GDP transform is therefore a right action:
``X @ O`` and ``A @ O``.  The module intentionally keeps model training out of
scope so its algebraic self-tests can run quickly on CPU or CUDA.  The shared
empirical reduction is the reference's uncentered, unnormalized public Gaussian
random projection; reconstruction uses its Moore--Penrose inverse rather than
assuming an orthonormal basis.
"""

from __future__ import annotations

import os
import random
import time
from dataclasses import dataclass
from typing import List, Literal, Optional, Sequence, Tuple, Union

import torch

Tensor = torch.Tensor
TensorOrBlocks = Union[Tensor, Sequence[Tensor]]
PROTOCOL_DTYPE = torch.float64


@dataclass
class GaussianProjectionReducer:
    """Low-level public Gaussian linear reducer.

    ``F_pinv`` is cached because Gaussian projection columns are not
    orthonormal and hence the decoder is not generally ``F.T``.
    """

    F: Tensor
    singular_values: Tensor
    F_pinv: Tensor

    def transform(self, x: Tensor) -> Tensor:
        return _as_tensor(x, self.F.device) @ self.F

    def transform_blocks(self, blocks: Sequence[Tensor]) -> List[Tensor]:
        return [self.transform(x) for x in blocks]

    def decode(self, z: Tensor) -> Tensor:
        return _as_tensor(z, self.F.device) @ self.F_pinv

@dataclass
class GPMDiagnostics:
    """Numerical diagnostics for PA-I-GDP's block-Gram GPM."""

    iterations: int
    converged: bool
    elapsed_seconds: float
    anchor_singular_values: Tensor
    anchor_sigma_min: float
    anchor_sigma_max: float
    anchor_condition: float
    upload_singular_values: List[Tensor]
    gram_eigenvalues: Tensor
    eigengap: float
    relative_eigengap: float
    multistart_objective_gap: float
    gopp_residual: float
    relative_gopp_residual: float
    fixed_point_residual: float
    objective_history: List[float]
    step_history: List[float]


@dataclass
class ProtocolResult:
    """Protocol transcript plus the matrix supplied to the analyst's model."""

    name: str
    private_uploads: List[Tensor]
    collaboration_blocks: List[Tensor]
    collaboration: Tensor
    rotations: List[Tensor]
    translations: List[Tensor]
    anchor_uploads: Optional[List[Tensor]] = None
    alignment_rotations: Optional[List[Tensor]] = None
    private_noises: Optional[List[Tensor]] = None
    anchor_noises: Optional[List[Tensor]] = None
    diagnostics: Optional[GPMDiagnostics] = None
    reference: Optional[int] = None


@dataclass
class AttackResult:
    """Estimated target GDP parameters and reduced-space reconstruction."""

    rotation: Tensor
    translation: Tensor
    reduced: Tensor
    method: str


def seed_everything(seed: int, deterministic: bool = True) -> None:
    """Seed Python, NumPy when present, and PyTorch deterministically."""

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)
        if torch.backends.cudnn.is_available():
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True


def make_generator(seed: int, device: Union[str, torch.device] = "cpu") -> torch.Generator:
    """Return a seeded generator on ``device``."""

    dev = torch.device(device)
    return torch.Generator(device=dev).manual_seed(seed)


def _as_tensor(x: Tensor, device: Optional[torch.device] = None) -> Tensor:
    dev = x.device if device is None and isinstance(x, Tensor) else device
    return torch.as_tensor(x, dtype=PROTOCOL_DTYPE, device=dev)


def _blocks(blocks: Sequence[Tensor], device: Optional[torch.device] = None) -> List[Tensor]:
    if not blocks:
        raise ValueError("At least one participant block is required.")
    dev = torch.device(device) if device is not None else torch.as_tensor(blocks[0]).device
    out = [_as_tensor(x, dev) for x in blocks]
    h = out[0].shape[1]
    if any(x.ndim != 2 or x.shape[1] != h for x in out):
        raise ValueError("All participant blocks must be 2-D with the same feature dimension.")
    return out


def _randn(shape: Tuple[int, ...], ref: Tensor, generator: Optional[torch.Generator]) -> Tensor:
    return torch.randn(shape, dtype=PROTOCOL_DTYPE, device=ref.device, generator=generator)


def haar_orthogonal(
    h: int,
    *,
    generator: Optional[torch.Generator] = None,
    device: Union[str, torch.device] = "cpu",
) -> Tensor:
    """Sample a Haar orthogonal matrix in O(h) by sign-corrected Gaussian QR."""

    z = torch.randn((h, h), dtype=PROTOCOL_DTYPE, device=device, generator=generator)
    q, r = torch.linalg.qr(z)
    signs = torch.where(torch.diagonal(r) >= 0, 1.0, -1.0).to(q)
    return q * signs.unsqueeze(0)


def sample_gdp_parameters(
    num_users: int,
    h: int,
    *,
    translation_scale: float = 1.0,
    generator: Optional[torch.Generator] = None,
    device: Union[str, torch.device] = "cpu",
) -> Tuple[List[Tensor], List[Tensor]]:
    """Sample independent ``(O_i, Psi_i)`` pairs."""

    rotations = [haar_orthogonal(h, generator=generator, device=device) for _ in range(num_users)]
    translations = [
        translation_scale
        * torch.randn((h,), dtype=PROTOCOL_DTYPE, device=device, generator=generator)
        for _ in range(num_users)
    ]
    return rotations, translations


def make_shared_gaussian_projection(
    raw_dimension: int,
    h: int,
    *,
    seed: int,
    device: Union[str, torch.device] = "cpu",
) -> GaussianProjectionReducer:
    """Create the public Gaussian map ``F`` with iid ``N(0, 1)`` entries.

    For row-oriented observations the sole reduction relation is
    ``X_tilde = X @ F``.  A tall Gaussian matrix has full column rank with
    probability one, and its cached decoder is
    ``F^dagger = (F.T @ F)^(-1) @ F.T`` (computed numerically by ``pinv``).
    """

    if raw_dimension < 1:
        raise ValueError("raw_dimension must be positive.")
    if not 1 <= h <= raw_dimension:
        raise ValueError("h must be between 1 and raw_dimension.")
    dev = torch.device(device)
    generator = make_generator(seed, dev)
    F = torch.randn(
        (raw_dimension, h),
        dtype=PROTOCOL_DTYPE,
        device=dev,
        generator=generator,
    )
    singular_values = torch.linalg.svdvals(F)
    tolerance = torch.finfo(PROTOCOL_DTYPE).eps * max(F.shape) * singular_values[0]
    if singular_values[-1] <= tolerance:
        raise RuntimeError("Seeded Gaussian projection was numerically rank deficient.")
    return GaussianProjectionReducer(
        F=F,
        singular_values=singular_values,
        F_pinv=torch.linalg.pinv(F),
    )


def reduce_blocks(blocks: Sequence[Tensor], F: Tensor) -> List[Tensor]:
    """Apply the shared composition ``Xtilde_i = X_i F``."""

    F = _as_tensor(F)
    return [_as_tensor(x, F.device) @ F for x in blocks]


def decode_reduced(z: Tensor, F: Tensor, F_pinv: Optional[Tensor] = None) -> Tensor:
    """Decode a reduced reconstruction as ``Xhat = Xtildehat F^dagger``."""

    F = _as_tensor(F)
    pinv = torch.linalg.pinv(F) if F_pinv is None else _as_tensor(F_pinv, F.device)
    return _as_tensor(z, F.device) @ pinv


def projection_floor(x: Tensor, F: Tensor) -> Tensor:
    """Return the public map's Moore--Penrose linear reconstruction control."""

    F = _as_tensor(F)
    return decode_reduced(_as_tensor(x, F.device) @ F, F)


def orthogonal_polar(matrix: Tensor) -> Tensor:
    """Project a square matrix onto O(h) using ``U V^T``."""

    u, _, vh = torch.linalg.svd(matrix, full_matrices=False)
    return u @ vh


def blockwise_polar(stacked: Tensor, h: int) -> Tensor:
    """Apply the polar projection to each h-by-h vertical block."""

    if stacked.ndim != 2 or stacked.shape[1] != h or stacked.shape[0] % h:
        raise ValueError("Expected a (p*h)-by-h stacked matrix.")
    return torch.cat([orthogonal_polar(b) for b in stacked.split(h, dim=0)], dim=0)


def _parameters(
    p: int,
    h: int,
    ref: Tensor,
    rotations: Optional[Sequence[Tensor]],
    translations: Optional[Sequence[Tensor]],
    generator: Optional[torch.Generator],
    translation_scale: float,
) -> Tuple[List[Tensor], List[Tensor]]:
    if rotations is None or translations is None:
        sampled_o, sampled_p = sample_gdp_parameters(
            p,
            h,
            translation_scale=translation_scale,
            generator=generator,
            device=ref.device,
        )
        rotations = sampled_o if rotations is None else rotations
        translations = sampled_p if translations is None else translations
    if len(rotations) != p or len(translations) != p:
        raise ValueError("One rotation and translation are required per participant.")
    os_ = [_as_tensor(o, ref.device) for o in rotations]
    psis = [_as_tensor(psi, ref.device).reshape(h) for psi in translations]
    eye = torch.eye(h, dtype=PROTOCOL_DTYPE, device=ref.device)
    if any(o.shape != (h, h) or not torch.allclose(o.mT @ o, eye, atol=1e-8, rtol=1e-8) for o in os_):
        raise ValueError("Every O_i must be h-by-h and orthogonal.")
    return os_, psis


def _noise_blocks(
    blocks: Sequence[Tensor],
    scale: float,
    noises: Optional[Sequence[Tensor]],
    generator: Optional[torch.Generator],
) -> List[Tensor]:
    if noises is None:
        return [scale * _randn(tuple(x.shape), x, generator) for x in blocks]
    if len(noises) != len(blocks):
        raise ValueError("One noise matrix is required per participant.")
    out = [_as_tensor(e, x.device) for e, x in zip(noises, blocks)]
    if any(e.shape != x.shape for e, x in zip(out, blocks)):
        raise ValueError("Noise matrices must match their participant block shapes.")
    return out


def _result(
    name: str,
    uploads: List[Tensor],
    blocks: List[Tensor],
    rotations: List[Tensor],
    translations: List[Tensor],
    **kwargs,
) -> ProtocolResult:
    return ProtocolResult(
        name=name,
        private_uploads=uploads,
        collaboration_blocks=blocks,
        collaboration=torch.cat(blocks, dim=0),
        rotations=rotations,
        translations=translations,
        **kwargs,
    )


def c_gdp(
    x_blocks: Sequence[Tensor],
    *,
    rotation: Optional[Tensor] = None,
    translation: Optional[Tensor] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """Common GDP: all users upload ``Xtilde_i O + Psi``."""

    xs = _blocks(x_blocks)
    h = xs[0].shape[1]
    os_, psis = _parameters(1, h, xs[0], None if rotation is None else [rotation], None if translation is None else [translation], generator, translation_scale)
    O, psi = os_[0], psis[0]
    uploads = [x @ O + psi for x in xs]
    return _result("C-GDP", uploads, uploads, [O] * len(xs), [psi] * len(xs))


def c_gdp_an(
    x_blocks: Sequence[Tensor],
    sigma: float,
    *,
    rotation: Optional[Tensor] = None,
    translation: Optional[Tensor] = None,
    noises: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """Common GDP with iid private-upload Gaussian noise."""

    base = c_gdp(
        x_blocks,
        rotation=rotation,
        translation=translation,
        generator=generator,
        translation_scale=translation_scale,
    )
    noise = _noise_blocks(base.private_uploads, sigma, noises, generator)
    uploads = [y + e for y, e in zip(base.private_uploads, noise)]
    return _result("C-GDP-AN", uploads, uploads, base.rotations, base.translations, private_noises=noise)


def i_gdp(
    x_blocks: Sequence[Tensor],
    *,
    rotations: Optional[Sequence[Tensor]] = None,
    translations: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """Independent GDP with no analyst-side alignment."""

    xs = _blocks(x_blocks)
    os_, psis = _parameters(len(xs), xs[0].shape[1], xs[0], rotations, translations, generator, translation_scale)
    uploads = [x @ o + psi for x, o, psi in zip(xs, os_, psis)]
    return _result("I-GDP", uploads, uploads, os_, psis)


def aa_alignment(
    private_uploads: Sequence[Tensor],
    anchor_uploads: Sequence[Tensor],
    reference: int = 0,
) -> Tuple[List[Tensor], List[Tensor], List[Tensor]]:
    """Apply the draft's AA map using ``Bbar_i.T @ Bbar_s``."""

    ys = _blocks(private_uploads)
    bs_upload = _blocks(anchor_uploads, ys[0].device)
    p, h = len(ys), ys[0].shape[1]
    if len(bs_upload) != p or not 0 <= reference < p:
        raise ValueError("Invalid number of anchors or reference index.")
    means = [b.mean(dim=0) for b in bs_upload]
    centered = [b - mean for b, mean in zip(bs_upload, means)]
    rotations: List[Tensor] = []
    eye = torch.eye(h, dtype=PROTOCOL_DTYPE, device=ys[0].device)
    for i in range(p):
        rotations.append(eye if i == reference else orthogonal_polar(centered[i].mT @ centered[reference]))
    ref_mean = means[reference]
    aligned = [(y - mean) @ r + ref_mean for y, mean, r in zip(ys, means, rotations)]
    return aligned, rotations, means


def aa_i_gdp(
    x_blocks: Sequence[Tensor],
    anchor: Tensor,
    *,
    reference: int = 0,
    rotations: Optional[Sequence[Tensor]] = None,
    translations: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """Noiseless anchor-aligned independent GDP."""

    xs = _blocks(x_blocks)
    A = _as_tensor(anchor, xs[0].device)
    p, h = len(xs), xs[0].shape[1]
    if A.ndim != 2 or A.shape[1] != h:
        raise ValueError("anchor must be r-by-h.")
    os_, psis = _parameters(p, h, xs[0], rotations, translations, generator, translation_scale)
    ys = [x @ o + psi for x, o, psi in zip(xs, os_, psis)]
    Bs = [A @ o + psi for o, psi in zip(os_, psis)]
    aligned, rs, _ = aa_alignment(ys, Bs, reference)
    return _result(
        "AA-I-GDP",
        ys,
        aligned,
        os_,
        psis,
        anchor_uploads=Bs,
        alignment_rotations=rs,
        reference=reference,
    )


def aa_i_gdp_an(
    x_blocks: Sequence[Tensor],
    anchor: Tensor,
    sigma: float,
    *,
    reference: int = 0,
    rotations: Optional[Sequence[Tensor]] = None,
    translations: Optional[Sequence[Tensor]] = None,
    noises: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
) -> ProtocolResult:
    """AA-I-GDP with Gaussian noise on private uploads only."""

    base = aa_i_gdp(
        x_blocks,
        anchor,
        reference=reference,
        rotations=rotations,
        translations=translations,
        generator=generator,
        translation_scale=translation_scale,
    )
    noise = _noise_blocks(base.private_uploads, sigma, noises, generator)
    ys = [y + e for y, e in zip(base.private_uploads, noise)]
    aligned, rs, _ = aa_alignment(ys, base.anchor_uploads or [], reference)
    return _result(
        "AA-I-GDP-AN",
        ys,
        aligned,
        base.rotations,
        base.translations,
        anchor_uploads=base.anchor_uploads,
        alignment_rotations=rs,
        private_noises=noise,
        reference=reference,
    )


def build_block_gram(centered_anchor_uploads: Sequence[Tensor]) -> Tensor:
    """Build PA-I-GDP blocks ``G_ij = Bbar_i.T @ Bbar_j``."""

    Bs = _blocks(centered_anchor_uploads)
    rows = [torch.cat([bi.mT @ bj for bj in Bs], dim=1) for bi in Bs]
    G = torch.cat(rows, dim=0)
    return 0.5 * (G + G.mT)


def gpm_align(
    centered_anchor_uploads: Sequence[Tensor],
    *,
    raw_anchor: Optional[Tensor] = None,
    max_iters: int = 200,
    tol: float = 1e-10,
    n_random_starts: int = 0,
    restart_seed: int = 0,
) -> Tuple[List[Tensor], GPMDiagnostics]:
    """Run spectral initialization and blockwise-polar GPM."""

    started = time.perf_counter()
    Bs = _blocks(centered_anchor_uploads)
    p, h = len(Bs), Bs[0].shape[1]
    G = build_block_gram(Bs)
    evals, evecs = torch.linalg.eigh(G)
    stacked = blockwise_polar(evecs[:, -h:], h)

    def objective(r: Tensor) -> float:
        return float(torch.trace(r.mT @ G @ r).detach().cpu())

    objectives = [objective(stacked)]
    steps: List[float] = []
    converged = False
    for _ in range(max_iters):
        nxt = blockwise_polar(G @ stacked, h)
        step = float(torch.linalg.norm(nxt - stacked).detach().cpu())
        steps.append(step)
        stacked = nxt
        objectives.append(objective(stacked))
        if step <= tol:
            converged = True
            break
    rs = list(stacked.split(h, dim=0))
    template = sum((b @ r for b, r in zip(Bs, rs)), torch.zeros_like(Bs[0])) / p
    residual = sum(float(torch.linalg.norm(template - b @ r).square().detach().cpu()) for b, r in zip(Bs, rs))
    denom = sum(float(torch.linalg.norm(b).square().detach().cpu()) for b in Bs)
    fixed = torch.linalg.norm(blockwise_polar(G @ stacked, h) - stacked) / max(1.0, float(torch.linalg.norm(stacked).detach().cpu()))
    gap = float((evals[-h] - evals[-h - 1]).detach().cpu()) if G.shape[0] > h else float("nan")
    lambda_h = abs(float(evals[-h].detach().cpu()))
    top_scale = max(lambda_h, torch.finfo(PROTOCOL_DTYPE).eps)
    spectral_objective = objectives[-1]
    best_restart_objective = spectral_objective
    if n_random_starts < 0:
        raise ValueError("n_random_starts must be nonnegative.")
    if n_random_starts:
        restart_generator = make_generator(restart_seed, Bs[0].device)
        for _ in range(n_random_starts):
            trial = torch.cat(
                [haar_orthogonal(h, generator=restart_generator, device=Bs[0].device) for _ in range(p)],
                dim=0,
            )
            for _ in range(max_iters):
                trial_next = blockwise_polar(G @ trial, h)
                trial_step = float(torch.linalg.norm(trial_next - trial).detach().cpu())
                trial = trial_next
                if trial_step <= tol:
                    break
            best_restart_objective = max(best_restart_objective, objective(trial))
    if raw_anchor is None:
        anchor_svals = torch.empty(0, dtype=PROTOCOL_DTYPE, device=Bs[0].device)
        anchor_sigma_min = anchor_sigma_max = anchor_condition = float("nan")
    else:
        A = _as_tensor(raw_anchor, Bs[0].device)
        anchor_svals = torch.linalg.svdvals(A - A.mean(dim=0))
        anchor_sigma_max = float(anchor_svals[0].detach().cpu())
        anchor_sigma_min = float(anchor_svals[-1].detach().cpu())
        anchor_condition = anchor_sigma_max / anchor_sigma_min if anchor_sigma_min > 0 else float("inf")
    diagnostics = GPMDiagnostics(
        iterations=len(steps),
        converged=converged,
        elapsed_seconds=time.perf_counter() - started,
        anchor_singular_values=anchor_svals,
        anchor_sigma_min=anchor_sigma_min,
        anchor_sigma_max=anchor_sigma_max,
        anchor_condition=anchor_condition,
        upload_singular_values=[torch.linalg.svdvals(b) for b in Bs],
        gram_eigenvalues=evals.flip(0),
        eigengap=gap,
        relative_eigengap=gap / top_scale,
        multistart_objective_gap=max(0.0, best_restart_objective - spectral_objective),
        gopp_residual=residual,
        relative_gopp_residual=residual / max(denom, torch.finfo(PROTOCOL_DTYPE).eps),
        fixed_point_residual=float(fixed.detach().cpu()),
        objective_history=objectives,
        step_history=steps,
    )
    return rs, diagnostics


def pa_i_gdp(
    x_blocks: Sequence[Tensor],
    anchor: Tensor,
    v: float,
    *,
    rotations: Optional[Sequence[Tensor]] = None,
    translations: Optional[Sequence[Tensor]] = None,
    anchor_noises: Optional[Sequence[Tensor]] = None,
    generator: Optional[torch.Generator] = None,
    translation_scale: float = 1.0,
    max_iters: int = 200,
    tol: float = 1e-10,
    n_random_starts: int = 0,
    restart_seed: int = 0,
) -> ProtocolResult:
    """PA-I-GDP with noisy anchors and block-Gram generalized Procrustes."""

    xs = _blocks(x_blocks)
    A = _as_tensor(anchor, xs[0].device)
    p, h = len(xs), xs[0].shape[1]
    if A.ndim != 2 or A.shape[1] != h:
        raise ValueError("anchor must be r-by-h.")
    os_, psis = _parameters(p, h, xs[0], rotations, translations, generator, translation_scale)
    ys = [x @ o + psi for x, o, psi in zip(xs, os_, psis)]
    clean_Bs = [A @ o + psi for o, psi in zip(os_, psis)]
    noise = _noise_blocks(clean_Bs, v, anchor_noises, generator)
    Bs = [b + e for b, e in zip(clean_Bs, noise)]
    means = [b.mean(dim=0) for b in Bs]
    centered = [b - mean for b, mean in zip(Bs, means)]
    rs, diagnostics = gpm_align(
        centered,
        raw_anchor=A,
        max_iters=max_iters,
        tol=tol,
        n_random_starts=n_random_starts,
        restart_seed=restart_seed,
    )
    aligned = [(y - mean) @ r for y, mean, r in zip(ys, means, rs)]
    return _result(
        "PA-I-GDP",
        ys,
        aligned,
        os_,
        psis,
        anchor_uploads=Bs,
        alignment_rotations=rs,
        anchor_noises=noise,
        diagnostics=diagnostics,
    )


def _stack(x: TensorOrBlocks, device: Optional[torch.device] = None) -> Tensor:
    if isinstance(x, Tensor):
        return _as_tensor(x, device)
    return torch.cat(_blocks(x, device), dim=0)


def _center(x: Tensor) -> Tuple[Tensor, Tensor]:
    mean = x.mean(dim=0)
    return mean, x - mean


def attack_c_known_input(
    colluding_x: TensorOrBlocks,
    colluding_y: TensorOrBlocks,
    target_y: Tensor,
    *,
    estimator: Literal["exact", "mp", "op"] = "exact",
) -> AttackResult:
    """Known-input attack against C-GDP; ``mp`` uses a pseudoinverse on decoding."""

    Xq = _stack(colluding_x)
    Yq = _stack(colluding_y, Xq.device)
    if Xq.shape != Yq.shape:
        raise ValueError("Colluding raw/uploaded rows must be paired and shape-compatible.")
    xbar, Xc = _center(Xq)
    ybar, Yc = _center(Yq)
    if estimator == "op":
        O = orthogonal_polar(Xc.mT @ Yc)
    else:
        O = torch.linalg.pinv(Xc) @ Yc
    psi = ybar - O.mT @ xbar
    inverse = torch.linalg.pinv(O) if estimator == "mp" else O.mT
    reduced = (_as_tensor(target_y, Xq.device) - psi) @ inverse
    return AttackResult(O, psi, reduced, f"C-{estimator}")


def invert_known_gdp(target_y: Tensor, rotation: Tensor, translation: Tensor) -> AttackResult:
    """Invert a released GDP matrix when its orthogonal secret is revealed."""

    O = _as_tensor(rotation)
    psi = _as_tensor(translation, O.device)
    return AttackResult(O, psi, (_as_tensor(target_y, O.device) - psi) @ O.mT, "known-secret")


def attack_i_colluder_negative_control(
    target_y: Tensor,
    colluder_rotation: Tensor,
    colluder_translation: Tensor,
) -> AttackResult:
    """Apply a colluder's unrelated I-GDP secret to a noncolluding target."""

    result = invert_known_gdp(target_y, colluder_rotation, colluder_translation)
    result.method = "I-colluder-negative-control"
    return result


def attack_i_distribution_moment(
    colluder_reduced: TensorOrBlocks,
    target_reference_upload: TensorOrBlocks,
    target_y: Tensor,
) -> AttackResult:
    """Unpaired I-GDP distribution-alignment sensitivity attack.

    This attack is not an algebraic inversion theorem.  It uses reduced raw
    records revealed by a colluder as a reference population and estimates a
    non-colluding target's orthogonal transform from the eigensystems of the
    two centered covariance matrices.  Eigenvector signs are resolved with
    standardized third moments, falling back to fifth moments when skewness is
    nearly zero.  It is threat-model compatible for iid participant samples
    but relies on the additional population-matching assumption and must be
    reported as a distribution-prior sensitivity.
    """

    Xq = _stack(colluder_reduced)
    Yref = _stack(target_reference_upload, Xq.device)
    Ytarget = _as_tensor(target_y, Xq.device)
    if Xq.ndim != 2 or Yref.ndim != 2 or Xq.shape[1] != Yref.shape[1]:
        raise ValueError("Reference reduced data and target uploads need the same feature dimension.")
    if Ytarget.ndim != 2 or Ytarget.shape[1] != Xq.shape[1]:
        raise ValueError("Target reconstruction rows need the same feature dimension.")
    if len(Xq) < 2 or len(Yref) < 2:
        raise ValueError("Distribution alignment requires at least two rows per reference sample.")

    xbar, Xc = _center(Xq)
    ybar, Yc = _center(Yref)
    cov_x = Xc.mT @ Xc / max(1, len(Xc) - 1)
    cov_y = Yc.mT @ Yc / max(1, len(Yc) - 1)
    eval_x, Ux = torch.linalg.eigh(cov_x)
    eval_y, Uy = torch.linalg.eigh(cov_y)
    order_x = torch.argsort(eval_x, descending=True)
    order_y = torch.argsort(eval_y, descending=True)
    Ux = Ux[:, order_x]
    Uy = Uy[:, order_y]

    px = Xc @ Ux
    py = Yc @ Uy
    eps = torch.finfo(PROTOCOL_DTYPE).eps**0.5
    px = px / px.square().mean(dim=0).sqrt().clamp_min(eps)
    py = py / py.square().mean(dim=0).sqrt().clamp_min(eps)
    skew_x, skew_y = px.pow(3).mean(dim=0), py.pow(3).mean(dim=0)
    fifth_x, fifth_y = px.pow(5).mean(dim=0), py.pow(5).mean(dim=0)
    use_fifth = (skew_x.abs() + skew_y.abs()) < 1.0e-3
    odd_x = torch.where(use_fifth, fifth_x, skew_x)
    odd_y = torch.where(use_fifth, fifth_y, skew_y)
    signs = torch.where(odd_x * odd_y >= 0, 1.0, -1.0).to(Xq)

    O = Ux @ torch.diag(signs) @ Uy.mT
    psi = ybar - xbar @ O
    reduced = (Ytarget - psi) @ O.mT
    return AttackResult(O, psi, reduced, "I-distribution-moment")


def attack_aa_known_anchor(anchor: Tensor, target_B: Tensor, target_y: Tensor) -> AttackResult:
    """Draft pseudoinverse known-anchor attack against AA-I-GDP or AA-I-GDP-AN."""

    A = _as_tensor(anchor)
    B = _as_tensor(target_B, A.device)
    a, Ac = _center(A)
    b, Bc = _center(B)
    O = torch.linalg.pinv(Ac) @ Bc
    psi = b - O.mT @ a
    reduced = (_as_tensor(target_y, A.device) - psi) @ O.mT
    return AttackResult(O, psi, reduced, "AA-known-anchor")


def attack_pa_mp(anchor: Tensor, target_B: Tensor, target_y: Tensor) -> AttackResult:
    """Unconstrained Moore--Penrose known-anchor attack against PA-I-GDP."""

    A = _as_tensor(anchor)
    B = _as_tensor(target_B, A.device)
    a, Ac = _center(A)
    b, Bc = _center(B)
    O = torch.linalg.pinv(Ac) @ Bc
    psi = b - O.mT @ a
    reduced = (_as_tensor(target_y, A.device) - psi) @ torch.linalg.pinv(O)
    return AttackResult(O, psi, reduced, "PA-MP")


def attack_pa_op(anchor: Tensor, target_B: Tensor, target_y: Tensor) -> AttackResult:
    """Orthogonal-Procrustes known-anchor attack against PA-I-GDP."""

    A = _as_tensor(anchor)
    B = _as_tensor(target_B, A.device)
    a, Ac = _center(A)
    b, Bc = _center(B)
    O = orthogonal_polar(Ac.mT @ Bc)
    psi = b - O.mT @ a
    reduced = (_as_tensor(target_y, A.device) - psi) @ O.mT
    return AttackResult(O, psi, reduced, "PA-OP")


def attack_pa_alignment_calibration(
    anchor: Tensor,
    target_B: Tensor,
    target_y: Tensor,
    target_alignment: Tensor,
    colluder_rotations: Sequence[Tensor],
    colluder_alignments: Sequence[Tensor],
) -> AttackResult:
    """Calibrate PA's global rotation from colluders' revealed local transforms."""

    if not colluder_rotations or len(colluder_rotations) != len(colluder_alignments):
        raise ValueError("Matching nonempty colluder rotation/alignment lists are required.")
    A = _as_tensor(anchor)
    B = _as_tensor(target_B, A.device)
    q_sum = sum(
        (_as_tensor(o, A.device) @ _as_tensor(r, A.device) for o, r in zip(colluder_rotations, colluder_alignments)),
        torch.zeros_like(_as_tensor(colluder_rotations[0], A.device)),
    )
    Q = orthogonal_polar(q_sum)
    Rj = _as_tensor(target_alignment, A.device)
    O = Q @ Rj.mT
    a = A.mean(dim=0)
    b = B.mean(dim=0)
    psi = b - O.mT @ a
    reduced = (_as_tensor(target_y, A.device) - psi) @ O.mT
    return AttackResult(O, psi, reduced, "PA-AM")


def _relative_error(x: Tensor, y: Tensor) -> float:
    num = torch.linalg.norm(x - y)
    den = torch.clamp(torch.linalg.norm(y), min=1.0)
    return float((num / den).detach().cpu())


def affine_orthogonal_registration(
    source: Tensor,
    target: Tensor,
) -> Tuple[Tensor, Tensor, Tensor, float]:
    """Register row embeddings by one right-orthogonal map and common shift."""

    x = _as_tensor(source)
    y = _as_tensor(target, x.device)
    if x.shape != y.shape or x.ndim != 2:
        raise ValueError("source and target must be same-shaped matrices.")
    x_mean = x.mean(dim=0)
    y_mean = y.mean(dim=0)
    rotation = orthogonal_polar((x - x_mean).mT @ (y - y_mean))
    translation = y_mean - x_mean @ rotation
    registered = x @ rotation + translation
    return registered, rotation, translation, _relative_error(registered, y)


def _assert_close(x: Tensor, y: Tensor, name: str, tol: float = 1e-9) -> float:
    error = _relative_error(x, y)
    if error > tol:
        raise AssertionError(f"{name} relative error {error:.3e} exceeds {tol:.3e}.")
    return error


def run_algebraic_self_tests(
    device: Union[str, torch.device] = "cpu",
    seed: int = 20260711,
) -> dict:
    """Run synthetic orientation, attack, GPM, and C/AA equivalence gates."""

    seed_everything(seed)
    dev = torch.device(device)
    gen = make_generator(seed, dev)
    p, raw_d, h, r = 3, 9, 4, 8
    ns = [12, 11, 10]
    raw = [torch.randn((n, raw_d), dtype=PROTOCOL_DTYPE, device=dev, generator=gen) for n in ns]
    reducer = make_shared_gaussian_projection(raw_d, h, seed=seed + 17, device=dev)
    xs = reducer.transform_blocks(raw)

    # A centered, optimally conditioned anchor with A^T A = I.
    g = torch.randn((r, h), dtype=PROTOCOL_DTYPE, device=dev, generator=gen)
    q, r_factor = torch.linalg.qr(g - g.mean(dim=0), mode="reduced")
    diagonal = torch.diagonal(r_factor)
    signs = torch.where(diagonal < 0, -torch.ones_like(diagonal), torch.ones_like(diagonal))
    A = q * signs.unsqueeze(0)
    anchor_svals = torch.linalg.svdvals(A - A.mean(dim=0))
    anchor_rank = int(torch.linalg.matrix_rank(A - A.mean(dim=0)).detach().cpu())
    if anchor_rank != h or float(anchor_svals[-1].detach().cpu()) <= 1e-10:
        raise AssertionError("Centered anchor is not full column rank.")
    os_, psis = sample_gdp_parameters(p, h, generator=gen, device=dev)
    s = 0

    aa = aa_i_gdp(xs, A, reference=s, rotations=os_, translations=psis)
    common = c_gdp(xs, rotation=os_[s], translation=psis[s])
    exact_error = _assert_close(aa.collaboration, common.collaboration, "C/AA equality")
    r_error = max(_relative_error(rr, oi.mT @ os_[s]) for rr, oi in zip(aa.alignment_rotations or [], os_))
    if r_error > 1e-9:
        raise AssertionError(f"AA relative-rotation error {r_error:.3e}.")

    sigma = 0.17
    shared_noise = [sigma * _randn(tuple(x.shape), x, gen) for x in xs]
    coupled_local = [g_i @ rr.mT for g_i, rr in zip(shared_noise, aa.alignment_rotations or [])]
    aa_an = aa_i_gdp_an(
        xs,
        A,
        sigma,
        reference=s,
        rotations=os_,
        translations=psis,
        noises=coupled_local,
    )
    common_an = c_gdp_an(
        xs,
        sigma,
        rotation=os_[s],
        translation=psis[s],
        noises=shared_noise,
    )
    noisy_error = _assert_close(aa_an.collaboration, common_an.collaboration, "coupled C-AN/AA-AN equality")

    c_attack = attack_c_known_input(xs[0], common.private_uploads[0], common.private_uploads[1])
    c_attack_error = _assert_close(c_attack.reduced, xs[1], "C attack")
    aa_attack = attack_aa_known_anchor(A, aa.anchor_uploads[1], aa.private_uploads[1])  # type: ignore[index]
    aa_attack_error = _assert_close(aa_attack.reduced, xs[1], "AA attack")
    aa_an_attack = attack_aa_known_anchor(A, aa_an.anchor_uploads[1], aa_an.private_uploads[1])  # type: ignore[index]
    aa_an_endpoint = xs[1] + coupled_local[1] @ os_[1].mT
    aa_an_endpoint_error = _assert_close(
        aa_an_attack.reduced,
        aa_an_endpoint,
        "AA-AN attack noise endpoint",
    )

    independent = i_gdp(xs, rotations=os_, translations=psis)
    i_negative = attack_i_colluder_negative_control(
        independent.private_uploads[1], os_[0], psis[0]
    )
    i_negative_error = _relative_error(i_negative.reduced, xs[1])
    if i_negative_error < 1e-3:
        raise AssertionError("I-GDP colluder negative control unexpectedly reconstructed the target.")

    pa = pa_i_gdp(
        xs,
        A,
        0.0,
        rotations=os_,
        translations=psis,
        generator=gen,
        max_iters=100,
        tol=1e-12,
        n_random_starts=1,
        restart_seed=seed + 1,
    )
    if pa.diagnostics is None:
        raise AssertionError("PA diagnostics are missing.")
    if not pa.diagnostics.converged:
        raise AssertionError("Noiseless PA GPM failed to converge.")
    if pa.diagnostics.relative_gopp_residual > 1e-10:
        raise AssertionError(
            f"PA relative GOPP residual {pa.diagnostics.relative_gopp_residual:.3e}."
        )
    if pa.diagnostics.fixed_point_residual > 1e-10:
        raise AssertionError(
            f"PA fixed-point residual {pa.diagnostics.fixed_point_residual:.3e}."
        )
    if pa.diagnostics.anchor_sigma_min <= 1e-10:
        raise AssertionError("PA diagnostic reports a rank-deficient centered anchor.")
    effective = [o @ rr for o, rr in zip(os_, pa.alignment_rotations or [])]
    pa_sync_error = max(_relative_error(qi, effective[0]) for qi in effective)
    if pa_sync_error > 1e-8:
        raise AssertionError(f"PA noiseless synchronization error {pa_sync_error:.3e}.")
    _, _, _, pa_c_registration_error = affine_orthogonal_registration(
        pa.collaboration, common.collaboration
    )
    if pa_c_registration_error > 1e-8:
        raise AssertionError(
            f"PA v=0/C affine-orthogonal registration error {pa_c_registration_error:.3e}."
        )
    pa_op = attack_pa_op(A, pa.anchor_uploads[1], pa.private_uploads[1])  # type: ignore[index]
    pa_op_error = _assert_close(pa_op.reduced, xs[1], "PA OP attack", tol=1e-8)
    pa_mp = attack_pa_mp(A, pa.anchor_uploads[1], pa.private_uploads[1])  # type: ignore[index]
    pa_mp_error = _assert_close(pa_mp.reduced, xs[1], "PA MP attack", tol=1e-8)
    pa_am = attack_pa_alignment_calibration(
        A,
        pa.anchor_uploads[1],  # type: ignore[index]
        pa.private_uploads[1],
        pa.alignment_rotations[1],  # type: ignore[index]
        [os_[0]],
        [pa.alignment_rotations[0]],  # type: ignore[index]
    )
    pa_am_error = _assert_close(pa_am.reduced, xs[1], "PA AM attack", tol=1e-8)

    decoded = decode_reduced(xs[1], reducer.F, reducer.F_pinv)
    floor = projection_floor(raw[1], reducer.F)
    decode_error = _assert_close(decoded, floor, "Gaussian projection decode")
    return {
        "c_aa_relative_error": exact_error,
        "c_aa_an_coupled_relative_error": noisy_error,
        "aa_rotation_relative_error": r_error,
        "c_attack_relative_error": c_attack_error,
        "aa_attack_relative_error": aa_attack_error,
        "aa_an_attack_endpoint_relative_error": aa_an_endpoint_error,
        "i_colluder_negative_control_relative_error": i_negative_error,
        "pa_sync_relative_error": pa_sync_error,
        "pa_c_affine_registration_relative_error": pa_c_registration_error,
        "pa_mp_relative_error": pa_mp_error,
        "pa_op_relative_error": pa_op_error,
        "pa_am_relative_error": pa_am_error,
        "grp_decode_relative_error": decode_error,
        "grp_matrix_sigma_min": float(reducer.singular_values[-1].detach().cpu()),
        "grp_matrix_condition": float(
            (reducer.singular_values[0] / reducer.singular_values[-1]).detach().cpu()
        ),
        "centered_anchor_rank": anchor_rank,
        "centered_anchor_sigma_min": float(anchor_svals[-1].detach().cpu()),
        "pa_gopp_residual": pa.diagnostics.gopp_residual if pa.diagnostics else float("nan"),
        "pa_relative_gopp_residual": pa.diagnostics.relative_gopp_residual if pa.diagnostics else float("nan"),
        "pa_fixed_point_residual": pa.diagnostics.fixed_point_residual if pa.diagnostics else float("nan"),
        "pa_converged": pa.diagnostics.converged if pa.diagnostics else False,
    }


def run_self_tests(
    device: Union[str, torch.device] = "cpu",
    seed: int = 20260711,
) -> dict:
    """Run all algebra gates and add a notebook-friendly success flag."""

    metrics = run_algebraic_self_tests(device=device, seed=seed)
    return {"all_passed": True, **metrics}


__all__ = [
    "PROTOCOL_DTYPE",
    "GaussianProjectionReducer",
    "GPMDiagnostics",
    "ProtocolResult",
    "AttackResult",
    "seed_everything",
    "make_generator",
    "haar_orthogonal",
    "sample_gdp_parameters",
    "make_shared_gaussian_projection",
    "reduce_blocks",
    "decode_reduced",
    "projection_floor",
    "orthogonal_polar",
    "blockwise_polar",
    "c_gdp",
    "c_gdp_an",
    "i_gdp",
    "aa_alignment",
    "aa_i_gdp",
    "aa_i_gdp_an",
    "build_block_gram",
    "gpm_align",
    "pa_i_gdp",
    "attack_c_known_input",
    "invert_known_gdp",
    "attack_i_colluder_negative_control",
    "attack_i_distribution_moment",
    "attack_aa_known_anchor",
    "attack_pa_mp",
    "attack_pa_op",
    "attack_pa_alignment_calibration",
    "affine_orthogonal_registration",
    "run_algebraic_self_tests",
    "run_self_tests",
]


In [ ]:
%%writefile mnist_runner.py
"""Colab-ready MNIST experiment utilities for the I-GDP-PA study.

This module deliberately keeps the data/model/statistics layer separate from
``core_protocols.py``.  Protocol functions consume participant blocks in the
public reduced space and return a ``ProtocolResult``; this runner reshapes the
collaboration matrix for the reference CNN, trains/evaluates it, decodes attack
outputs, and persists restart-safe records.

The notebook-facing API is:

``ExperimentConfig.for_mode(mode, **overrides)``
``make_config(mode, **overrides)``
``run_experiment(config, output_dir)``

All random quantities are derived from stable, named seeds.  The test split is
never used by the noise upper-bound pilot or model selection.
"""

from __future__ import annotations

import argparse
import copy
import csv
import dataclasses
import hashlib
import inspect
import json
import math
import os
import random
import tempfile
import time
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, MutableMapping, Optional, Sequence

import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset


NUM_CLASSES = 10
RAW_SIDE = 28
RAW_DIM = RAW_SIDE * RAW_SIDE


def _json_default(value: Any) -> Any:
    if dataclasses.is_dataclass(value):
        return asdict(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Tensor):
        if value.numel() == 1:
            return value.detach().cpu().item()
        return value.detach().cpu().tolist()
    if isinstance(value, tuple):
        return list(value)
    raise TypeError(f"Cannot serialize {type(value)!r}")


def stable_hash(value: Any, length: int = 16) -> str:
    payload = json.dumps(
        value, sort_keys=True, separators=(",", ":"), default=_json_default
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:length]


def named_seed(base_seed: int, *parts: Any) -> int:
    """Return a stable positive 63-bit seed independent of Python hash salt."""

    digest = hashlib.sha256(
        json.dumps([int(base_seed), *parts], default=_json_default).encode("utf-8")
    ).digest()
    return int.from_bytes(digest[:8], "little") & ((1 << 63) - 1)


def set_global_determinism(seed: int, deterministic: bool = True) -> None:
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=True)


def resolve_device(spec: str = "auto") -> torch.device:
    if spec == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(spec)


@dataclass(frozen=True)
class ExperimentConfig:
    """Fully serializable runner configuration.

    ``cnn_side`` is the side length presented to the CNN; the public Gaussian
    projection dimension is therefore ``cnn_side**2``.  MNIST uses
    ``cnn_side=10`` (h=100).  The 10-by-10 array is symmetrically zero-padded
    to 16-by-16 inside the learner so the unchanged valid 5-by-5 convolution
    stack remains well-defined; padding adds no projected coordinates.
    """

    mode: str = "smoke"
    seed: int = 20260711
    data_root: str = "/content/data"
    output_dir: str = "/content/igdppa_results"
    device: str = "auto"
    deterministic: bool = True
    cnn_side: int = 10
    cnn_minimum_input_side: int = 16
    n_participants: int = 5
    partition: str = "stratified_iid"
    dirichlet_alpha: float = 0.5
    min_participant_size: int = 16
    val_size: int = 2_000
    train_limit: Optional[int] = 10_000
    test_limit: Optional[int] = 2_000
    batch_size: int = 128
    eval_batch_size: int = 512
    downstream_epochs: int = 1
    auditor_epochs: int = 1
    learning_rate: float = 1.0e-3
    weight_decay: float = 0.0
    num_workers: int = 0
    pin_memory: bool = True
    protocols: tuple[str, ...] = (
        "c_gdp",
        "c_gdp_an",
        "i_gdp",
        "aa_i_gdp",
        "aa_i_gdp_an",
        "pa_i_gdp",
    )
    outer_seeds: tuple[int, ...] = (0,)
    draws_per_seed: int = 1
    private_lambda_upper: Optional[float] = None
    anchor_lambda_upper: Optional[float] = None
    pilot_manifest_path: Optional[str] = None
    pilot_manifest_sha256: Optional[str] = None
    pilot_candidate_lambdas: tuple[float, ...] = (
        0.0,
        2.0**-8,
        2.0**-7,
        2.0**-6,
        2.0**-5,
        2.0**-4,
        2.0**-3,
        2.0**-2,
        2.0**-1,
        2.0**0,
        2.0**1,
    )
    pilot_max_grid_extensions: int = 6
    anchor_ratio: float = 4.0
    anchor_design: str = "centered_unit_isometric_v2"
    anchor_scale_rule: str = "unit_singular_values"
    anchor_noise_parameterization: str = "absolute_v"
    pa_iterations: int = 500
    pa_tolerance: float = 1.0e-7
    reference_participant: int = 0
    save_models: bool = True
    ssim_max_images: int = 256
    linkage_max_queries: int = 256
    linkage_candidates: int = 10
    bootstrap_resamples: int = 2_000
    equivalence_alpha: float = 0.05
    equivalence_margins: Mapping[str, float] = field(
        default_factory=lambda: {
            "utility.test_accuracy": 0.005,
            "utility.test_balanced_accuracy": 0.005,
            "utility.test_macro_f1": 0.005,
            "best_attack.metrics.auditor_accuracy": 0.010,
            "best_attack.metrics.auditor_macro_f1": 0.010,
            "best_attack.metrics.reduced_nrmse": 0.010,
            "best_attack.metrics.pixel_nrmse": 0.010,
        }
    )
    pilot_max_accuracy_drop: float = 0.01
    run_attacks: bool = True
    run_equivalence: bool = True

    def __post_init__(self) -> None:
        if self.cnn_side < 4 or self.cnn_side > RAW_SIDE:
            raise ValueError("cnn_side must be in [4, 28] for the padded reference-style CNN")
        if self.cnn_side * self.cnn_side > RAW_DIM:
            raise ValueError("cnn_side**2 cannot exceed the 784 raw MNIST pixels")
        if self.cnn_minimum_input_side < 16:
            raise ValueError("cnn_minimum_input_side must be at least 16")
        if self.n_participants < 1:
            raise ValueError("n_participants must be positive")
        if not self.outer_seeds:
            raise ValueError("outer_seeds cannot be empty")
        if self.draws_per_seed < 1:
            raise ValueError("draws_per_seed must be positive")
        for upper in (self.private_lambda_upper, self.anchor_lambda_upper):
            if upper is not None and upper < 0:
                raise ValueError("lambda upper bounds must be nonnegative")
        if self.partition not in {"stratified_iid", "dirichlet"}:
            raise ValueError("partition must be 'stratified_iid' or 'dirichlet'")
        if self.anchor_ratio <= 1.0:
            raise ValueError("anchor_ratio must exceed 1 so centered anchors can be full rank")
        if self.anchor_design != "centered_unit_isometric_v2":
            raise ValueError("This notebook requires the centered isometric anchor design")
        if self.anchor_scale_rule != "unit_singular_values":
            raise ValueError("This notebook fixes every centered-anchor singular value to one")
        if self.anchor_noise_parameterization != "absolute_v":
            raise ValueError("This notebook requires direct anchor-noise standard deviation v")

    @classmethod
    def for_mode(cls, mode: str, **overrides: Any) -> "ExperimentConfig":
        mode = mode.lower().strip()
        presets: dict[str, dict[str, Any]] = {
            "smoke": {},
            "pilot": {
                "mode": "pilot",
                "cnn_side": 10,
                "n_participants": 10,
                "val_size": 5_000,
                "train_limit": 20_000,
                "test_limit": 0,
                "batch_size": 256,
                "downstream_epochs": 3,
                "auditor_epochs": 3,
                "num_workers": 2,
                "outer_seeds": (0,),
                "draws_per_seed": 1,
                "ssim_max_images": 1_000,
                "linkage_max_queries": 1_000,
                "bootstrap_resamples": 5_000,
            },
            "full": {
                "mode": "full",
                "cnn_side": 10,
                "n_participants": 10,
                "val_size": 10_000,
                "train_limit": 50_000,
                "test_limit": None,
                "batch_size": 256,
                "downstream_epochs": 15,
                "auditor_epochs": 10,
                "num_workers": 2,
                "outer_seeds": (0,),
                "draws_per_seed": 100,
                "ssim_max_images": 2_000,
                "linkage_max_queries": 2_000,
                "bootstrap_resamples": 10_000,
            },
        }
        if mode not in presets:
            raise ValueError(f"Unknown mode {mode!r}; expected smoke, pilot, or full")
        values = {"mode": mode, **presets[mode], **overrides}
        if mode == "smoke":
            values.setdefault("private_lambda_upper", 0.05)
            values.setdefault("anchor_lambda_upper", 0.05)
        return cls(**values)

    @property
    def reduced_dim(self) -> int:
        return self.cnn_side * self.cnn_side

    @property
    def h(self) -> int:
        """Paper reduced dimension, equal to the number of projected columns."""

        return self.reduced_dim

    @property
    def anchor_rows(self) -> int:
        return int(math.ceil(self.anchor_ratio * self.reduced_dim))

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def make_config(mode: str = "smoke", **overrides: Any) -> dict[str, Any]:
    """Return a notebook-friendly plain dictionary configuration."""

    return ExperimentConfig.for_mode(mode, **overrides).to_dict()


def coerce_config(config: Mapping[str, Any] | ExperimentConfig) -> ExperimentConfig:
    if isinstance(config, ExperimentConfig):
        return config
    values = dict(config)
    if "h" in values:
        paper_h = int(values.pop("h"))
        side = math.isqrt(paper_h)
        if side * side != paper_h:
            raise ValueError("Paper dimension h must be a perfect square for the CNN")
        if "cnn_side" in values and int(values["cnn_side"]) != side:
            raise ValueError("Configured h and cnn_side disagree")
        values["cnn_side"] = side
    mode = str(values.pop("mode", "smoke"))
    return ExperimentConfig.for_mode(mode, **values)


@dataclass
class SplitTensors:
    raw: Tensor
    labels: Tensor
    source_indices: Tensor
    participant_ids: Tensor
    reduced: Optional[Tensor] = None

    def __len__(self) -> int:
        return int(self.labels.numel())

    def to(self, device: torch.device | str) -> "SplitTensors":
        return SplitTensors(
            raw=self.raw.to(device),
            labels=self.labels.to(device),
            source_indices=self.source_indices.to(device),
            participant_ids=self.participant_ids.to(device),
            reduced=None if self.reduced is None else self.reduced.to(device),
        )


@dataclass
class PublicReducer:
    """Data-independent public map ``Z = X @ F`` and its MP decoder."""

    F: Tensor
    F_pinv: Tensor
    matrix_singular_values: Tensor
    projection_side: int
    projection_seed: int

    @property
    def input_dim(self) -> int:
        return int(self.F.shape[0])

    @property
    def output_dim(self) -> int:
        return int(self.F.shape[1])

    @property
    def decoder(self) -> Tensor:
        return self.F_pinv

    def transform(self, x: Tensor) -> Tensor:
        x = x.to(self.F.device, self.F.dtype)
        return x @ self.F

    def decode(self, z: Tensor) -> Tensor:
        return z.to(self.F.device, self.F.dtype) @ self.decoder

    def manifest(self) -> dict[str, Any]:
        f_cpu = self.F.detach().cpu().contiguous()
        singular = self.matrix_singular_values.detach().cpu()
        sigma_max = float(singular[0].item())
        sigma_min = float(singular[-1].item())
        return {
            "kind": "gaussian_random_projection",
            "matrix_data_dependent": False,
            "centering": "none",
            "entry_distribution": "iid_normal",
            "entry_standard_deviation": 1.0,
            "projection_seed": self.projection_seed,
            "input_dim": self.input_dim,
            "output_dim": self.output_dim,
            "h": self.output_dim,
            "projection_side": self.projection_side,
            "cnn_side": self.projection_side,
            "reduced_dim": self.output_dim,
            "paper_reduced_dimension": self.output_dim,
            "matrix_sha256": hashlib.sha256(f_cpu.numpy().tobytes()).hexdigest(),
            "decoder_sha256": hashlib.sha256(
                self.F_pinv.detach().cpu().contiguous().numpy().tobytes()
            ).hexdigest(),
            "matrix_sigma_min": sigma_min,
            "matrix_sigma_max": sigma_max,
            "matrix_condition_number": sigma_max / sigma_min,
            "matrix_singular_values": singular.tolist(),
            "pseudoinverse_frobenius_norm": float(
                torch.linalg.norm(self.F_pinv.detach().cpu(), ord="fro").item()
            ),
        }


@dataclass
class MNISTDataBundle:
    train: SplitTensors
    val: SplitTensors
    test: SplitTensors
    reducer: PublicReducer
    manifest: dict[str, Any]

    def split(self, name: str) -> SplitTensors:
        if name not in {"train", "val", "test"}:
            raise KeyError(name)
        return getattr(self, name)

    def participant_blocks(
        self, split: str, *, reduced: bool = True
    ) -> tuple[list[Tensor], list[Tensor], list[Tensor]]:
        data = self.split(split)
        values = data.reduced if reduced else data.raw
        if values is None:
            raise RuntimeError("Reduced tensors have not been prepared")
        x_blocks: list[Tensor] = []
        y_blocks: list[Tensor] = []
        index_blocks: list[Tensor] = []
        for participant in range(int(data.participant_ids.max().item()) + 1):
            idx = torch.nonzero(data.participant_ids == participant, as_tuple=False).flatten()
            x_blocks.append(values[idx])
            y_blocks.append(data.labels[idx])
            index_blocks.append(idx)
        return x_blocks, y_blocks, index_blocks


def _stratified_sample_indices(labels: np.ndarray, size: int, seed: int) -> np.ndarray:
    labels = np.asarray(labels, dtype=np.int64)
    if size >= labels.size:
        return np.arange(labels.size, dtype=np.int64)
    if size <= 0:
        return np.empty(0, dtype=np.int64)
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(labels, return_counts=True)
    exact = size * counts / counts.sum()
    allocations = np.floor(exact).astype(int)
    allocations = np.minimum(allocations, counts)
    remaining = size - int(allocations.sum())
    order = np.argsort(-(exact - allocations))
    for pos in order:
        if remaining == 0:
            break
        if allocations[pos] < counts[pos]:
            allocations[pos] += 1
            remaining -= 1
    chosen: list[np.ndarray] = []
    for cls, n_take in zip(classes, allocations):
        idx = np.flatnonzero(labels == cls)
        rng.shuffle(idx)
        chosen.append(idx[:n_take])
    result = np.concatenate(chosen)
    rng.shuffle(result)
    return result.astype(np.int64, copy=False)


def participant_partition(
    labels: Tensor | np.ndarray,
    n_participants: int,
    seed: int,
    scheme: str = "stratified_iid",
    dirichlet_alpha: float = 0.5,
    min_size: int = 1,
) -> Tensor:
    """Return deterministic participant IDs aligned with ``labels``."""

    y = np.asarray(labels.detach().cpu() if isinstance(labels, Tensor) else labels)
    y = y.astype(np.int64, copy=False)
    if n_participants == 1:
        return torch.zeros(y.size, dtype=torch.long)
    rng = np.random.default_rng(seed)
    assignment = np.full(y.size, -1, dtype=np.int64)
    classes = np.unique(y)
    if scheme == "stratified_iid":
        offsets = np.zeros(n_participants, dtype=np.int64)
        for cls in classes:
            idx = np.flatnonzero(y == cls)
            rng.shuffle(idx)
            # Rotating the start avoids systematically favoring participant 0.
            start = int(rng.integers(0, n_participants))
            for j, sample_idx in enumerate(idx):
                participant = (start + j) % n_participants
                assignment[sample_idx] = participant
                offsets[participant] += 1
    elif scheme == "dirichlet":
        for attempt in range(200):
            assignment.fill(-1)
            for cls in classes:
                idx = np.flatnonzero(y == cls)
                rng.shuffle(idx)
                proportions = rng.dirichlet(
                    np.full(n_participants, float(dirichlet_alpha))
                )
                counts = rng.multinomial(idx.size, proportions)
                cursor = 0
                for participant, count in enumerate(counts):
                    assignment[idx[cursor : cursor + count]] = participant
                    cursor += count
            sizes = np.bincount(assignment, minlength=n_participants)
            if sizes.min() >= min_size:
                break
        else:
            raise RuntimeError(
                "Could not create a Dirichlet partition meeting min_size; "
                "increase data or reduce min_size/participants"
            )
    else:
        raise ValueError(f"Unknown participant partition scheme {scheme!r}")
    if np.any(assignment < 0):
        raise RuntimeError("Internal error: unassigned samples")
    return torch.from_numpy(assignment)


def make_public_gaussian_projection(
    projection_side: int,
    seed: int,
) -> PublicReducer:
    """Create the reference paper's uncentered public Gaussian reducer.

    The returned matrix has the paper's row-oriented convention ``F`` with
    shape ``(784, projection_side**2)`` so that ``Z = X @ F``.  Its entries are iid standard
    normal, exactly matching the supplied reference.  No dataset record,
    participant statistic, or centering mean enters its construction.
    """

    output_dim = projection_side * projection_side
    if not 1 <= output_dim <= RAW_DIM:
        raise ValueError(f"Gaussian reduction dimension must be in [1, {RAW_DIM}]")

    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    matrix64 = torch.randn(
        (RAW_DIM, output_dim), generator=generator, dtype=torch.float64
    )
    singular64 = torch.linalg.svdvals(matrix64)
    tolerance = (
        torch.finfo(torch.float64).eps
        * max(matrix64.shape)
        * singular64[0]
    )
    if singular64[-1] <= tolerance:
        raise RuntimeError("Seeded Gaussian projection was numerically rank deficient")
    decoder64 = torch.linalg.pinv(matrix64)
    matrix = matrix64.float()
    decoder = decoder64.float()
    return PublicReducer(
        F=matrix,
        F_pinv=decoder,
        matrix_singular_values=singular64.float(),
        projection_side=projection_side,
        projection_seed=int(seed),
    )


def _select_rows(tensor: Tensor, indices: np.ndarray) -> Tensor:
    return tensor[torch.as_tensor(indices, dtype=torch.long)].clone()


def prepare_mnist_data(
    config: Mapping[str, Any] | ExperimentConfig,
    *,
    download: bool = True,
    include_test: Optional[bool] = None,
) -> MNISTDataBundle:
    """Download MNIST and create deterministic train/val/test/client splits.

    Pilot mode forcibly excludes the official test split.  This is stronger
    than merely not scoring it: the test dataset object is never constructed,
    so pilot code cannot accidentally inspect test examples.
    """

    cfg = coerce_config(config)
    if cfg.mode == "pilot":
        include_test = False
    elif include_test is None:
        include_test = True
    set_global_determinism(cfg.seed, cfg.deterministic)
    try:
        from torchvision.datasets import MNIST
    except ImportError as exc:
        raise RuntimeError("torchvision is required to load MNIST") from exc

    train_ds = MNIST(cfg.data_root, train=True, download=download)
    all_train_x = train_ds.data.float().reshape(-1, RAW_DIM) / 255.0
    all_train_y = train_ds.targets.long()
    if include_test:
        test_ds = MNIST(cfg.data_root, train=False, download=download)
        all_test_x = test_ds.data.float().reshape(-1, RAW_DIM) / 255.0
        all_test_y = test_ds.targets.long()
    else:
        all_test_x = torch.empty((0, RAW_DIM), dtype=torch.float32)
        all_test_y = torch.empty((0,), dtype=torch.long)

    val_idx = _stratified_sample_indices(
        all_train_y.numpy(), cfg.val_size, named_seed(cfg.seed, "val")
    )
    train_mask = np.ones(len(all_train_y), dtype=bool)
    train_mask[val_idx] = False
    train_idx = np.flatnonzero(train_mask)
    if cfg.train_limit is not None and cfg.train_limit < train_idx.size:
        local = _stratified_sample_indices(
            all_train_y[torch.from_numpy(train_idx)].numpy(),
            cfg.train_limit,
            named_seed(cfg.seed, "train_subset"),
        )
        train_idx = train_idx[local]
    test_idx = np.arange(len(all_test_y), dtype=np.int64)
    if include_test and cfg.test_limit is not None and cfg.test_limit < test_idx.size:
        test_idx = _stratified_sample_indices(
            all_test_y.numpy(), cfg.test_limit, named_seed(cfg.seed, "test_subset")
        )

    train_x, train_y = _select_rows(all_train_x, train_idx), _select_rows(all_train_y, train_idx)
    val_x, val_y = _select_rows(all_train_x, val_idx), _select_rows(all_train_y, val_idx)
    test_x, test_y = _select_rows(all_test_x, test_idx), _select_rows(all_test_y, test_idx)

    # Generate the public matrix without passing any dataset or participant
    # record into its constructor.  The same uncentered Z = X @ F map is then
    # applied to every split and participant.
    reducer = make_public_gaussian_projection(
        cfg.cnn_side,
        named_seed(cfg.seed, "public_gaussian_projection", cfg.reduced_dim),
    )
    train_z = reducer.transform(train_x)
    val_z = reducer.transform(val_x)
    test_z = reducer.transform(test_x)

    train_participant_ids = participant_partition(
        train_y,
        cfg.n_participants,
        named_seed(cfg.seed, "participant_partition", "train"),
        scheme=cfg.partition,
        dirichlet_alpha=cfg.dirichlet_alpha,
        min_size=min(
            cfg.min_participant_size, max(1, len(train_y) // (2 * cfg.n_participants))
        ),
    )
    def build_split(
        name: str,
        x: Tensor,
        y: Tensor,
        idx: np.ndarray,
        z: Tensor,
        participant_ids: Optional[Tensor] = None,
    ) -> SplitTensors:
        if participant_ids is None:
            participant_ids = participant_partition(
                y,
                cfg.n_participants,
                named_seed(cfg.seed, "participant_partition", name),
                scheme=cfg.partition,
                dirichlet_alpha=cfg.dirichlet_alpha,
                min_size=min(
                    cfg.min_participant_size, max(1, len(y) // (2 * cfg.n_participants))
                ),
            )
        return SplitTensors(
            raw=x,
            labels=y,
            source_indices=torch.as_tensor(idx, dtype=torch.long),
            participant_ids=participant_ids,
            reduced=z,
        )

    train = build_split(
        "train", train_x, train_y, train_idx, train_z, train_participant_ids
    )
    val = build_split("val", val_x, val_y, val_idx, val_z)
    test = build_split("test", test_x, test_y, test_idx, test_z)
    manifest = {
        "dataset": "MNIST",
        "seed": cfg.seed,
        "mode": cfg.mode,
        "cnn_side": cfg.cnn_side,
        "reduced_dim": cfg.reduced_dim,
        "anchor_rows": cfg.anchor_rows,
        "train_size": len(train),
        "val_size": len(val),
        "test_size": len(test),
        "test_loaded": bool(include_test),
        "partition": cfg.partition,
        "n_participants": cfg.n_participants,
        "participant_sizes": {
            name: torch.bincount(split.participant_ids, minlength=cfg.n_participants).tolist()
            for name, split in (("train", train), ("val", val), ("test", test))
        },
        "source_index_hashes": {
            "train": stable_hash(train.source_indices.tolist(), 32),
            "val": stable_hash(val.source_indices.tolist(), 32),
            "test": stable_hash(test.source_indices.tolist(), 32),
        },
        "reducer": reducer.manifest(),
    }
    return MNISTDataBundle(train=train, val=val, test=test, reducer=reducer, manifest=manifest)


def as_square_images(values: Tensor | np.ndarray, h: Optional[int] = None) -> Tensor:
    tensor = torch.as_tensor(values, dtype=torch.float32)
    if tensor.ndim == 4:
        if tensor.shape[1] != 1 or tensor.shape[2] != tensor.shape[3]:
            raise ValueError(f"Expected (n,1,h,h), got {tuple(tensor.shape)}")
        return tensor
    if tensor.ndim == 3:
        if tensor.shape[1] != tensor.shape[2]:
            raise ValueError(f"Expected square images, got {tuple(tensor.shape)}")
        return tensor.unsqueeze(1)
    if tensor.ndim != 2:
        raise ValueError(f"Cannot reshape representation with shape {tuple(tensor.shape)}")
    if h is None:
        h = math.isqrt(int(tensor.shape[1]))
    if h * h != tensor.shape[1]:
        raise ValueError(f"Feature dimension {tensor.shape[1]} is not h**2 for h={h}")
    return tensor.reshape(-1, 1, h, h)


class ReferencePaperCNN(nn.Module):
    """Reference-style MNIST CNN with the flatten size adapted to square h.

    Architecture: 10x5x5 conv, max-pool; 20x5x5 conv, max-pool;
    fully connected 50 -> 10, with ReLU hidden activations.  For the 10x10
    projected representation, the input is symmetrically zero-padded to
    16-by-16 before the unchanged valid-convolution stack.  The padding adds no
    projected features or learned parameters.
    """

    def __init__(
        self,
        h: int,
        n_classes: int = NUM_CLASSES,
        *,
        minimum_input_side: int = 16,
    ) -> None:
        super().__init__()
        if h < 1:
            raise ValueError("h must be positive")
        if minimum_input_side < 16:
            raise ValueError("minimum_input_side must be at least 16")
        self.h = int(h)
        self.minimum_input_side = int(minimum_input_side)
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.pool = nn.MaxPool2d(2)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, h, h)
            encoded = self._convolve(dummy)
            flat_dim = int(encoded.numel())
        self.fc1 = nn.Linear(flat_dim, 50)
        self.fc2 = nn.Linear(50, n_classes)

    def _convolve(self, x: Tensor) -> Tensor:
        if x.shape[-2:] != (self.h, self.h):
            raise ValueError(
                f"Expected {self.h}x{self.h} inputs, got {tuple(x.shape[-2:])}"
            )
        vertical = max(0, self.minimum_input_side - self.h)
        horizontal = max(0, self.minimum_input_side - self.h)
        if vertical or horizontal:
            left, right = horizontal // 2, horizontal - horizontal // 2
            top, bottom = vertical // 2, vertical - vertical // 2
            x = F.pad(x, (left, right, top, bottom))
        x = self.pool(F.relu(self.conv1(x)))
        return self.pool(F.relu(self.conv2(x)))

    def forward_features(self, x: Tensor) -> Tensor:
        return F.relu(self.fc1(torch.flatten(self._convolve(x), 1)))

    def forward(self, x: Tensor) -> Tensor:
        return self.fc2(self.forward_features(x))


class CleanMNISTAuditor(nn.Module):
    """Independent raw-image digit auditor; freeze after clean training."""

    def __init__(self, n_classes: int = NUM_CLASSES) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward_features(self, x: Tensor) -> Tensor:
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        return F.relu(self.fc1(torch.flatten(x, 1)))

    def forward(self, x: Tensor) -> Tensor:
        return self.fc2(self.forward_features(x))


def _make_loader(
    x: Tensor,
    y: Tensor,
    batch_size: int,
    *,
    shuffle: bool,
    seed: int,
    num_workers: int,
    pin_memory: bool,
) -> DataLoader:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    def worker_init_fn(worker_id: int) -> None:
        worker_seed = named_seed(seed, "worker", worker_id) % (2**32 - 1)
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    return DataLoader(
        TensorDataset(x, y),
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=num_workers,
        pin_memory=pin_memory and torch.cuda.is_available(),
        worker_init_fn=worker_init_fn if num_workers else None,
        persistent_workers=bool(num_workers),
    )


def classification_metrics(
    labels: Tensor | np.ndarray,
    predictions: Tensor | np.ndarray,
    *,
    n_classes: int = NUM_CLASSES,
) -> dict[str, Any]:
    y = torch.as_tensor(labels, dtype=torch.long).flatten().cpu()
    pred = torch.as_tensor(predictions, dtype=torch.long).flatten().cpu()
    if y.shape != pred.shape:
        raise ValueError("labels and predictions must have identical shape")
    confusion = torch.zeros(n_classes, n_classes, dtype=torch.long)
    for true, guessed in zip(y.tolist(), pred.tolist()):
        if 0 <= true < n_classes and 0 <= guessed < n_classes:
            confusion[true, guessed] += 1
    support = confusion.sum(dim=1).float()
    predicted = confusion.sum(dim=0).float()
    tp = confusion.diag().float()
    recall = torch.where(support > 0, tp / support, torch.zeros_like(tp))
    precision = torch.where(predicted > 0, tp / predicted, torch.zeros_like(tp))
    f1 = torch.where(
        precision + recall > 0,
        2 * precision * recall / (precision + recall),
        torch.zeros_like(precision),
    )
    return {
        "accuracy": float(tp.sum().item() / max(1, len(y))),
        "balanced_accuracy": float(recall.mean().item()),
        "macro_f1": float(f1.mean().item()),
        "per_class_recall": recall.tolist(),
        "per_class_f1": f1.tolist(),
        "confusion_matrix": confusion.tolist(),
        "n": int(len(y)),
    }


def macro_ovr_auroc(
    labels: Tensor | np.ndarray,
    probabilities: Tensor | np.ndarray,
    *,
    n_classes: int = NUM_CLASSES,
) -> float:
    """Macro one-vs-rest AUROC with a dependency-light rank fallback."""

    y = np.asarray(torch.as_tensor(labels, dtype=torch.long).cpu())
    scores = np.asarray(torch.as_tensor(probabilities, dtype=torch.float64).cpu())
    if scores.shape != (len(y), n_classes):
        raise ValueError("probabilities must have shape (n, n_classes)")
    try:
        from sklearn.metrics import roc_auc_score

        return float(
            roc_auc_score(y, scores, multi_class="ovr", average="macro", labels=range(n_classes))
        )
    except (ImportError, ValueError):
        aucs: list[float] = []
        for cls in range(n_classes):
            positive = y == cls
            n_pos = int(positive.sum())
            n_neg = int((~positive).sum())
            if not n_pos or not n_neg:
                continue
            order = np.argsort(scores[:, cls], kind="mergesort")
            sorted_scores = scores[order, cls]
            ranks = np.empty(len(y), dtype=np.float64)
            start = 0
            while start < len(y):
                stop = start + 1
                while stop < len(y) and sorted_scores[stop] == sorted_scores[start]:
                    stop += 1
                ranks[order[start:stop]] = 0.5 * (start + 1 + stop)
                start = stop
            rank_sum = float(ranks[positive].sum())
            aucs.append((rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))
        return float(np.mean(aucs)) if aucs else math.nan


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device | str,
) -> dict[str, Any]:
    model.eval()
    device = torch.device(device)
    total_loss = 0.0
    labels: list[Tensor] = []
    predictions: list[Tensor] = []
    probabilities: list[Tensor] = []
    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)
        logits = model(x_batch)
        total_loss += float(F.cross_entropy(logits, y_batch, reduction="sum").item())
        labels.append(y_batch.cpu())
        predictions.append(logits.argmax(dim=1).cpu())
        probabilities.append(logits.softmax(dim=1).cpu())
    y = torch.cat(labels)
    pred = torch.cat(predictions)
    probability = torch.cat(probabilities)
    metrics = classification_metrics(y, pred)
    metrics["macro_ovr_auroc"] = macro_ovr_auroc(y, probability)
    metrics["loss"] = total_loss / max(1, len(y))
    return metrics


def train_classifier(
    model: nn.Module,
    train_x: Tensor,
    train_y: Tensor,
    val_x: Tensor,
    val_y: Tensor,
    config: Mapping[str, Any] | ExperimentConfig,
    *,
    seed: Optional[int] = None,
    epochs: Optional[int] = None,
    checkpoint_path: Optional[str | Path] = None,
) -> dict[str, Any]:
    cfg = coerce_config(config)
    seed = cfg.seed if seed is None else int(seed)
    epochs = cfg.downstream_epochs if epochs is None else int(epochs)
    set_global_determinism(seed, cfg.deterministic)
    device = resolve_device(cfg.device)
    model.to(device)
    train_loader = _make_loader(
        train_x,
        train_y,
        cfg.batch_size,
        shuffle=True,
        seed=named_seed(seed, "train_loader"),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )
    val_loader = _make_loader(
        val_x,
        val_y,
        cfg.eval_batch_size,
        shuffle=False,
        seed=named_seed(seed, "val_loader"),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    best_state = copy.deepcopy(model.state_dict())
    best_balanced_accuracy = -math.inf
    best_epoch = 0
    history: list[dict[str, Any]] = []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_batch)
            loss = F.cross_entropy(logits, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item()) * len(y_batch)
            seen += len(y_batch)
        val_metrics = evaluate_model(model, val_loader, device)
        entry = {
            "epoch": epoch,
            "train_loss": running_loss / max(1, seen),
            **{f"val_{key}": value for key, value in val_metrics.items()},
        }
        history.append(entry)
        if val_metrics["balanced_accuracy"] > best_balanced_accuracy:
            best_balanced_accuracy = float(val_metrics["balanced_accuracy"])
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    if checkpoint_path is not None:
        path = Path(checkpoint_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "state_dict": best_state,
                "best_epoch": best_epoch,
                "best_val_balanced_accuracy": best_balanced_accuracy,
                "seed": seed,
            },
            path,
        )
    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_val_balanced_accuracy": best_balanced_accuracy,
        # Backward-compatible label used by the smoke report; the selected
        # value is balanced accuracy, not raw accuracy.
        "best_val_accuracy": best_balanced_accuracy,
        "history": history,
    }


def freeze_model(model: nn.Module) -> nn.Module:
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def train_or_load_auditor(
    data: MNISTDataBundle,
    config: Mapping[str, Any] | ExperimentConfig,
    *,
    checkpoint_path: Optional[str | Path] = None,
) -> tuple[CleanMNISTAuditor, dict[str, Any]]:
    """Train a clean raw-image auditor once, then return it frozen."""

    cfg = coerce_config(config)
    path = Path(checkpoint_path or Path(cfg.output_dir) / "clean_mnist_auditor.pt")
    device = resolve_device(cfg.device)
    fingerprint = stable_hash(
        {
            "train": data.manifest["source_index_hashes"]["train"],
            "val": data.manifest["source_index_hashes"]["val"],
            "epochs": cfg.auditor_epochs,
            "seed": named_seed(cfg.seed, "auditor"),
        },
        32,
    )
    model = CleanMNISTAuditor()
    metadata: dict[str, Any]
    if path.exists():
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        if checkpoint.get("fingerprint") == fingerprint:
            model.load_state_dict(checkpoint["state_dict"])
            metadata = dict(checkpoint.get("metadata", {}))
            model.to(device)
            return freeze_model(model), metadata

    trained = train_classifier(
        model,
        as_square_images(data.train.raw, RAW_SIDE),
        data.train.labels,
        as_square_images(data.val.raw, RAW_SIDE),
        data.val.labels,
        replace(cfg, downstream_epochs=cfg.auditor_epochs),
        seed=named_seed(cfg.seed, "auditor"),
        epochs=cfg.auditor_epochs,
    )
    model = trained["model"]
    val_loader = _make_loader(
        as_square_images(data.val.raw, RAW_SIDE),
        data.val.labels,
        cfg.eval_batch_size,
        shuffle=False,
        seed=named_seed(cfg.seed, "auditor_val_eval"),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )
    metadata = {
        "fingerprint": fingerprint,
        "best_epoch": trained["best_epoch"],
        "validation": evaluate_model(model, val_loader, device),
        "test": None,
        "test_used": False,
    }
    if len(data.test):
        test_loader = _make_loader(
            as_square_images(data.test.raw, RAW_SIDE),
            data.test.labels,
            cfg.eval_batch_size,
            shuffle=False,
            seed=named_seed(cfg.seed, "auditor_test_eval"),
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
        )
        metadata["test"] = evaluate_model(model, test_loader, device)
        metadata["test_used"] = True
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "fingerprint": fingerprint,
            "state_dict": model.state_dict(),
            "metadata": metadata,
        },
        path,
    )
    return freeze_model(model), metadata


@torch.no_grad()
def auditor_semantic_metrics(
    auditor: CleanMNISTAuditor,
    reconstructed_raw: Tensor | np.ndarray,
    labels: Tensor | np.ndarray,
    *,
    device: torch.device | str = "cpu",
    batch_size: int = 512,
) -> dict[str, Any]:
    """Measure 10-class recognizability with a frozen independent auditor."""

    images = as_square_images(torch.as_tensor(reconstructed_raw).float(), RAW_SIDE)
    y = torch.as_tensor(labels, dtype=torch.long).cpu()
    loader = DataLoader(TensorDataset(images, y), batch_size=batch_size, shuffle=False)
    metrics = evaluate_model(auditor.to(device), loader, device)
    return {f"auditor_{key}": value for key, value in metrics.items()}


def _fallback_ssim(image_a: np.ndarray, image_b: np.ndarray) -> float:
    """Global SSIM fallback used only when scikit-image is unavailable."""

    a = image_a.astype(np.float64, copy=False)
    b = image_b.astype(np.float64, copy=False)
    mu_a, mu_b = float(a.mean()), float(b.mean())
    var_a, var_b = float(a.var()), float(b.var())
    covariance = float(((a - mu_a) * (b - mu_b)).mean())
    c1, c2 = 0.01**2, 0.03**2
    numerator = (2 * mu_a * mu_b + c1) * (2 * covariance + c2)
    denominator = (mu_a**2 + mu_b**2 + c1) * (var_a + var_b + c2)
    return float(numerator / denominator) if denominator else 1.0


def compute_reconstruction_metrics(
    original_raw: Tensor | np.ndarray,
    reconstructed_raw: Tensor | np.ndarray,
    *,
    data_range: float = 1.0,
    ssim_max_images: Optional[int] = 2_000,
    seed: int = 0,
) -> dict[str, Any]:
    """Compute raw-pixel NRMSE, PSNR and SSIM.

    MSE/RMSE/NRMSE are evaluated on the *unclipped* reconstruction.  PSNR and
    SSIM use a [0, 1] clipped copy because they are image-quality measures.
    """

    original = torch.as_tensor(original_raw, dtype=torch.float32).reshape(-1, RAW_DIM)
    recovered = torch.as_tensor(reconstructed_raw, dtype=torch.float32).reshape(-1, RAW_DIM)
    if original.shape != recovered.shape:
        raise ValueError(
            f"Original/reconstruction shapes differ: {tuple(original.shape)} vs "
            f"{tuple(recovered.shape)}"
        )
    error = recovered - original
    mse_per = error.square().mean(dim=1)
    rmse = float(torch.sqrt(mse_per.mean()).item())
    denominator = float(torch.linalg.vector_norm(original).item())
    nrmse = float(torch.linalg.vector_norm(error).item() / max(denominator, 1e-12))
    clipped = recovered.clamp(0.0, data_range)
    clipped_mse_per = (clipped - original).square().mean(dim=1)
    finite_psnr = 10.0 * torch.log10(
        torch.tensor(data_range**2) / clipped_mse_per.clamp_min(1e-12)
    )

    n_images = len(original)
    if ssim_max_images is None or ssim_max_images >= n_images:
        chosen = np.arange(n_images)
    else:
        chosen = _stratified_sample_indices(
            np.zeros(n_images, dtype=np.int64), ssim_max_images, seed
        )
    original_np = original[torch.from_numpy(chosen)].reshape(-1, RAW_SIDE, RAW_SIDE).numpy()
    clipped_np = clipped[torch.from_numpy(chosen)].reshape(-1, RAW_SIDE, RAW_SIDE).numpy()
    try:
        from skimage.metrics import structural_similarity

        ssim_values = [
            float(structural_similarity(a, b, data_range=data_range))
            for a, b in zip(original_np, clipped_np)
        ]
        ssim_backend = "skimage"
    except ImportError:
        ssim_values = [_fallback_ssim(a, b) for a, b in zip(original_np, clipped_np)]
        ssim_backend = "global_fallback"
    return {
        "mse": float(mse_per.mean().item()),
        "rmse": rmse,
        "nrmse": nrmse,
        "psnr_db": float(finite_psnr.mean().item()),
        "psnr_db_std": float(finite_psnr.std(unbiased=False).item()),
        "ssim": float(np.mean(ssim_values)) if ssim_values else math.nan,
        "ssim_std": float(np.std(ssim_values)) if ssim_values else math.nan,
        "ssim_n": int(len(ssim_values)),
        "ssim_backend": ssim_backend,
        "clipped_fraction": float(
            ((recovered < 0.0) | (recovered > data_range)).float().mean().item()
        ),
        "n": int(n_images),
    }


def reduced_reconstruction_metrics(
    original_reduced: Tensor | np.ndarray,
    recovered_reduced: Tensor | np.ndarray,
) -> dict[str, float]:
    original = torch.as_tensor(original_reduced, dtype=torch.float32)
    recovered = torch.as_tensor(recovered_reduced, dtype=torch.float32)
    if original.shape != recovered.shape:
        raise ValueError("Reduced reconstruction shapes differ")
    error = recovered - original
    return {
        "reduced_mse": float(error.square().mean().item()),
        "reduced_rmse": float(torch.sqrt(error.square().mean()).item()),
        "reduced_nrmse": float(
            torch.linalg.vector_norm(error).item()
            / max(torch.linalg.vector_norm(original).item(), 1e-12)
        ),
    }


@torch.no_grad()
def _auditor_embeddings(
    auditor: CleanMNISTAuditor,
    raw_images: Tensor,
    device: torch.device | str,
    batch_size: int,
) -> Tensor:
    auditor = auditor.to(device).eval()
    outputs: list[Tensor] = []
    for start in range(0, len(raw_images), batch_size):
        batch = as_square_images(raw_images[start : start + batch_size], RAW_SIDE).to(device)
        outputs.append(auditor.forward_features(batch).cpu())
    return torch.cat(outputs, dim=0)


@dataclass(frozen=True)
class LinkageManifest:
    query_indices: tuple[int, ...]
    candidate_indices: tuple[tuple[int, ...], ...]
    n_candidates: int
    labels_hash: str
    seed: int

    @property
    def manifest_hash(self) -> str:
        return stable_hash(asdict(self), 32)

    def to_dict(self) -> dict[str, Any]:
        return {**asdict(self), "manifest_hash": self.manifest_hash}


def make_same_digit_linkage_manifest(
    labels: Tensor | np.ndarray,
    *,
    n_candidates: int = 10,
    max_queries: Optional[int] = 2_000,
    seed: int = 0,
) -> LinkageManifest:
    """Freeze exact-source + distinct same-label galleries across methods."""

    y = torch.as_tensor(labels, dtype=torch.long).flatten().cpu().numpy()
    if n_candidates < 2:
        raise ValueError("n_candidates must be at least two")
    if max_queries is None or max_queries >= len(y):
        query_indices = np.arange(len(y))
    else:
        query_indices = _stratified_sample_indices(y, max_queries, seed)
    rng = np.random.default_rng(seed)
    pools = {digit: np.flatnonzero(y == digit) for digit in range(NUM_CLASSES)}
    kept_queries: list[int] = []
    galleries: list[tuple[int, ...]] = []
    for query in query_indices.tolist():
        alternatives = pools[int(y[query])]
        alternatives = alternatives[alternatives != query]
        if alternatives.size < n_candidates - 1:
            continue
        distractors = rng.choice(alternatives, n_candidates - 1, replace=False)
        gallery = np.concatenate(([query], distractors))
        rng.shuffle(gallery)
        if len(np.unique(gallery)) != n_candidates or query not in gallery:
            raise RuntimeError("Invalid linkage gallery construction")
        kept_queries.append(query)
        galleries.append(tuple(int(item) for item in gallery))
    return LinkageManifest(
        query_indices=tuple(kept_queries),
        candidate_indices=tuple(galleries),
        n_candidates=n_candidates,
        labels_hash=stable_hash(y.tolist(), 32),
        seed=seed,
    )


def same_digit_source_linkage(
    original_raw: Tensor | np.ndarray,
    reconstructed_raw: Tensor | np.ndarray,
    labels: Tensor | np.ndarray,
    auditor: Optional[CleanMNISTAuditor] = None,
    *,
    n_candidates: int = 10,
    max_queries: Optional[int] = 2_000,
    seed: int = 0,
    device: torch.device | str = "cpu",
    batch_size: int = 512,
    manifest: Optional[LinkageManifest] = None,
) -> dict[str, Any]:
    """Link each reconstruction to its source among same-digit candidates.

    Candidate sets contain the true source plus ``n_candidates-1`` other test
    images with the *same digit*.  Thus the attack cannot succeed merely by
    classifying the digit.  Pixel L2 and auditor-embedding cosine linkage are
    reported; chance top-1 is ``1/n_candidates``.
    """

    original = torch.as_tensor(original_raw, dtype=torch.float32).reshape(-1, RAW_DIM)
    recovered = torch.as_tensor(reconstructed_raw, dtype=torch.float32).reshape(-1, RAW_DIM)
    y = torch.as_tensor(labels, dtype=torch.long).flatten()
    if original.shape != recovered.shape or len(y) != len(original):
        raise ValueError("Linkage inputs must contain the same samples")
    if manifest is None:
        manifest = make_same_digit_linkage_manifest(
            y,
            n_candidates=n_candidates,
            max_queries=max_queries,
            seed=seed,
        )
    if manifest.labels_hash != stable_hash(y.tolist(), 32):
        raise ValueError("Linkage manifest was built for different labels/order")
    n_candidates = manifest.n_candidates

    original_embeddings: Optional[Tensor] = None
    recovered_embeddings: Optional[Tensor] = None
    if auditor is not None:
        original_embeddings = F.normalize(
            _auditor_embeddings(auditor, original, device, batch_size), dim=1
        )
        recovered_embeddings = F.normalize(
            _auditor_embeddings(auditor, recovered, device, batch_size), dim=1
        )

    ranks: dict[str, list[int]] = {"pixel": [], "embedding": []}
    for query, frozen_candidates in zip(
        manifest.query_indices, manifest.candidate_indices
    ):
        candidates = np.asarray(frozen_candidates, dtype=np.int64)
        true_position = int(np.flatnonzero(candidates == query)[0])
        candidate_tensor = torch.from_numpy(candidates)
        pixel_distances = torch.linalg.vector_norm(
            original[candidate_tensor] - recovered[query], dim=1
        )
        order = torch.argsort(pixel_distances)
        ranks["pixel"].append(int(torch.nonzero(order == true_position)[0].item()) + 1)
        if original_embeddings is not None and recovered_embeddings is not None:
            cosine_distance = 1.0 - (
                original_embeddings[candidate_tensor] @ recovered_embeddings[query]
            )
            order = torch.argsort(cosine_distance)
            ranks["embedding"].append(
                int(torch.nonzero(order == true_position)[0].item()) + 1
            )

    result: dict[str, Any] = {
        "n_candidates": n_candidates,
        "chance_top1": 1.0 / n_candidates,
        "n_queries": len(ranks["pixel"]),
        "skipped_queries": 0,
        "manifest_hash": manifest.manifest_hash,
    }
    for name, observed in ranks.items():
        if not observed:
            continue
        rank_array = np.asarray(observed)
        result[f"{name}_top1"] = float(np.mean(rank_array == 1))
        # Persist record-level successes so the pilot can form a source-record
        # bootstrap interval instead of treating one aggregate proportion as
        # if it were a full sampling distribution.
        result[f"{name}_top1_successes"] = (rank_array == 1).astype(np.int8).tolist()
        result[f"{name}_top3"] = float(np.mean(rank_array <= min(3, n_candidates)))
        result[f"{name}_mrr"] = float(np.mean(1.0 / rank_array))
        result[f"{name}_mean_rank"] = float(np.mean(rank_array))
    return result


@dataclass(frozen=True)
class UniformNoiseManifest:
    """Named common-random-number manifest shared by every protocol.

    Each noisy cell draws its normalized magnitude as
    ``lambda ~ Uniform[0, lambda_upper]``. Conditional on lambda, tensor entries
    are iid standard Gaussian and are scaled later. The manifest stores seeds,
    rather than dense tensors, so runs are paired and restart-safe.
    """

    base_seed: int
    participants: tuple[int, ...]
    outer_seeds: tuple[int, ...]
    draws_per_seed: int
    splits: tuple[str, ...] = ("train", "val", "test", "anchor")
    roles: tuple[str, ...] = ("private", "anchor")

    def seed_for(
        self,
        split: str,
        participant: int,
        role: str = "private",
        *,
        outer_seed: int = 0,
        draw: int = 0,
        family: str = "default",
    ) -> int:
        if split not in self.splits:
            raise KeyError(f"Unknown split {split!r}")
        if participant not in self.participants:
            raise KeyError(f"Unknown participant {participant}")
        if role not in self.roles:
            raise KeyError(f"Unknown noise role {role!r}")
        if outer_seed not in self.outer_seeds:
            raise KeyError(f"Unknown outer seed {outer_seed}")
        if not 0 <= draw < self.draws_per_seed:
            raise KeyError(f"Draw {draw} is outside [0, {self.draws_per_seed})")
        return named_seed(
            self.base_seed, "noise", family, outer_seed, draw, split, participant, role
        )

    def lambda_value(
        self,
        upper: float,
        *,
        outer_seed: int,
        draw: int,
        role: str,
        family: str,
    ) -> float:
        """Draw lambda ~ Uniform[0, upper] with a named, restart-safe seed."""

        if upper < 0:
            raise ValueError("upper must be nonnegative")
        seed = named_seed(self.base_seed, "lambda", family, outer_seed, draw, role)
        return float(np.random.default_rng(seed).uniform(0.0, upper))

    def model_seed(self, outer_seed: int, draw: int, family: str) -> int:
        """Shared by declared equivalence pairs, including minibatch order."""

        return named_seed(self.base_seed, "model", family, outer_seed, draw)

    def standard_normal(
        self,
        shape: Sequence[int],
        split: str,
        participant: int,
        role: str = "private",
        *,
        outer_seed: int = 0,
        draw: int = 0,
        family: str = "default",
        dtype: torch.dtype = torch.float32,
        device: torch.device | str = "cpu",
    ) -> Tensor:
        generator = torch.Generator(device="cpu")
        generator.manual_seed(
            self.seed_for(
                split,
                participant,
                role,
                outer_seed=outer_seed,
                draw=draw,
                family=family,
            )
        )
        noise = torch.randn(tuple(shape), generator=generator, dtype=dtype)
        return noise.to(device)

    def to_dict(self) -> dict[str, Any]:
        return {
            "base_seed": self.base_seed,
            "participants": list(self.participants),
            "outer_seeds": list(self.outer_seeds),
            "draws_per_seed": self.draws_per_seed,
            "splits": list(self.splits),
            "roles": list(self.roles),
            "manifest_hash": stable_hash(
                {
                    "base_seed": self.base_seed,
                    "participants": self.participants,
                    "outer_seeds": self.outer_seeds,
                    "draws_per_seed": self.draws_per_seed,
                    "splits": self.splits,
                    "roles": self.roles,
                },
                32,
            ),
        }


def make_noise_manifest(config: Mapping[str, Any] | ExperimentConfig) -> UniformNoiseManifest:
    cfg = coerce_config(config)
    return UniformNoiseManifest(
        base_seed=named_seed(cfg.seed, "uniform_noise_manifest"),
        participants=tuple(range(cfg.n_participants)),
        outer_seeds=tuple(cfg.outer_seeds),
        draws_per_seed=cfg.draws_per_seed,
    )


@dataclass
class PilotResult:
    selected_level: float
    baseline_metric: float
    threshold: float
    evaluations: list[dict[str, Any]]
    split: str = "validation"

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def validation_only_upper_bound_pilot(
    evaluate_validation: Callable[[float], Mapping[str, Any]],
    candidate_levels: Sequence[float],
    *,
    metric: str = "accuracy",
    max_drop: float = 0.01,
    baseline_level: float = 0.0,
    output_path: Optional[str | Path] = None,
) -> PilotResult:
    """Choose the largest acceptable perturbation using validation data only.

    The callback intentionally receives only a scalar level; callers should
    close over validation tensors, never test tensors.  Candidates and the
    stopping rule are all evaluated, avoiding outcome-dependent early stopping.
    """

    ordered = sorted({float(level) for level in candidate_levels} | {float(baseline_level)})
    evaluations: list[dict[str, Any]] = []
    for level in ordered:
        values = dict(evaluate_validation(level))
        if metric not in values:
            raise KeyError(f"Pilot callback did not return metric {metric!r}")
        evaluations.append({"level": level, **values})
    baseline = next(item for item in evaluations if item["level"] == baseline_level)
    baseline_metric = float(baseline[metric])
    threshold = baseline_metric - float(max_drop)
    acceptable = [item["level"] for item in evaluations if float(item[metric]) >= threshold]
    selected = max(acceptable) if acceptable else float(baseline_level)
    result = PilotResult(selected, baseline_metric, threshold, evaluations)
    if output_path is not None:
        _atomic_write_json(
            Path(output_path),
            {**result.to_dict(), "test_used": False, "created_unix": time.time()},
        )
    return result


def _atomic_write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        "w", encoding="utf-8", delete=False, dir=path.parent, suffix=".tmp"
    ) as handle:
        json.dump(payload, handle, indent=2, sort_keys=True, default=_json_default)
        handle.write("\n")
        temporary = Path(handle.name)
    os.replace(temporary, path)


def _flatten_for_csv(value: Mapping[str, Any], prefix: str = "") -> dict[str, Any]:
    flattened: dict[str, Any] = {}
    for key, item in value.items():
        name = f"{prefix}.{key}" if prefix else str(key)
        if isinstance(item, Mapping):
            flattened.update(_flatten_for_csv(item, name))
        elif isinstance(item, (list, tuple, np.ndarray, Tensor)):
            flattened[name] = json.dumps(item, default=_json_default, separators=(",", ":"))
        elif dataclasses.is_dataclass(item):
            flattened.update(_flatten_for_csv(asdict(item), name))
        else:
            flattened[name] = item
    return flattened


class ResultStore:
    """Atomic JSON + CSV store keyed by deterministic run IDs."""

    def __init__(self, output_dir: str | Path, stem: str = "mnist_results") -> None:
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.json_path = self.output_dir / f"{stem}.json"
        self.csv_path = self.output_dir / f"{stem}.csv"
        if self.json_path.exists():
            with self.json_path.open("r", encoding="utf-8") as handle:
                loaded = json.load(handle)
            self.records: list[dict[str, Any]] = list(loaded.get("records", loaded))
        else:
            self.records = []
        self._ids = {str(record["run_id"]) for record in self.records if "run_id" in record}

    def contains(self, run_id: str) -> bool:
        return run_id in self._ids

    def get(self, run_id: str) -> Optional[dict[str, Any]]:
        return next((record for record in self.records if record.get("run_id") == run_id), None)

    def append(self, record: Mapping[str, Any]) -> bool:
        payload = copy.deepcopy(dict(record))
        if "run_id" not in payload:
            raise KeyError("Every result record needs a run_id")
        run_id = str(payload["run_id"])
        if run_id in self._ids:
            return False
        self.records.append(payload)
        self._ids.add(run_id)
        self.flush()
        return True

    def flush(self) -> None:
        _atomic_write_json(
            self.json_path,
            {"schema_version": 1, "records": self.records, "updated_unix": time.time()},
        )
        rows = [_flatten_for_csv(record) for record in self.records]
        fields = sorted({key for row in rows for key in row})
        if "run_id" in fields:
            fields.remove("run_id")
            fields.insert(0, "run_id")
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.NamedTemporaryFile(
            "w",
            newline="",
            encoding="utf-8",
            delete=False,
            dir=self.csv_path.parent,
            suffix=".tmp",
        ) as handle:
            writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(rows)
            temporary = Path(handle.name)
        os.replace(temporary, self.csv_path)


def make_run_id(**parts: Any) -> str:
    return stable_hash(parts, 24)


def paired_tost(
    left: Sequence[float] | np.ndarray,
    right: Sequence[float] | np.ndarray,
    margin: float,
    *,
    alpha: float = 0.05,
) -> dict[str, Any]:
    """Paired two-one-sided equivalence test on seed-level differences."""

    x = np.asarray(left, dtype=np.float64)
    y = np.asarray(right, dtype=np.float64)
    if x.shape != y.shape or x.ndim != 1:
        raise ValueError("left and right must be paired one-dimensional arrays")
    if len(x) < 2:
        raise ValueError("TOST requires at least two paired runs")
    if margin <= 0:
        raise ValueError("margin must be positive")
    difference = x - y
    mean = float(difference.mean())
    sd = float(difference.std(ddof=1))
    se = sd / math.sqrt(len(difference))
    try:
        from scipy.stats import t as student_t
    except ImportError as exc:
        raise RuntimeError("scipy is required for TOST") from exc
    critical = float(student_t.ppf(1.0 - alpha, df=len(difference) - 1))
    if se == 0.0:
        ci_low = ci_high = mean
        p_lower = 0.0 if mean > -margin else 1.0
        p_upper = 0.0 if mean < margin else 1.0
        t_lower = math.inf if mean > -margin else -math.inf
        t_upper = -math.inf if mean < margin else math.inf
    else:
        ci_low, ci_high = mean - critical * se, mean + critical * se
        t_lower = (mean + margin) / se
        t_upper = (mean - margin) / se
        p_lower = float(student_t.sf(t_lower, df=len(difference) - 1))
        p_upper = float(student_t.cdf(t_upper, df=len(difference) - 1))
    equivalent = bool(ci_low > -margin and ci_high < margin and max(p_lower, p_upper) < alpha)
    return {
        "n_pairs": int(len(difference)),
        "mean_difference": mean,
        "sd_difference": sd,
        "margin": float(margin),
        "alpha": float(alpha),
        "ci_level": float(1.0 - 2.0 * alpha),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "t_lower": float(t_lower),
        "t_upper": float(t_upper),
        "p_lower": float(p_lower),
        "p_upper": float(p_upper),
        "tost_p_max": float(max(p_lower, p_upper)),
        "equivalent": equivalent,
    }


def paired_bootstrap_equivalence(
    left: Sequence[float] | np.ndarray,
    right: Sequence[float] | np.ndarray,
    margin: float,
    *,
    alpha: float = 0.05,
    n_resamples: int = 10_000,
    seed: int = 0,
    method: str = "BCa",
) -> dict[str, Any]:
    """Paired seed-block bootstrap interval for the mean difference."""

    x = np.asarray(left, dtype=np.float64)
    y = np.asarray(right, dtype=np.float64)
    if x.shape != y.shape or x.ndim != 1 or len(x) < 2:
        raise ValueError("Bootstrap requires at least two paired observations")
    difference = x - y
    confidence_level = 1.0 - 2.0 * alpha
    used_method = method
    if np.allclose(difference, difference[0]):
        low = high = float(difference[0])
        used_method = "degenerate_exact"
    else:
        try:
            from scipy.stats import bootstrap

            result = bootstrap(
                (difference,),
                np.mean,
                confidence_level=confidence_level,
                n_resamples=n_resamples,
                method=method,
                random_state=np.random.default_rng(seed),
                vectorized=False,
            )
            low = float(result.confidence_interval.low)
            high = float(result.confidence_interval.high)
            if not (math.isfinite(low) and math.isfinite(high)):
                raise FloatingPointError("non-finite bootstrap interval")
        except (ImportError, ValueError, FloatingPointError, RuntimeWarning):
            rng = np.random.default_rng(seed)
            indices = rng.integers(0, len(difference), size=(n_resamples, len(difference)))
            estimates = difference[indices].mean(axis=1)
            low, high = np.quantile(estimates, [alpha, 1.0 - alpha]).tolist()
            used_method = "percentile_fallback"
    return {
        "n_pairs": int(len(difference)),
        "mean_difference": float(difference.mean()),
        "margin": float(margin),
        "alpha": float(alpha),
        "ci_level": float(confidence_level),
        "ci_low": float(low),
        "ci_high": float(high),
        "method": used_method,
        "n_resamples": int(n_resamples),
        "equivalent": bool(low > -margin and high < margin),
    }


def _nested_get(record: Mapping[str, Any], dotted_key: str) -> float:
    value: Any = record
    for part in dotted_key.split("."):
        value = value[part]
    return float(value)


def evaluate_equivalence_cells(
    records: Sequence[Mapping[str, Any]],
    left_method: str,
    right_method: str,
    metric_margins: Mapping[str, float],
    *,
    group_keys: Sequence[str] = ("h", "snr_db"),
    seed_key: str = "seed",
    method_key: str = "protocol",
    alpha: float = 0.05,
    bootstrap_resamples: int = 10_000,
    bootstrap_seed: int = 0,
) -> dict[str, Any]:
    """Run paired TOST/bootstrap on every prespecified metric/condition cell."""

    grouped: dict[tuple[Any, ...], dict[str, dict[Any, Mapping[str, Any]]]] = {}
    for record in records:
        method = str(record[method_key])
        if method not in {left_method, right_method}:
            continue
        group = tuple(record.get(key) for key in group_keys)
        grouped.setdefault(group, {}).setdefault(method, {})[record[seed_key]] = record
    cells: list[dict[str, Any]] = []
    for group, by_method in sorted(grouped.items(), key=lambda item: str(item[0])):
        left_by_seed = by_method.get(left_method, {})
        right_by_seed = by_method.get(right_method, {})
        seeds = sorted(set(left_by_seed) & set(right_by_seed), key=str)
        for metric, margin in metric_margins.items():
            left_values = [_nested_get(left_by_seed[seed], metric) for seed in seeds]
            right_values = [_nested_get(right_by_seed[seed], metric) for seed in seeds]
            if len(seeds) < 2:
                cells.append(
                    {
                        "group": dict(zip(group_keys, group)),
                        "metric": metric,
                        "margin": margin,
                        "n_pairs": len(seeds),
                        "equivalent": False,
                        "reason": "fewer_than_two_paired_runs",
                    }
                )
                continue
            tost = paired_tost(left_values, right_values, margin, alpha=alpha)
            boot = paired_bootstrap_equivalence(
                left_values,
                right_values,
                margin,
                alpha=alpha,
                n_resamples=bootstrap_resamples,
                seed=named_seed(bootstrap_seed, group, metric),
            )
            cells.append(
                {
                    "group": dict(zip(group_keys, group)),
                    "metric": metric,
                    "margin": margin,
                    "seeds": seeds,
                    "tost": tost,
                    "bootstrap": boot,
                    "equivalent": bool(tost["equivalent"] and boot["equivalent"]),
                }
            )
    return {
        "left_method": left_method,
        "right_method": right_method,
        "alpha": alpha,
        "all_cells_equivalent": bool(cells) and all(cell["equivalent"] for cell in cells),
        "cells": cells,
    }


def relative_frobenius_error(left: Tensor, right: Tensor) -> float:
    # Diagnostic callers may legitimately combine a CPU-side known-anchor
    # estimate with a CUDA protocol transcript.  Preserve the left operand's
    # device and co-locate only the right operand; same-device comparisons of
    # large collaboration matrices therefore incur no transfer.
    left = torch.as_tensor(left, dtype=torch.float64)
    right = torch.as_tensor(right, dtype=torch.float64, device=left.device)
    if left.shape != right.shape:
        raise ValueError("Frobenius comparison shapes differ")
    return float(
        torch.linalg.vector_norm(left - right).item()
        / max(torch.linalg.vector_norm(right).item(), 1e-15)
    )


def affine_orthogonal_registration_error(source: Tensor, target: Tensor) -> dict[str, Any]:
    """Register two row-oriented representations by one affine-orthogonal map."""

    x, y = torch.as_tensor(source).double(), torch.as_tensor(target).double()
    if x.shape != y.shape or x.ndim != 2:
        raise ValueError("Representations must be same-shaped matrices")
    mean_x, mean_y = x.mean(dim=0, keepdim=True), y.mean(dim=0, keepdim=True)
    centered_x, centered_y = x - mean_x, y - mean_y
    u, _, vh = torch.linalg.svd(centered_x.T @ centered_y, full_matrices=False)
    rotation = u @ vh
    translation = mean_y - mean_x @ rotation
    registered = x @ rotation + translation
    return {
        "relative_frobenius_error": relative_frobenius_error(registered, y),
        "rotation": rotation,
        "translation": translation.flatten(),
        "registered": registered,
    }


# ---------------------------------------------------------------------------
# Notebook-facing end-to-end orchestration


def _core_protocols_module() -> Any:
    try:
        from . import core_protocols as core
    except ImportError:
        import core_protocols as core  # type: ignore[no-redef]
    return core


def _sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def print_config(config: Mapping[str, Any] | ExperimentConfig) -> dict[str, Any]:
    """Print and return the resolved, derived experiment configuration."""

    cfg = coerce_config(config)
    payload = {
        **cfg.to_dict(),
        "reduced_dim": cfg.reduced_dim,
        "anchor_rows": cfg.anchor_rows,
        "noise_distribution": "lambda ~ Uniform[0, lambda_upper]",
        "smoke_bounds_are_debug_only": cfg.mode == "smoke",
    }
    print(json.dumps(payload, indent=2, sort_keys=True, default=_json_default))
    return payload


def _participant_view(data: MNISTDataBundle, split_name: str) -> dict[str, Any]:
    z, labels, indices = data.participant_blocks(split_name, reduced=True)
    split = data.split(split_name)
    return {
        "z": [item.double().cpu() for item in z],
        "labels": labels,
        "indices": indices,
        "raw": [split.raw[index] for index in indices],
        "ordered_labels": torch.cat(labels),
        "ordered_raw": torch.cat([split.raw[index] for index in indices]),
    }


def _make_anchor_and_parameters(
    core: Any, cfg: ExperimentConfig, outer_seed: int
) -> tuple[Tensor, list[Tensor], list[Tensor]]:
    generator = core.make_generator(
        named_seed(cfg.seed, "protocol_parameters", outer_seed), "cpu"
    )
    raw_anchor = torch.randn(
        (cfg.anchor_rows, cfg.reduced_dim),
        generator=generator,
        dtype=core.PROTOCOL_DTYPE,
    )
    raw_anchor -= raw_anchor.mean(dim=0, keepdim=True)
    anchor, r_factor = torch.linalg.qr(raw_anchor, mode="reduced")
    diagonal = torch.diagonal(r_factor)
    signs = torch.where(diagonal < 0, -torch.ones_like(diagonal), torch.ones_like(diagonal))
    anchor = anchor * signs.unsqueeze(0)
    rotations, translations = core.sample_gdp_parameters(
        cfg.n_participants,
        cfg.reduced_dim,
        generator=generator,
        device="cpu",
    )
    return anchor, rotations, translations


def _apply_protocol(
    core: Any,
    method: str,
    blocks: Sequence[Tensor],
    split_name: str,
    anchor: Tensor,
    rotations: Sequence[Tensor],
    translations: Sequence[Tensor],
    manifest: UniformNoiseManifest,
    cfg: ExperimentConfig,
    outer_seed: int,
    draw: int,
    private_lambda: float,
    anchor_lambda: float,
    reduced_scale: float,
    cached_pa: Optional[Any] = None,
) -> Any:
    reference = cfg.reference_participant
    if method == "c_gdp":
        return core.c_gdp(
            blocks, rotation=rotations[reference], translation=translations[reference]
        )
    if method in {"c_gdp_an", "aa_i_gdp_an"}:
        sigma = private_lambda * reduced_scale
        common_noise = [
            sigma
            * manifest.standard_normal(
                block.shape,
                split_name,
                participant,
                "private",
                outer_seed=outer_seed,
                draw=draw,
                family="c_aa_private",
                dtype=block.dtype,
            )
            for participant, block in enumerate(blocks)
        ]
        if method == "c_gdp_an":
            return core.c_gdp_an(
                blocks,
                sigma,
                rotation=rotations[reference],
                translation=translations[reference],
                noises=common_noise,
            )
        relative = [rotation.mT @ rotations[reference] for rotation in rotations]
        local_noise = [noise @ alignment.mT for noise, alignment in zip(common_noise, relative)]
        return core.aa_i_gdp_an(
            blocks,
            anchor,
            sigma,
            reference=reference,
            rotations=rotations,
            translations=translations,
            noises=local_noise,
        )
    if method == "i_gdp":
        return core.i_gdp(blocks, rotations=rotations, translations=translations)
    if method == "aa_i_gdp":
        return core.aa_i_gdp(
            blocks,
            anchor,
            reference=reference,
            rotations=rotations,
            translations=translations,
        )
    if method == "pa_i_gdp":
        if cached_pa is not None:
            uploads = [x @ o + psi for x, o, psi in zip(blocks, rotations, translations)]
            means = [b.mean(dim=0) for b in cached_pa.anchor_uploads]
            aligned = [
                (upload - mean) @ rotation
                for upload, mean, rotation in zip(
                    uploads, means, cached_pa.alignment_rotations
                )
            ]
            return core.ProtocolResult(
                name="PA-I-GDP",
                private_uploads=uploads,
                collaboration_blocks=aligned,
                collaboration=torch.cat(aligned),
                rotations=list(rotations),
                translations=list(translations),
                anchor_uploads=cached_pa.anchor_uploads,
                alignment_rotations=cached_pa.alignment_rotations,
                anchor_noises=cached_pa.anchor_noises,
                diagnostics=cached_pa.diagnostics,
            )
        sigma = anchor_lambda
        anchor_noise = [
            sigma
            * manifest.standard_normal(
                anchor.shape,
                "anchor",
                participant,
                "anchor",
                outer_seed=outer_seed,
                draw=draw,
                family="pa_anchor",
                dtype=anchor.dtype,
            )
            for participant in range(cfg.n_participants)
        ]
        return core.pa_i_gdp(
            blocks,
            anchor,
            sigma,
            rotations=rotations,
            translations=translations,
            anchor_noises=anchor_noise,
            max_iters=cfg.pa_iterations,
            tol=cfg.pa_tolerance,
        )
    raise KeyError(f"Unknown protocol {method!r}")


def _evaluate_trained_model(
    model: nn.Module,
    x: Tensor,
    y: Tensor,
    cfg: ExperimentConfig,
    seed: int,
) -> dict[str, Any]:
    loader = _make_loader(
        as_square_images(x, cfg.cnn_side),
        y,
        cfg.eval_batch_size,
        shuffle=False,
        seed=seed,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )
    return evaluate_model(model, loader, resolve_device(cfg.device))


def _score_attacks(
    core: Any,
    method: str,
    train_result: Any,
    test_result: Any,
    train_view: Mapping[str, Any],
    test_view: Mapping[str, Any],
    anchor: Tensor,
    reducer: PublicReducer,
    auditor: CleanMNISTAuditor,
    linkage_manifest: LinkageManifest,
    cfg: ExperimentConfig,
    seed: int,
) -> tuple[dict[str, Any], Optional[dict[str, Any]]]:
    target = 1 if cfg.n_participants > 1 else 0
    attacks: dict[str, Any] = {}
    if method in {"c_gdp", "c_gdp_an"}:
        for estimator in ("exact", "mp", "op"):
            attacks[f"known_input_{estimator}"] = core.attack_c_known_input(
                train_view["z"][0],
                train_result.private_uploads[0],
                test_result.private_uploads[target],
                estimator=estimator,
            )
    elif method == "i_gdp":
        attacks["colluder_transfer_negative_control"] = core.attack_c_known_input(
            train_view["z"][0],
            train_result.private_uploads[0],
            test_result.private_uploads[target],
            estimator="op",
        )
    elif method in {"aa_i_gdp", "aa_i_gdp_an"}:
        attacks["known_anchor"] = core.attack_aa_known_anchor(
            anchor,
            test_result.anchor_uploads[target],
            test_result.private_uploads[target],
        )
    elif method == "pa_i_gdp":
        attacks["mp"] = core.attack_pa_mp(
            anchor, test_result.anchor_uploads[target], test_result.private_uploads[target]
        )
        attacks["op"] = core.attack_pa_op(
            anchor, test_result.anchor_uploads[target], test_result.private_uploads[target]
        )
        attacks["alignment_calibration"] = core.attack_pa_alignment_calibration(
            anchor,
            test_result.anchor_uploads[target],
            test_result.private_uploads[target],
            test_result.alignment_rotations[target],
            [test_result.rotations[0]],
            [test_result.alignment_rotations[0]],
        )
    original_z = test_view["z"][target].float()
    original_raw = test_view["raw"][target].float()
    labels = test_view["labels"][target]
    scored: dict[str, Any] = {}
    for name, attack in attacks.items():
        recovered_z = attack.reduced.float()
        recovered_raw = reducer.decode(recovered_z).cpu()
        reduced = reduced_reconstruction_metrics(original_z, recovered_z)
        pixel = compute_reconstruction_metrics(
            original_raw,
            recovered_raw,
            ssim_max_images=cfg.ssim_max_images,
            seed=named_seed(seed, "ssim", name),
        )
        semantic = auditor_semantic_metrics(
            auditor,
            recovered_raw,
            labels,
            device=resolve_device(cfg.device),
            batch_size=cfg.eval_batch_size,
        )
        linkage = same_digit_source_linkage(
            original_raw,
            recovered_raw,
            labels,
            auditor,
            manifest=linkage_manifest,
            device=resolve_device(cfg.device),
            batch_size=cfg.eval_batch_size,
        )
        metrics = {
            "pixel_nrmse": pixel["nrmse"],
            "reduced_nrmse": reduced["reduced_nrmse"],
            "auditor_accuracy": semantic["auditor_accuracy"],
            "auditor_macro_f1": semantic["auditor_macro_f1"],
        }
        scored[name] = {
            "method": attack.method,
            "metrics": metrics,
            "pixel": pixel,
            "reduced": reduced,
            "semantic": semantic,
            "linkage": linkage,
        }
    if not scored:
        return scored, None
    best_name = min(scored, key=lambda key: scored[key]["metrics"]["pixel_nrmse"])
    return scored, {"name": best_name, **scored[best_name]}


def run_experiment(config: Mapping[str, Any] | ExperimentConfig, output_dir: str) -> dict[str, Any]:
    """Run the six-method MNIST experiment and persist each completed cell."""

    cfg = replace(coerce_config(config), output_dir=str(output_dir))
    if cfg.mode == "pilot":
        raise RuntimeError(
            "Pilot calibration must be run through validation_only_upper_bound_pilot; "
            "the six-method runner intentionally cannot access test data in pilot mode."
        )
    if cfg.mode == "full":
        if not cfg.pilot_manifest_path or not cfg.pilot_manifest_sha256:
            raise ValueError("Full mode requires a frozen pilot manifest path and SHA-256")
        actual = _sha256_file(cfg.pilot_manifest_path)
        if actual.lower() != cfg.pilot_manifest_sha256.lower():
            raise ValueError("Frozen pilot manifest checksum mismatch")
        with Path(cfg.pilot_manifest_path).open("r", encoding="utf-8") as handle:
            pilot = json.load(handle)
        if pilot.get("test_used") is not False:
            raise ValueError("Pilot manifest must attest test_used=false")
        if pilot.get("anchor_design") != cfg.anchor_design:
            raise ValueError("Frozen pilot anchor design does not match this run")
        cfg = replace(
            cfg,
            private_lambda_upper=float(pilot["private_lambda_upper"]),
            anchor_lambda_upper=float(pilot["anchor_lambda_upper"]),
        )
    if cfg.private_lambda_upper is None or cfg.anchor_lambda_upper is None:
        raise ValueError("Both normalized noise upper bounds must be resolved")

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    _atomic_write_json(output / "resolved_config.json", print_config(cfg))
    core = _core_protocols_module()
    self_tests = core.run_self_tests(device="cpu", seed=named_seed(cfg.seed, "core_tests"))
    _atomic_write_json(output / "algebraic_self_tests.json", self_tests)
    data = prepare_mnist_data(cfg, include_test=True)
    _atomic_write_json(output / "data_manifest.json", data.manifest)
    noise_manifest = make_noise_manifest(cfg)
    _atomic_write_json(output / "noise_manifest.json", noise_manifest.to_dict())
    auditor, auditor_metadata = train_or_load_auditor(
        data, cfg, checkpoint_path=output / "clean_mnist_auditor.pt"
    )
    _atomic_write_json(output / "auditor_metrics.json", auditor_metadata)
    views = {name: _participant_view(data, name) for name in ("train", "val", "test")}
    target = 1 if cfg.n_participants > 1 else 0
    linkage_manifest = make_same_digit_linkage_manifest(
        views["test"]["labels"][target],
        n_candidates=cfg.linkage_candidates,
        max_queries=cfg.linkage_max_queries,
        seed=named_seed(cfg.seed, "frozen_linkage", target),
    )
    _atomic_write_json(output / "linkage_manifest.json", linkage_manifest.to_dict())
    store = ResultStore(output)
    reduced_scale = float(data.train.reduced.std(unbiased=False).item())
    noisy = {"c_gdp_an", "aa_i_gdp_an", "pa_i_gdp"}

    for outer_seed in cfg.outer_seeds:
        anchor, rotations, translations = _make_anchor_and_parameters(core, cfg, outer_seed)
        for method in cfg.protocols:
            draws = range(cfg.draws_per_seed) if method in noisy else range(1)
            for draw in draws:
                family = (
                    "c_aa_private"
                    if method in {"c_gdp_an", "aa_i_gdp_an"}
                    else "c_aa_clean"
                    if method in {"c_gdp", "aa_i_gdp"}
                    else method
                )
                private_lambda = (
                    noise_manifest.lambda_value(
                        cfg.private_lambda_upper,
                        outer_seed=outer_seed,
                        draw=draw,
                        role="private",
                        family="c_aa_private",
                    )
                    if method in {"c_gdp_an", "aa_i_gdp_an"}
                    else 0.0
                )
                anchor_lambda = (
                    noise_manifest.lambda_value(
                        cfg.anchor_lambda_upper,
                        outer_seed=outer_seed,
                        draw=draw,
                        role="anchor",
                        family="pa_anchor",
                    )
                    if method == "pa_i_gdp"
                    else 0.0
                )
                run_id = make_run_id(
                    config=stable_hash(cfg.to_dict(), 32),
                    data=stable_hash(data.manifest, 32),
                    method=method,
                    outer_seed=outer_seed,
                    draw=draw,
                    private_lambda=private_lambda,
                    anchor_lambda=anchor_lambda,
                )
                if store.contains(run_id):
                    continue
                protocol_results: dict[str, Any] = {}
                for split_name in ("train", "val", "test"):
                    protocol_results[split_name] = _apply_protocol(
                        core,
                        method,
                        views[split_name]["z"],
                        split_name,
                        anchor,
                        rotations,
                        translations,
                        noise_manifest,
                        cfg,
                        outer_seed,
                        draw,
                        private_lambda,
                        anchor_lambda,
                        reduced_scale,
                        protocol_results.get("train") if method == "pa_i_gdp" else None,
                    )
                model_seed = noise_manifest.model_seed(outer_seed, draw, family)
                set_global_determinism(model_seed, cfg.deterministic)
                model = ReferencePaperCNN(
                    cfg.cnn_side,
                    minimum_input_side=cfg.cnn_minimum_input_side,
                )
                trained = train_classifier(
                    model,
                    as_square_images(protocol_results["train"].collaboration.float(), cfg.cnn_side),
                    views["train"]["ordered_labels"],
                    as_square_images(protocol_results["val"].collaboration.float(), cfg.cnn_side),
                    views["val"]["ordered_labels"],
                    cfg,
                    seed=model_seed,
                    checkpoint_path=(output / "models" / f"{run_id}.pt") if cfg.save_models else None,
                )
                utility: dict[str, Any] = {}
                for split_name in ("val", "test"):
                    metrics = _evaluate_trained_model(
                        trained["model"],
                        protocol_results[split_name].collaboration.float(),
                        views[split_name]["ordered_labels"],
                        cfg,
                        named_seed(model_seed, split_name, "eval"),
                    )
                    utility.update({f"{split_name}_{key}": value for key, value in metrics.items()})
                attacks, best_attack = ({}, None)
                if cfg.run_attacks:
                    attacks, best_attack = _score_attacks(
                        core,
                        method,
                        protocol_results["train"],
                        protocol_results["test"],
                        views["train"],
                        views["test"],
                        anchor,
                        data.reducer,
                        auditor,
                        linkage_manifest,
                        cfg,
                        named_seed(cfg.seed, outer_seed, draw, method, "attacks"),
                    )
                record = {
                    "run_id": run_id,
                    "mode": cfg.mode,
                    "protocol": protocol_results["train"].name,
                    "method_key": method,
                    "outer_seed": outer_seed,
                    "seed": outer_seed,
                    "draw": draw,
                    "pair_id": f"{family}:{outer_seed}:{draw}",
                    "cnn_side": cfg.cnn_side,
                    "cnn_minimum_input_side": cfg.cnn_minimum_input_side,
                    "reduced_dim": cfg.reduced_dim,
                    "anchor_rows": cfg.anchor_rows,
                    "private_lambda": private_lambda,
                    "anchor_lambda": anchor_lambda,
                    "private_lambda_upper": cfg.private_lambda_upper,
                    "anchor_lambda_upper": cfg.anchor_lambda_upper,
                    "normalized_reduced_scale": reduced_scale,
                    "utility": utility,
                    "attacks": attacks,
                    "best_attack": best_attack,
                    "created_unix": time.time(),
                }
                store.append(record)

    equivalence: dict[str, Any] = {}
    if cfg.run_equivalence:
        for left, right, key in (
            ("C-GDP", "AA-I-GDP", "clean"),
            ("C-GDP-AN", "AA-I-GDP-AN", "private_noise"),
        ):
            equivalence[key] = evaluate_equivalence_cells(
                store.records,
                left,
                right,
                cfg.equivalence_margins,
                group_keys=("draw",),
                seed_key="outer_seed",
                alpha=cfg.equivalence_alpha,
                bootstrap_resamples=cfg.bootstrap_resamples,
                bootstrap_seed=named_seed(cfg.seed, "equivalence", key),
            )
    _atomic_write_json(output / "equivalence_gates.json", equivalence)
    result = {
        "config": print_config(cfg),
        "data_manifest": data.manifest,
        "noise_manifest": noise_manifest.to_dict(),
        "auditor": auditor_metadata,
        "algebraic_self_tests": self_tests,
        "records": store.records,
        "equivalence": equivalence,
        "paths": {"json": str(store.json_path), "csv": str(store.csv_path)},
    }
    result["report"] = render_report(result, output)
    return result


def render_report(result: Mapping[str, Any], output_dir: str | Path) -> dict[str, Any]:
    """Aggregate method-level means and write compact JSON/Markdown reports."""

    output = Path(output_dir)
    records = list(result.get("records", []))
    grouped: dict[str, list[Mapping[str, Any]]] = {}
    for record in records:
        grouped.setdefault(str(record.get("protocol", "unknown")), []).append(record)
    methods: dict[str, Any] = {}
    for method, rows in sorted(grouped.items()):
        numeric_keys = (
            "test_accuracy",
            "test_balanced_accuracy",
            "test_macro_f1",
            "test_macro_ovr_auroc",
        )
        summary = {"n_runs": len(rows)}
        for key in numeric_keys:
            values = [float(row["utility"][key]) for row in rows if key in row.get("utility", {})]
            if values:
                summary[f"mean_{key}"] = float(np.mean(values))
                summary[f"sd_{key}"] = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
        attack_values = [
            float(row["best_attack"]["metrics"]["pixel_nrmse"])
            for row in rows
            if row.get("best_attack")
        ]
        if attack_values:
            summary["mean_best_attack_pixel_nrmse"] = float(np.mean(attack_values))
        methods[method] = summary
    summary_payload = {
        "n_records": len(records),
        "methods": methods,
        "equivalence": result.get("equivalence", {}),
        "generated_unix": time.time(),
    }
    json_path = output / "mnist_report.json"
    markdown_path = output / "mnist_report.md"
    _atomic_write_json(json_path, summary_payload)
    lines = ["# MNIST empirical experiment report", "", f"Completed cells: {len(records)}", ""]
    lines.append("| Method | Runs | Test balanced accuracy | Best-attack pixel NRMSE |")
    lines.append("|---|---:|---:|---:|")
    for method, values in methods.items():
        lines.append(
            f"| {method} | {values['n_runs']} | "
            f"{values.get('mean_test_balanced_accuracy', float('nan')):.4f} | "
            f"{values.get('mean_best_attack_pixel_nrmse', float('nan')):.4f} |"
        )
    markdown_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return {
        "summary": summary_payload,
        "json_path": str(json_path),
        "markdown_path": str(markdown_path),
    }


def _main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=("smoke", "pilot", "full"), default="smoke")
    parser.add_argument("--output-dir", default="/content/igdppa_results")
    parser.add_argument("--data-root", default="/content/data")
    parser.add_argument("--device", default="auto")
    args = parser.parse_args()
    config = make_config(args.mode, data_root=args.data_root, device=args.device)
    run_experiment(config, args.output_dir)


if __name__ == "__main__":
    _main()


In [ ]:
%%writefile mnist_orchestrator.py
"""Notebook-facing orchestration for the MNIST I-GDP smoke benchmark.

This file intentionally imports the data/model/statistics layer as ``mr`` and
the float64 protocol layer as ``cp``.  Its three public functions can be copied
into ``mnist_runner.py`` after review without changing either implementation.
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import time
from dataclasses import asdict, replace
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

import core_protocols as cp
import mnist_runner as mr


METHODS = ("c_gdp", "c_gdp_an", "i_gdp", "aa_i_gdp", "aa_i_gdp_an", "pa_i_gdp")

PILOT_SCHEMA_VERSION = 5
PILOT_CHANCE_ACCURACY = 0.10
PILOT_BASELINE_EPSILON = 1.0e-6


_CONDITIONAL_METRICS: tuple[dict[str, str], ...] = (
    {
        "key": "utility.test_balanced_accuracy",
        "label": "Test balanced accuracy",
        "family": "utility",
        "scale": "linear",
    },
    {
        "key": "utility.test_accuracy",
        "label": "Test accuracy",
        "family": "utility",
        "scale": "linear",
    },
    {
        "key": "utility.test_macro_f1",
        "label": "Test macro-F1",
        "family": "utility",
        "scale": "linear",
    },
    {
        "key": "best_attack.metrics.linkage_embedding_top1",
        "label": "10-way embedding top-1 linkage",
        "family": "privacy",
        "scale": "linear",
    },
    {
        "key": "best_attack.metrics.linkage_pixel_top1",
        "label": "10-way pixel top-1 linkage",
        "family": "privacy",
        "scale": "linear",
    },
    {
        "key": "best_attack.metrics.pixel_nrmse",
        "label": "Recovered-image pixel NRMSE",
        "family": "privacy",
        "scale": "linear",
    },
    {
        "key": "gpm.relative_gopp_residual",
        "label": "Relative GOPP residual",
        "family": "gpm",
        "scale": "log",
    },
    {
        "key": "gpm.anchor_sigma_min",
        "label": "Anchor smallest singular value",
        "family": "gpm",
        "scale": "linear",
    },
    {
        "key": "gpm.anchor_condition",
        "label": "Anchor condition number",
        "family": "gpm",
        "scale": "log",
    },
)


def _write_or_validate_frozen_json(
    path: Path,
    payload: Mapping[str, Any],
    *,
    label: str,
) -> None:
    """Write a frozen manifest once and reject incompatible resume attempts."""

    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        if mr.stable_hash(existing, 32) != mr.stable_hash(payload, 32):
            raise RuntimeError(
                f"Refusing to resume with a different {label}: {path}"
            )
        return
    mr._atomic_write_json(path, payload)


def _source_sha256(path: str | Path) -> str:
    source = Path(path)
    if not source.is_file():
        raise FileNotFoundError(f"Cannot fingerprint missing implementation source: {source}")
    return hashlib.sha256(source.read_bytes()).hexdigest()


def _implementation_sha256() -> dict[str, str]:
    return {
        "core_protocols": _source_sha256(cp.__file__),
        "mnist_runner": _source_sha256(mr.__file__),
        "mnist_orchestrator": _source_sha256(__file__),
    }


def _make_centered_isometric_anchor(
    cfg: mr.ExperimentConfig, outer_seed: int
) -> tuple[torch.Tensor, dict[str, Any]]:
    """Return the frozen centered anchor with unit singular values."""

    rows, h = cfg.anchor_rows, cfg.reduced_dim
    if rows - 1 < h:
        raise ValueError("A centered r-by-h isometric anchor requires r-1 >= h")
    generator = torch.Generator(device="cpu").manual_seed(
        mr.named_seed(cfg.seed, "centered_isometric_anchor", outer_seed)
    )
    raw = torch.randn((rows, h), generator=generator, dtype=torch.float64)
    centered = raw - raw.mean(dim=0, keepdim=True)
    q, r_factor = torch.linalg.qr(centered, mode="reduced")
    diagonal = torch.diagonal(r_factor)
    signs = torch.where(diagonal < 0, -torch.ones_like(diagonal), torch.ones_like(diagonal))
    q = q * signs.unsqueeze(0)
    scale = 1.0
    anchor = q.contiguous()
    target_gram = torch.eye(h, dtype=torch.float64)
    mean_residual = float(torch.linalg.vector_norm(anchor.mean(0)).item())
    gram_relative_error = float(
        torch.linalg.matrix_norm(anchor.mT @ anchor - target_gram).item()
        / torch.linalg.matrix_norm(target_gram).item()
    )
    singular = torch.linalg.svdvals(anchor)
    anchor_hash = hashlib.sha256(anchor.numpy().tobytes(order="C")).hexdigest()
    diagnostics = {
        "design": cfg.anchor_design,
        "scale_rule": cfg.anchor_scale_rule,
        "seed": int(mr.named_seed(cfg.seed, "centered_isometric_anchor", outer_seed)),
        "outer_seed": int(outer_seed),
        "rows": int(rows),
        "dimension": int(h),
        "scale": float(scale),
        "alpha": 1.0,
        "rms": float(torch.sqrt(anchor.square().mean()).item()),
        "expected_rms": float(1.0 / math.sqrt(rows)),
        "mean_residual_l2": mean_residual,
        "gram_relative_error": gram_relative_error,
        "sigma_min": float(singular[-1].item()),
        "sigma_max": float(singular[0].item()),
        "expected_sigma_min": 1.0,
        "expected_sigma_max": 1.0,
        "condition_number": float((singular[0] / singular[-1]).item()),
        "sha256_float64_c_order": anchor_hash,
    }
    if (
        mean_residual > 1.0e-10
        or gram_relative_error > 1.0e-10
        or abs(float(singular[-1].item()) - 1.0) > 1.0e-10
        or abs(float(singular[0].item()) - 1.0) > 1.0e-10
    ):
        raise RuntimeError(f"Centered isometric anchor validation failed: {diagnostics}")
    return anchor, diagnostics


def _pilot_identity(cfg: mr.ExperimentConfig) -> dict[str, Any]:
    """Return the frozen identity used to make pilot resume fail closed."""

    implementation = _implementation_sha256()
    payload = {
        "schema_version": PILOT_SCHEMA_VERSION,
        "config": cfg.to_dict(),
        "implementation_sha256": implementation,
    }
    return {**payload, "pilot_fingerprint": mr.stable_hash(payload, 32)}


def print_config(config: Mapping[str, Any] | mr.ExperimentConfig) -> dict[str, Any]:
    """Print and return the effective, fully expanded experiment configuration."""

    cfg = mr.coerce_config(config)
    payload = cfg.to_dict()
    print(json.dumps(payload, indent=2, sort_keys=True, default=mr._json_default))
    random_and_clean = sum(
        1 if m in {"c_gdp", "i_gdp", "aa_i_gdp"} else cfg.draws_per_seed
        for m in cfg.protocols
    ) * len(cfg.outer_seeds)
    endpoint_fits = (
        2
        * sum(m in {"c_gdp_an", "aa_i_gdp_an", "pa_i_gdp"} for m in cfg.protocols)
        * len(cfg.outer_seeds)
        if cfg.mode == "full" else 0
    )
    planned_fits = random_and_clean + endpoint_fits
    print(
        f"\nmode={cfg.mode}; device={mr.resolve_device(cfg.device)}; "
        f"p={cfg.n_participants}; reduced dimension={cfg.reduced_dim} "
        f"({cfg.cnn_side}x{cfg.cnn_side}); anchor rows={cfg.anchor_rows}; "
        f"planned fits={planned_fits}"
    )
    return payload


def _blocks(data: mr.MNISTDataBundle, split: str, device: torch.device | str = "cpu"):
    x, y, idx = data.participant_blocks(split, reduced=True)
    return [v.to(device=device, dtype=torch.float64) for v in x], y, idx


def _fixed_aa(
    name: str,
    blocks: Sequence[torch.Tensor],
    template: cp.ProtocolResult,
    noises: Sequence[torch.Tensor] | None = None,
) -> cp.ProtocolResult:
    means = [b.mean(0) for b in template.anchor_uploads or []]
    ref_mean = means[int(template.reference or 0)]
    uploads = [x @ o + psi for x, o, psi in zip(blocks, template.rotations, template.translations)]
    if noises is not None:
        uploads = [y + e for y, e in zip(uploads, noises)]
    aligned = [
        (y - b) @ r + ref_mean
        for y, b, r in zip(uploads, means, template.alignment_rotations or [])
    ]
    return cp.ProtocolResult(
        name=name,
        private_uploads=uploads,
        collaboration_blocks=aligned,
        collaboration=torch.cat(aligned),
        rotations=template.rotations,
        translations=template.translations,
        anchor_uploads=template.anchor_uploads,
        alignment_rotations=template.alignment_rotations,
        private_noises=None if noises is None else list(noises),
        reference=template.reference,
    )


def _fixed_pa(blocks: Sequence[torch.Tensor], template: cp.ProtocolResult) -> cp.ProtocolResult:
    means = [b.mean(0) for b in template.anchor_uploads or []]
    uploads = [x @ o + psi for x, o, psi in zip(blocks, template.rotations, template.translations)]
    aligned = [(y - b) @ r for y, b, r in zip(uploads, means, template.alignment_rotations or [])]
    return cp.ProtocolResult(
        name="PA-I-GDP",
        private_uploads=uploads,
        collaboration_blocks=aligned,
        collaboration=torch.cat(aligned),
        rotations=template.rotations,
        translations=template.translations,
        anchor_uploads=template.anchor_uploads,
        alignment_rotations=template.alignment_rotations,
        anchor_noises=template.anchor_noises,
        diagnostics=template.diagnostics,
    )


def _protocol_cell(
    cfg,
    data,
    manifest,
    anchor,
    outer_seed: int,
    draw: int,
    lx: float,
    la: float,
    run_role: str = "random",
    active_methods: Sequence[str] | None = None,
):
    active = set(active_methods or METHODS)
    unknown = active.difference(METHODS)
    if unknown:
        raise ValueError(f"Unknown active methods: {sorted(unknown)}")
    protocol_device = mr.resolve_device(cfg.device)
    split_blocks = {
        s: _blocks(data, s, protocol_device) for s in ("train", "val", "test")
    }
    h, p = cfg.reduced_dim, cfg.n_participants
    device = split_blocks["train"][0][0].device
    generator = cp.make_generator(mr.named_seed(cfg.seed, "gdp", outer_seed), device)
    rotations, translations = cp.sample_gdp_parameters(p, h, generator=generator, device=device)
    ref = cfg.reference_participant
    common_o, common_psi = rotations[ref], translations[ref]
    s_x = float(torch.sqrt(data.train.reduced.double().square().mean()).item())
    sigma, v = lx * s_x, la

    need_aa = bool(active.intersection({"aa_i_gdp", "aa_i_gdp_an"}))
    need_private_noise = bool(active.intersection({"c_gdp_an", "aa_i_gdp_an"}))
    need_pa = "pa_i_gdp" in active
    clean_aa_train = (
        cp.aa_i_gdp(
            split_blocks["train"][0], anchor, reference=ref,
            rotations=rotations, translations=translations,
        )
        if need_aa else None
    )
    use_multistart = bool(
        cfg.mode == "smoke"
        or run_role.startswith("endpoint_")
        or (
            cfg.mode == "full"
            and run_role == "random"
            and draw % 5 == 0
        )
    )
    pa_train = None
    if need_pa:
        anchor_noise = [
            v * manifest.standard_normal(
                anchor.shape, "anchor", i, "anchor", outer_seed=outer_seed, draw=draw,
                family="pa", dtype=torch.float64, device=device,
            ) for i in range(p)
        ]
        pa_train = cp.pa_i_gdp(
            split_blocks["train"][0], anchor, v,
            rotations=rotations, translations=translations, anchor_noises=anchor_noise,
            max_iters=cfg.pa_iterations, tol=cfg.pa_tolerance,
            n_random_starts=4 if use_multistart else 0,
            restart_seed=mr.named_seed(cfg.seed, "gpm_restart", outer_seed, draw),
        )
    output: dict[str, dict[str, cp.ProtocolResult]] = {m: {} for m in active}
    for split, (xs, _, _) in split_blocks.items():
        clean_aa = None
        if need_aa:
            assert clean_aa_train is not None
            clean_aa = (
                clean_aa_train
                if split == "train"
                else _fixed_aa("AA-I-GDP", xs, clean_aa_train)
            )
        common_noise = None
        if need_private_noise:
            common_noise = [
                sigma * manifest.standard_normal(
                    x.shape, split, i, "private", outer_seed=outer_seed, draw=draw,
                    family="c_aa_coupled", dtype=torch.float64, device=device,
                ) for i, x in enumerate(xs)
            ]
        if "c_gdp" in active:
            output["c_gdp"][split] = cp.c_gdp(
                xs, rotation=common_o, translation=common_psi
            )
        if "c_gdp_an" in active:
            assert common_noise is not None
            output["c_gdp_an"][split] = cp.c_gdp_an(
                xs, sigma, rotation=common_o, translation=common_psi, noises=common_noise
            )
        if "i_gdp" in active:
            output["i_gdp"][split] = cp.i_gdp(
                xs, rotations=rotations, translations=translations
            )
        if "aa_i_gdp" in active:
            assert clean_aa is not None
            output["aa_i_gdp"][split] = clean_aa
        if "aa_i_gdp_an" in active:
            assert clean_aa is not None and clean_aa_train is not None and common_noise is not None
            aa_noise = [
                e @ r.mT
                for e, r in zip(common_noise, clean_aa.alignment_rotations or [])
            ]
            output["aa_i_gdp_an"][split] = _fixed_aa(
                "AA-I-GDP-AN", xs, clean_aa_train, aa_noise
            )
        if "pa_i_gdp" in active:
            assert pa_train is not None
            output["pa_i_gdp"][split] = (
                pa_train if split == "train" else _fixed_pa(xs, pa_train)
            )
    structural: dict[str, Any] = {}
    if {"c_gdp", "aa_i_gdp"}.issubset(active):
        structural["c_aa_float64"] = mr.relative_frobenius_error(
            output["c_gdp"]["train"].collaboration,
            output["aa_i_gdp"]["train"].collaboration,
        )
    if {"c_gdp_an", "aa_i_gdp_an"}.issubset(active):
        structural["c_aa_an_coupled_float64"] = mr.relative_frobenius_error(
            output["c_gdp_an"]["train"].collaboration,
            output["aa_i_gdp_an"]["train"].collaboration,
        )
    structural["passed"] = bool(
        structural and max(float(value) for value in structural.values()) <= 1e-8
    )
    return output, split_blocks, structural, sigma, v


def _evaluate_utility(cfg, result, labels, run_id, output_dir, seed):
    # Pair members must share the initial weights as well as the minibatch RNG.
    mr.set_global_determinism(seed, cfg.deterministic)
    model = mr.ReferencePaperCNN(
        cfg.cnn_side,
        minimum_input_side=cfg.cnn_minimum_input_side,
    )
    checkpoint = Path(output_dir) / "models" / f"{run_id}.pt" if cfg.save_models else None
    trained = mr.train_classifier(
        model,
        mr.as_square_images(result["train"].collaboration.float().cpu(), cfg.cnn_side), labels["train"],
        mr.as_square_images(result["val"].collaboration.float().cpu(), cfg.cnn_side), labels["val"],
        cfg, seed=seed, checkpoint_path=checkpoint,
    )
    utility = {"best_epoch": trained["best_epoch"], "best_val_accuracy": trained["best_val_accuracy"]}
    for split in ("val", "test"):
        if not len(labels[split]):
            continue
        loader = DataLoader(
            TensorDataset(
                mr.as_square_images(result[split].collaboration.float().cpu(), cfg.cnn_side),
                labels[split],
            ),
            batch_size=cfg.eval_batch_size, shuffle=False,
        )
        metrics = mr.evaluate_model(trained["model"], loader, mr.resolve_device(cfg.device))
        prefix = "validation" if split == "val" else "test"
        utility.update({f"{prefix}_{k}": v for k, v in metrics.items()})
    return utility


def _attack_results(
    cfg, method, result, data, blocks, auditor, anchor, seed,
    preview_path: Path | None = None,
    linkage_manifest: mr.LinkageManifest | None = None,
):
    split = "test" if len(data.test) else "val"
    xs, ys, indices = blocks[split]
    colluder, target = cfg.reference_participant, (cfg.reference_participant + 1) % cfg.n_participants
    transcript = result[split]
    # Each entry carries its evidential role.  In particular, applying an
    # unrelated colluder secret to I-GDP is a negative control, not a valid
    # target-transform recovery attack.  The target-secret inversion is an
    # evaluator-only oracle ceiling and is likewise excluded from the
    # strongest-valid-attack envelope.
    attacks: list[tuple[cp.AttackResult, str]] = []
    if method in {"c_gdp", "c_gdp_an"}:
        train_x, _, _ = blocks["train"]
        attacks.append((cp.attack_c_known_input(
            train_x[colluder], result["train"].private_uploads[colluder],
            transcript.private_uploads[target], estimator="mp" if method.endswith("_an") else "exact",
        ), "primary"))
        if method == "c_gdp_an":
            # Draft-faithful primary noisy attack: a colluder reveals the
            # shared common secret. The noisy known-input fit above remains a
            # separately visible extension.
            attacks.append((cp.invert_known_gdp(
                transcript.private_uploads[target],
                transcript.rotations[colluder],
                transcript.translations[colluder],
            ), "primary"))
    elif method == "i_gdp":
        if hasattr(cp, "attack_i_distribution_moment"):
            train_x, _, _ = blocks["train"]
            attacks.append((cp.attack_i_distribution_moment(
                train_x[colluder],
                result["train"].private_uploads[target],
                transcript.private_uploads[target],
            ), "primary-distribution-sensitivity"))
        attacks.append((cp.attack_i_colluder_negative_control(
            transcript.private_uploads[target], transcript.rotations[colluder], transcript.translations[colluder]
        ), "negative-control"))
        oracle = cp.invert_known_gdp(
            transcript.private_uploads[target],
            transcript.rotations[target],
            transcript.translations[target],
        )
        oracle.method = "I-target-secret-oracle"
        attacks.append((oracle, "oracle-outside-threat-model"))
    elif method in {"aa_i_gdp", "aa_i_gdp_an"}:
        attacks.append((cp.attack_aa_known_anchor(
            anchor, transcript.anchor_uploads[target], transcript.private_uploads[target]
        ), "primary"))
    else:
        attacks.extend([
            (cp.attack_pa_mp(anchor, transcript.anchor_uploads[target], transcript.private_uploads[target]), "primary"),
            (cp.attack_pa_op(anchor, transcript.anchor_uploads[target], transcript.private_uploads[target]), "primary"),
            (cp.attack_pa_alignment_calibration(
                anchor, transcript.anchor_uploads[target], transcript.private_uploads[target],
                transcript.alignment_rotations[target], [transcript.rotations[colluder]],
                [transcript.alignment_rotations[colluder]],
            ), "primary"),
        ])
    original_raw = data.split(split).raw[indices[target]]
    original_reduced = xs[target].float().cpu()
    target_labels = ys[target]
    if linkage_manifest is None:
        linkage_manifest = mr.make_same_digit_linkage_manifest(
            target_labels, n_candidates=cfg.linkage_candidates,
            max_queries=cfg.linkage_max_queries, seed=seed,
        )
    evaluated = []
    decoded_by_name: dict[str, torch.Tensor] = {}
    for attack, role in attacks:
        attack_reduced = attack.reduced.float().cpu()
        decoded = data.reducer.decode(attack_reduced).cpu()
        decoded_image = decoded.clamp(0.0, 1.0)
        decoded_by_name[attack.method] = decoded
        metrics = {}
        metrics.update(mr.reduced_reconstruction_metrics(original_reduced, attack_reduced))
        metrics["rotation_relative_error"] = mr.relative_frobenius_error(
            attack.rotation, transcript.rotations[target]
        )
        true_translation = transcript.translations[target].reshape(-1)
        estimated_translation = attack.translation.reshape(-1).to(true_translation)
        metrics["translation_relative_error"] = float(
            torch.linalg.vector_norm(estimated_translation - true_translation).item()
            / max(torch.linalg.vector_norm(true_translation).item(), 1.0e-15)
        )
        metrics["exact_reduced_recovery_1e-6"] = bool(
            metrics["reduced_nrmse"] <= 1.0e-6
        )
        metrics.update({f"pixel_{k}": v for k, v in mr.compute_reconstruction_metrics(
            original_raw, decoded, ssim_max_images=cfg.ssim_max_images, seed=seed
        ).items()})
        metrics.update(mr.auditor_semantic_metrics(
            auditor, decoded_image, target_labels,
            device=mr.resolve_device(cfg.device), batch_size=cfg.eval_batch_size,
        ))
        metrics.update({f"linkage_{k}": v for k, v in mr.same_digit_source_linkage(
            original_raw, decoded_image, target_labels, auditor,
            manifest=linkage_manifest, device=mr.resolve_device(cfg.device),
            batch_size=cfg.eval_batch_size,
        ).items()})
        evaluated.append({"name": attack.method, "role": role, "metrics": metrics})
    valid = [item for item in evaluated if item["role"].startswith("primary")]
    if valid:
        # The security endpoint is 10-way linkage, so maximize that first;
        # pixel linkage and reconstruction fidelity break ties.
        best = max(
            valid,
            key=lambda item: (
                item["metrics"].get("linkage_embedding_top1", -math.inf),
                item["metrics"].get("linkage_pixel_top1", -math.inf),
                -item["metrics"].get("pixel_nrmse", math.inf),
            ),
        )
    else:
        best = None
    if preview_path is not None:
        import matplotlib.pyplot as plt

        chosen: list[int] = []
        for digit in range(mr.NUM_CLASSES):
            candidates = torch.nonzero(target_labels == digit, as_tuple=False).flatten()
            if len(candidates):
                chosen.append(int(candidates[0]))
        floor_raw = data.reducer.decode(original_reduced.float()).cpu().clamp(0, 1)
        floor_linkage = mr.same_digit_source_linkage(
            original_raw,
            floor_raw,
            target_labels,
            auditor,
            manifest=linkage_manifest,
            device=mr.resolve_device(cfg.device),
            batch_size=cfg.eval_batch_size,
        )
        preview_path.parent.mkdir(parents=True, exist_ok=True)
        image_rows: list[tuple[torch.Tensor, str]] = [(original_raw, "Original source")]
        image_rows.append((
            floor_raw,
            "GRP floor\n"
            f"10-way E={100*floor_linkage.get('embedding_top1', math.nan):.1f}% "
            f"P={100*floor_linkage.get('pixel_top1', math.nan):.1f}%",
        ))
        evaluated_by_name = {item["name"]: item for item in evaluated}
        for attack, _ in attacks:
            item = evaluated_by_name[attack.method]
            metric = item["metrics"]
            image_rows.append((
                decoded_by_name[attack.method].clamp(0, 1),
                f"{attack.method}\n{item['role']}\n"
                f"10-way E={100*metric.get('linkage_embedding_top1', math.nan):.1f}% "
                f"P={100*metric.get('linkage_pixel_top1', math.nan):.1f}%",
            ))
        # Reserve a real first column for row labels.  Axis y-labels combined
        # with ``bbox_inches='tight'`` were easy to crop in Colab/Drive
        # previews, whereas a dedicated axes column is part of the figure's
        # layout and remains readable at every export size.
        fig, axes = plt.subplots(
            len(image_rows),
            len(chosen) + 1,
            figsize=(3.1 + 1.45 * len(chosen), 1.55 * len(image_rows) + 1.1),
            squeeze=False,
            gridspec_kw={"width_ratios": [3.0] + [1.0] * len(chosen)},
        )
        for row, (images, label) in enumerate(image_rows):
            axes[row, 0].axis("off")
            axes[row, 0].text(
                0.98,
                0.5,
                label,
                fontsize=8,
                ha="right",
                va="center",
                wrap=True,
                transform=axes[row, 0].transAxes,
            )
            for col, index in enumerate(chosen):
                image_axis = axes[row, col + 1]
                image_axis.imshow(images[index].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
                image_axis.axis("off")
                if row == 0:
                    image_axis.set_title(str(int(target_labels[index])))
        fig.suptitle(
            f"{method}: fixed digit columns; row scores use the full frozen gallery "
            f"(E=embedding, P=pixel, chance=10%)",
            fontsize=11,
        )
        fig.subplots_adjust(left=0.02, right=0.995, top=0.92, bottom=0.02, hspace=0.18, wspace=0.04)
        fig.savefig(preview_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        if best is not None:
            best["preview_path"] = str(preview_path)
        else:
            # Still surface the diagnostic grid when no threat-model-valid
            # reconstruction attack is available.
            for item in evaluated:
                if item["role"] == "negative-control":
                    item["preview_path"] = str(preview_path)
                    break
    return evaluated, best, split


def _diag(result: cp.ProtocolResult, anchor_rows: int | None = None) -> dict[str, Any] | None:
    d = result.diagnostics
    if d is None:
        return None
    anchor_energy = sum(
        float(torch.linalg.norm(block).square().detach().cpu().item())
        for block in (result.anchor_uploads or [])
    )
    return {
        "iterations": d.iterations, "converged": d.converged,
        "elapsed_seconds": d.elapsed_seconds,
        "anchor_singular_values": d.anchor_singular_values,
        "anchor_sigma_min": d.anchor_sigma_min,
        "anchor_sigma_max": d.anchor_sigma_max,
        "anchor_condition": d.anchor_condition,
        "anchor_sigma_min_normalized": d.anchor_sigma_min,
        "upload_singular_values": d.upload_singular_values,
        "gram_eigenvalues": d.gram_eigenvalues,
        "eigengap": d.eigengap, "relative_eigengap": d.relative_eigengap,
        "gopp_residual": d.gopp_residual,
        "relative_gopp_residual": d.relative_gopp_residual,
        "fixed_point_residual": d.fixed_point_residual,
        "multistart_objective_gap": d.multistart_objective_gap,
        "multistart_objective_gap_normalized": (
            d.multistart_objective_gap / anchor_energy
            if anchor_energy > 0 else math.nan
        ),
        "objective_history": d.objective_history,
        "step_history": d.step_history,
    }


def _pa_alignment_theory_diagnostics(
    cfg: mr.ExperimentConfig, result: cp.ProtocolResult, v: float
) -> dict[str, Any] | None:
    """Record simulation-only rotation and mean-anchor-noise diagnostics."""

    if result.diagnostics is None or result.alignment_rotations is None:
        return None
    effective = [
        rotation @ alignment
        for rotation, alignment in zip(result.rotations, result.alignment_rotations)
    ]
    common = cp.orthogonal_polar(sum(effective, torch.zeros_like(effective[0])))
    rotation_errors = [
        float(torch.linalg.matrix_norm(item - common).detach().cpu().item())
        for item in effective
    ]
    eta_norms = [
        float(torch.linalg.vector_norm(noise.mean(0)).detach().cpu().item())
        for noise in (result.anchor_noises or [])
    ]
    p, h, rows = cfg.n_participants, cfg.reduced_dim, cfg.anchor_rows
    sigma_min = float(result.diagnostics.anchor_sigma_min)
    empirical_v_pt = (
        1.89 * sigma_min * math.sqrt(p)
        / (math.sqrt(p * h) + math.sqrt(rows - 1) + 2.0 * math.sqrt(p * math.log(p)))
    )
    gamma = 3.0
    proof_v_sufficient = (
        sigma_min * math.sqrt(p)
        / (
            14336.0
            * math.sqrt(h)
            * (
                math.sqrt(p * h)
                + math.sqrt(rows - 1)
                + math.sqrt(2.0 * gamma * p * math.log(p))
            )
        )
    )
    return {
        "common_rotation_estimator": "polar_sum",
        "effective_rotation_frobenius_errors": rotation_errors,
        "effective_rotation_max_frobenius_error": max(rotation_errors),
        "effective_rotation_mean_frobenius_error": float(np.mean(rotation_errors)),
        "anchor_noise_mean_norms": eta_norms,
        "anchor_noise_mean_max_norm": max(eta_norms) if eta_norms else 0.0,
        "anchor_noise_mean_mean_norm": float(np.mean(eta_norms)) if eta_norms else 0.0,
        "anchor_noise_mean_expected_squared_norm": h * v * v / rows,
        "v_over_anchor_sigma_min": v / sigma_min if sigma_min > 0 else math.nan,
        "ling_empirical_v_pt": empirical_v_pt,
        "v_over_ling_empirical_v_pt": v / empirical_v_pt if empirical_v_pt > 0 else math.nan,
        "ling_proof_sufficient_v_gamma_3": proof_v_sufficient,
        "v_over_ling_proof_sufficient_gamma_3": (
            v / proof_v_sufficient if proof_v_sufficient > 0 else math.nan
        ),
    }


def _i_geometry_leakage(original_reduced: torch.Tensor, upload: torch.Tensor) -> dict[str, float]:
    """Verify the exact within-participant geometry leaked by I-GDP."""

    count = min(256, len(original_reduced), len(upload))
    x = original_reduced[:count].double()
    y = upload[:count].to(x).double()
    xc, yc = x - x.mean(0), y - y.mean(0)
    gram_x, gram_y = xc @ xc.mT, yc @ yc.mT
    gram_error = float(
        (torch.linalg.norm(gram_x - gram_y) / torch.linalg.norm(gram_x).clamp_min(1.0e-15))
        .detach().cpu().item()
    )
    distances_x, distances_y = torch.pdist(x), torch.pdist(y)
    distance_error = float(
        (torch.linalg.norm(distances_x - distances_y) / torch.linalg.norm(distances_x).clamp_min(1.0e-15))
        .detach().cpu().item()
    )
    return {
        "sample_rows": float(count),
        "centered_gram_relative_error": gram_error,
        "pairwise_distance_relative_error": distance_error,
    }


def _bootstrap_binary_mean_ci(
    successes: Sequence[int | bool],
    *,
    seed: int,
    n_resamples: int,
) -> tuple[float, float]:
    """Percentile bootstrap CI clustered by source record."""

    values = np.asarray(successes, dtype=np.float64)
    if values.ndim != 1 or not len(values):
        return math.nan, math.nan
    if len(values) == 1:
        value = float(values[0])
        return value, value
    rng = np.random.default_rng(seed)
    means: list[np.ndarray] = []
    remaining = max(200, int(n_resamples))
    # Chunk the resamples so a 2,000-query gallery cannot allocate one giant
    # n_resamples-by-n_queries integer matrix.
    while remaining:
        batch = min(500, remaining)
        indices = rng.integers(0, len(values), size=(batch, len(values)))
        means.append(values[indices].mean(axis=1))
        remaining -= batch
    observed = np.concatenate(means)
    low, high = np.quantile(observed, [0.025, 0.975])
    return float(low), float(high)


def _pilot_linkage_successes(record: Mapping[str, Any]) -> list[int]:
    best = record.get("best_attack") or {}
    metrics = best.get("metrics", {})
    values = metrics.get("linkage_embedding_top1_successes")
    if values is not None:
        return [int(v) for v in values]
    n = int(metrics.get("linkage_n_queries", 0) or 0)
    rate = float(metrics.get("linkage_embedding_top1", math.nan))
    if n <= 0 or not math.isfinite(rate):
        return []
    successes = int(round(rate * n))
    return [1] * successes + [0] * max(0, n - successes)


def _pilot_evidence(
    records: Sequence[Mapping[str, Any]],
    family: str,
    cfg: mr.ExperimentConfig,
) -> list[dict[str, Any]]:
    """Aggregate validation evidence at every evaluated endpoint."""

    grouped: dict[float, list[Mapping[str, Any]]] = {}
    for record in records:
        if record.get("pilot_family") == family:
            grouped.setdefault(float(record["lambda"]), []).append(record)
    if 0.0 not in grouped:
        return []
    baseline = float(np.mean([
        row["utility"]["validation_balanced_accuracy"] for row in grouped[0.0]
    ]))
    if (
        not math.isfinite(baseline)
        or baseline <= PILOT_CHANCE_ACCURACY + PILOT_BASELINE_EPSILON
    ):
        raise RuntimeError(
            f"Pilot {family!r} zero-noise balanced accuracy {baseline!r} is not "
            f"meaningfully above the {PILOT_CHANCE_ACCURACY:.2f} chance baseline; "
            "a normalized retained-utility bound would be undefined."
        )
    denominator = baseline - PILOT_CHANCE_ACCURACY
    evidence: list[dict[str, Any]] = []
    for level in sorted(grouped):
        rows = grouped[level]
        utility_values = np.asarray([
            row["utility"]["validation_balanced_accuracy"] for row in rows
        ], dtype=np.float64)
        utility = float(utility_values.mean())
        retained = (
            float((utility - PILOT_CHANCE_ACCURACY) / denominator)
            if abs(denominator) > 1.0e-12
            else math.nan
        )
        successes = [
            success
            for row in rows
            for success in _pilot_linkage_successes(row)
        ]
        linkage = float(np.mean(successes)) if successes else math.nan
        ci_low, ci_high = _bootstrap_binary_mean_ci(
            successes,
            seed=mr.named_seed(cfg.seed, "pilot_bootstrap", family, level),
            n_resamples=min(cfg.bootstrap_resamples, 5_000),
        )
        privacy_trigger = bool(
            math.isfinite(linkage)
            and linkage <= 0.12
            and ci_low <= 0.10 <= ci_high
        )
        utility_trigger = bool(math.isfinite(retained) and retained <= 0.15)
        evidence.append({
            "lambda": level,
            "outer_seeds_completed": sorted(int(row["outer_seed"]) for row in rows),
            "validation_balanced_accuracy": utility,
            "validation_balanced_accuracy_by_seed": utility_values.tolist(),
            "baseline_validation_balanced_accuracy": baseline,
            "baseline_above_chance": True,
            "retained_utility": retained,
            "strongest_valid_embedding_linkage_top1": linkage,
            "linkage_bootstrap_ci_95": [ci_low, ci_high],
            "linkage_source_records": len(successes),
            "utility_trigger": utility_trigger,
            "privacy_trigger": privacy_trigger,
            "trigger": bool(utility_trigger or privacy_trigger),
        })
    return evidence


def _first_consecutive_trigger(evidence: Sequence[Mapping[str, Any]]) -> float | None:
    positive = [item for item in evidence if float(item["lambda"]) > 0]
    for left, right in zip(positive, positive[1:]):
        if bool(left.get("trigger")) and bool(right.get("trigger")):
            return float(left["lambda"])
    return None


def _pilot_protocol_cell(
    cfg: mr.ExperimentConfig,
    data: mr.MNISTDataBundle,
    blocks: Mapping[str, Any],
    manifest: mr.UniformNoiseManifest,
    *,
    family: str,
    method: str,
    level: float,
    outer_seed: int,
    anchor: torch.Tensor,
) -> tuple[dict[str, cp.ProtocolResult], float, float]:
    """Build one validation-only C-AN, coupled AA-AN, or PA endpoint."""

    device = mr.resolve_device(cfg.device)
    h, p = cfg.reduced_dim, cfg.n_participants
    generator = cp.make_generator(mr.named_seed(cfg.seed, "pilot_gdp", outer_seed), device)
    rotations, translations = cp.sample_gdp_parameters(
        p, h, generator=generator, device=device
    )
    reference = cfg.reference_participant
    reduced_scale = float(torch.sqrt(data.train.reduced.double().square().mean()).item())
    sigma, v = float(level * reduced_scale), float(level)

    if family == "private":
        if method not in {"c_gdp_an", "aa_i_gdp_an"}:
            raise ValueError(f"Private pilot family does not support method {method!r}")
        output: dict[str, cp.ProtocolResult] = {}
        clean_aa_train = None
        if method == "aa_i_gdp_an":
            clean_aa_train = cp.aa_i_gdp(
                blocks["train"][0],
                anchor,
                reference=reference,
                rotations=rotations,
                translations=translations,
            )
        for split in ("train", "val"):
            xs = blocks[split][0]
            common_noise = [
                sigma * manifest.standard_normal(
                    x.shape,
                    split,
                    participant,
                    "private",
                    outer_seed=outer_seed,
                    draw=0,
                    family="pilot_private_scaled_path",
                    dtype=torch.float64,
                    device=device,
                )
                for participant, x in enumerate(xs)
            ]
            if method == "c_gdp_an":
                output[split] = cp.c_gdp_an(
                    xs,
                    sigma,
                    rotation=rotations[reference],
                    translation=translations[reference],
                    noises=common_noise,
                )
            else:
                assert clean_aa_train is not None
                local_noise = [
                    noise @ alignment.mT
                    for noise, alignment in zip(
                        common_noise, clean_aa_train.alignment_rotations or []
                    )
                ]
                output[split] = _fixed_aa(
                    "AA-I-GDP-AN", xs, clean_aa_train, local_noise
                )
        return output, sigma, 0.0

    if family != "anchor":
        raise ValueError(f"Unknown pilot family {family!r}")
    if method != "pa_i_gdp":
        raise ValueError(f"Anchor pilot family does not support method {method!r}")
    anchor_noise = [
        v * manifest.standard_normal(
            anchor.shape,
            "anchor",
            participant,
            "anchor",
            outer_seed=outer_seed,
            draw=0,
            family="pilot_anchor_absolute_v_path",
            dtype=torch.float64,
            device=device,
        )
        for participant in range(p)
    ]
    train = cp.pa_i_gdp(
        blocks["train"][0],
        anchor,
        v,
        rotations=rotations,
        translations=translations,
        anchor_noises=anchor_noise,
        max_iters=cfg.pa_iterations,
        tol=cfg.pa_tolerance,
        n_random_starts=0,
    )
    return {
        "train": train,
        "val": _fixed_pa(blocks["val"][0], train),
    }, 0.0, v


def _run_pilot_endpoint(
    cfg: mr.ExperimentConfig,
    data: mr.MNISTDataBundle,
    blocks: Mapping[str, Any],
    labels: Mapping[str, torch.Tensor],
    manifest: mr.UniformNoiseManifest,
    auditor: mr.CleanMNISTAuditor,
    store: mr.ResultStore,
    pilot_out: Path,
    *,
    family: str,
    pilot_fingerprint: str,
    level: float,
    outer_seed: int,
    method: str | None = None,
) -> None:
    method = method or ("c_gdp_an" if family == "private" else "pa_i_gdp")
    record_family = (
        "private_aa_confirmation" if method == "aa_i_gdp_an" else family
    )
    device = mr.resolve_device(cfg.device)
    anchor_cpu, anchor_diagnostics = _make_centered_isometric_anchor(cfg, outer_seed)
    anchor = anchor_cpu.to(device)
    anchor_dir = pilot_out / "anchors"
    anchor_dir.mkdir(parents=True, exist_ok=True)
    anchor_manifest_path = anchor_dir / f"anchor_seed_{outer_seed}.json"
    _write_or_validate_frozen_json(
        anchor_manifest_path, anchor_diagnostics, label="pilot anchor manifest"
    )
    anchor_tensor_path = anchor_dir / f"anchor_seed_{outer_seed}.pt"
    if not anchor_tensor_path.exists():
        torch.save(anchor_cpu, anchor_tensor_path)
    run_id = mr.make_run_id(
        mode="pilot_grp_unit_isometric_anchor_v2",
        pilot_fingerprint=pilot_fingerprint,
        data=mr.stable_hash(data.manifest, 32),
        anchor_sha256=anchor_diagnostics["sha256_float64_c_order"],
        family=record_family,
        method=method,
        level=float(level),
        outer_seed=int(outer_seed),
    )
    if store.contains(run_id):
        return
    runtime_environment = {
        "device": str(mr.resolve_device(cfg.device)),
        "gpu_name": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        ),
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
    }
    runtime_session_id = mr.stable_hash(
        {"started_unix": time.time(), **runtime_environment}, 16
    )
    mr._atomic_write_json(
        pilot_out / "execution_sessions" / f"{runtime_session_id}.json",
        {"runtime_session_id": runtime_session_id, **runtime_environment},
    )
    started = time.perf_counter()
    protocol, sigma, v = _pilot_protocol_cell(
        cfg,
        data,
        blocks,
        manifest,
        family=family,
        method=method,
        level=level,
        outer_seed=outer_seed,
        anchor=anchor,
    )
    # C-GDP-AN and its AA confirmation deliberately share model initialization,
    # minibatch order, optimizer randomness, the scaled Gaussian path, and the
    # linkage gallery.  Only their protocol construction differs.
    model_seed = mr.named_seed(cfg.seed, "pilot_model", family, outer_seed)
    utility = _evaluate_utility(
        cfg,
        protocol,
        labels,
        run_id,
        pilot_out,
        model_seed,
    )
    attacks, best, split = _attack_results(
        cfg,
        method,
        protocol,
        data,
        blocks,
        auditor,
        anchor,
        mr.named_seed(cfg.seed, "pilot_linkage", outer_seed),
        pilot_out / "previews" / (
            f"{record_family}_{method}_lambda_{level:.8g}_seed_{outer_seed}.png"
        ),
    )
    confirmation = None
    if method == "aa_i_gdp_an":
        common_protocol, _, _ = _pilot_protocol_cell(
            cfg,
            data,
            blocks,
            manifest,
            family="private",
            method="c_gdp_an",
            level=level,
            outer_seed=outer_seed,
            anchor=anchor,
        )
        train_error = mr.relative_frobenius_error(
            common_protocol["train"].collaboration,
            protocol["train"].collaboration,
        )
        val_error = mr.relative_frobenius_error(
            common_protocol["val"].collaboration,
            protocol["val"].collaboration,
        )
        confirmation = {
            "c_aa_train_relative_error": train_error,
            "c_aa_validation_relative_error": val_error,
            "structural_tolerance": 1.0e-8,
            "structural_passed": max(train_error, val_error) <= 1.0e-8,
        }
    record = {
        "run_id": run_id,
        "mode": "pilot",
        "pilot_family": record_family,
        "noise_family": family,
        "noise_path": (
            "pilot_private_scaled_path"
            if family == "private"
            else "pilot_anchor_absolute_v_path"
        ),
        "protocol": method,
        "lambda": float(level),
        "outer_seed": int(outer_seed),
        "model_seed": int(model_seed),
        "pilot_fingerprint": pilot_fingerprint,
        "anchor": anchor_diagnostics,
        "utility": utility,
        "attacks": attacks,
        "best_attack": best,
        "evaluation_split": split,
        "noise_sigma": sigma,
        "anchor_noise_sigma": v,
        "gpm": _diag(protocol["train"]),
        "alignment_theory": (
            _pa_alignment_theory_diagnostics(cfg, protocol["train"], v)
            if method == "pa_i_gdp" else None
        ),
        "aa_confirmation": confirmation,
        "runtime_seconds": time.perf_counter() - started,
    }
    store.append(record)
    print(
        f"pilot completed family={record_family} method={method} lambda={level:.6g} "
        f"seed={outer_seed} utility={utility.get('validation_balanced_accuracy', math.nan):.4f}"
    )


def _calibrate_pilot_family(
    cfg: mr.ExperimentConfig,
    data: mr.MNISTDataBundle,
    blocks: Mapping[str, Any],
    labels: Mapping[str, torch.Tensor],
    manifest: mr.UniformNoiseManifest,
    auditor: mr.CleanMNISTAuditor,
    store: mr.ResultStore,
    pilot_out: Path,
    family: str,
    pilot_fingerprint: str,
) -> tuple[float, list[dict[str, Any]], str]:
    levels = sorted({float(v) for v in cfg.pilot_candidate_lambdas} | {0.0})
    upward_extensions = 0
    downward_extensions = 0
    selected: float | None = None
    censor_status = "none"
    while True:
        # Evaluate the complete currently declared grid. ResultStore makes this
        # loop restart-safe, so newly inserted endpoints do not retrain old ones.
        for level in sorted(levels):
            for outer_seed in cfg.outer_seeds:
                _run_pilot_endpoint(
                    cfg,
                    data,
                    blocks,
                    labels,
                    manifest,
                    auditor,
                    store,
                    pilot_out,
                    family=family,
                    pilot_fingerprint=pilot_fingerprint,
                    level=level,
                    outer_seed=outer_seed,
                )

        evidence = _pilot_evidence(store.records, family, cfg)
        positive = [item for item in evidence if float(item["lambda"]) > 0.0]
        if not positive:
            raise RuntimeError(f"Pilot {family!r} grid has no positive endpoint")

        smallest = min(positive, key=lambda item: float(item["lambda"]))
        if bool(smallest.get("trigger")):
            # The plan requires searching below the initial 2^-8 endpoint when
            # it already triggers. Continue halving until a lower nontrigger is
            # observed; otherwise the chosen upper boundary is left-censored.
            if downward_extensions >= cfg.pilot_max_grid_extensions:
                selected = float(smallest["lambda"])
                censor_status = "below_grid"
                break
            levels.append(float(smallest["lambda"]) / 2.0)
            levels = sorted(set(levels))
            downward_extensions += 1
            continue

        selected = _first_consecutive_trigger(evidence)
        if selected is not None:
            break

        if upward_extensions >= cfg.pilot_max_grid_extensions:
            censor_status = "upper_not_reached"
            break
        largest = max(levels)
        levels.append(2.0 * largest if largest > 0 else 2.0**-8)
        levels = sorted(set(levels))
        upward_extensions += 1

    evidence = _pilot_evidence(store.records, family, cfg)
    if selected is None:
        selected = max(float(item["lambda"]) for item in evidence)
    return float(selected), evidence, censor_status


def _confirmation_levels(
    selected: float,
    evidence: Sequence[Mapping[str, Any]],
) -> tuple[float, ...]:
    """Return zero, the selected endpoint, and its supporting larger endpoint."""

    evaluated = sorted({float(item["lambda"]) for item in evidence})
    larger = next((level for level in evaluated if level > selected), None)
    if larger is None:
        raise RuntimeError(
            "The selected private upper bound lacks its required next-larger "
            "confirmation endpoint."
        )
    return tuple(sorted({0.0, float(selected), float(larger)}))


def _nested_metric(record: Mapping[str, Any], path: str) -> float:
    value: Any = record
    for part in path.split("."):
        if not isinstance(value, Mapping) or part not in value:
            return math.nan
        value = value[part]
    try:
        return float(value)
    except (TypeError, ValueError):
        return math.nan


def _summarize_private_aa_confirmation(
    records: Sequence[Mapping[str, Any]],
    levels: Sequence[float],
    outer_seeds: Sequence[int],
) -> dict[str, Any]:
    """Validate the coupled C/AA confirmation cells without pseudo-inference."""

    utility_paths = (
        "utility.validation_accuracy",
        "utility.validation_balanced_accuracy",
        "utility.validation_macro_f1",
    )
    attack_paths = (
        "best_attack.metrics.linkage_embedding_top1",
        "best_attack.metrics.linkage_pixel_top1",
        "best_attack.metrics.pixel_nrmse",
    )
    comparisons: list[dict[str, Any]] = []
    missing: list[dict[str, Any]] = []
    for outer_seed in outer_seeds:
        for level in levels:
            common = next((
                record for record in records
                if record.get("pilot_family") == "private"
                and record.get("protocol") == "c_gdp_an"
                and int(record.get("outer_seed", -1)) == int(outer_seed)
                and math.isclose(float(record.get("lambda", math.nan)), float(level))
            ), None)
            aligned = next((
                record for record in records
                if record.get("pilot_family") == "private_aa_confirmation"
                and record.get("protocol") == "aa_i_gdp_an"
                and int(record.get("outer_seed", -1)) == int(outer_seed)
                and math.isclose(float(record.get("lambda", math.nan)), float(level))
            ), None)
            if common is None or aligned is None:
                missing.append({"outer_seed": int(outer_seed), "lambda": float(level)})
                continue

            utility_deltas = {
                path: _nested_metric(aligned, path) - _nested_metric(common, path)
                for path in utility_paths
            }
            attack_deltas = {
                path: _nested_metric(aligned, path) - _nested_metric(common, path)
                for path in attack_paths
            }
            confirmation = aligned.get("aa_confirmation") or {}
            # Manifest hashes are hexadecimal strings, so compare them directly
            # rather than through the numeric metric accessor.
            common_hash = (
                (common.get("best_attack") or {}).get("metrics", {})
                .get("linkage_manifest_hash")
            )
            aligned_hash = (
                (aligned.get("best_attack") or {}).get("metrics", {})
                .get("linkage_manifest_hash")
            )
            finite_utility = all(math.isfinite(value) for value in utility_deltas.values())
            comparison_passed = bool(
                confirmation.get("structural_passed")
                and common.get("model_seed") == aligned.get("model_seed")
                and common.get("noise_path") == aligned.get("noise_path")
                and common_hash is not None
                and common_hash == aligned_hash
                and finite_utility
                and max(abs(value) for value in utility_deltas.values()) <= 1.0e-8
            )
            comparisons.append({
                "outer_seed": int(outer_seed),
                "lambda": float(level),
                "model_seed_shared": common.get("model_seed") == aligned.get("model_seed"),
                "noise_path_shared": common.get("noise_path") == aligned.get("noise_path"),
                "gallery_manifest_shared": common_hash is not None and common_hash == aligned_hash,
                "gallery_manifest_hash": common_hash,
                "structural": confirmation,
                "utility_deltas_aa_minus_c": utility_deltas,
                "attack_deltas_aa_minus_c": attack_deltas,
                "passed": comparison_passed,
            })
    return {
        "levels": [float(level) for level in levels],
        "expected_cells": len(levels) * len(outer_seeds),
        "completed_cells": len(comparisons),
        "missing_cells": missing,
        "comparisons": comparisons,
        "passed": not missing and bool(comparisons) and all(
            item["passed"] for item in comparisons
        ),
    }


def run_pilot_calibration(
    config: Mapping[str, Any] | mr.ExperimentConfig,
    output_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Calibrate both GRP experiment upper bounds using validation data only."""

    base_cfg = mr.coerce_config(config)
    cfg = replace(
        base_cfg,
        output_dir=str(output_dir or base_cfg.output_dir),
        save_models=False,
    )
    if cfg.mode != "pilot":
        raise ValueError("run_pilot_calibration requires mode='pilot'")
    out = Path(cfg.output_dir)
    pilot_out = out / "pilot"
    pilot_out.mkdir(parents=True, exist_ok=True)
    identity = _pilot_identity(cfg)
    identity_path = pilot_out / "pilot_run_identity.json"
    legacy_artifacts = [
        path for path in (
            pilot_out / "config.json",
            pilot_out / "data_manifest.json",
            pilot_out / "noise_seed_manifest.json",
            pilot_out / "clean_mnist_auditor.pt",
            pilot_out / "pilot_cells.json",
            pilot_out / "frozen_pilot_manifest.json",
        )
        if path.exists()
    ]
    if legacy_artifacts and not identity_path.exists():
        raise RuntimeError(
            "Legacy pilot artifacts cannot be resumed safely because they lack "
            f"an implementation fingerprint: {legacy_artifacts}. Use a new results "
            "subfolder for the corrected h=100 pilot."
        )
    _write_or_validate_frozen_json(
        identity_path, identity, label="pilot implementation/config identity"
    )
    _write_or_validate_frozen_json(
        pilot_out / "config.json", cfg.to_dict(), label="pilot configuration"
    )
    data = mr.prepare_mnist_data(cfg, download=True, include_test=False)
    if len(data.test):
        raise RuntimeError("Pilot data unexpectedly contains official test records")
    _write_or_validate_frozen_json(
        pilot_out / "data_manifest.json", data.manifest,
        label="pilot data/projection manifest",
    )
    auditor, auditor_meta = mr.train_or_load_auditor(
        data,
        cfg,
        checkpoint_path=pilot_out / "clean_mnist_auditor.pt",
    )
    manifest = mr.make_noise_manifest(cfg)
    _write_or_validate_frozen_json(
        pilot_out / "noise_seed_manifest.json", manifest.to_dict(),
        label="pilot noise seed manifest",
    )
    device = mr.resolve_device(cfg.device)
    blocks = {
        split: _blocks(data, split, device)
        for split in ("train", "val")
    }
    labels = {
        "train": torch.cat(blocks["train"][1]),
        "val": torch.cat(blocks["val"][1]),
        "test": torch.empty(0, dtype=torch.long),
    }
    store = mr.ResultStore(pilot_out, stem="pilot_cells")
    started = time.perf_counter()
    pilot_fingerprint = str(identity["pilot_fingerprint"])
    private_upper, private_evidence, private_censor_status = _calibrate_pilot_family(
        cfg, data, blocks, labels, manifest, auditor, store, pilot_out,
        "private", pilot_fingerprint,
    )
    anchor_upper, anchor_evidence, anchor_censor_status = _calibrate_pilot_family(
        cfg, data, blocks, labels, manifest, auditor, store, pilot_out,
        "anchor", pilot_fingerprint,
    )

    aa_confirmation: dict[str, Any]
    if private_censor_status == "none":
        confirmation_levels = _confirmation_levels(private_upper, private_evidence)
        for level in confirmation_levels:
            for outer_seed in cfg.outer_seeds:
                _run_pilot_endpoint(
                    cfg,
                    data,
                    blocks,
                    labels,
                    manifest,
                    auditor,
                    store,
                    pilot_out,
                    family="private",
                    pilot_fingerprint=pilot_fingerprint,
                    level=level,
                    outer_seed=outer_seed,
                    method="aa_i_gdp_an",
                )
        aa_confirmation = _summarize_private_aa_confirmation(
            store.records, confirmation_levels, cfg.outer_seeds
        )
    else:
        aa_confirmation = {
            "levels": [],
            "expected_cells": 0,
            "completed_cells": 0,
            "missing_cells": [],
            "comparisons": [],
            "passed": False,
            "skipped_reason": f"private calibration is {private_censor_status}",
        }

    pilot_records = [
        record for record in store.records
        if record.get("pilot_family") in {
            "private", "anchor", "private_aa_confirmation"
        }
    ]
    gallery_values = [
        ((record.get("best_attack") or {}).get("metrics", {})).get(
            "linkage_manifest_hash"
        )
        for record in pilot_records
    ]
    gallery_hashes = {str(value) for value in gallery_values if value is not None}
    gallery_consistent = (
        bool(gallery_values)
        and all(value is not None for value in gallery_values)
        and len(gallery_hashes) == 1
    )
    pilot_anchor_manifests = {
        str(outer_seed): _make_centered_isometric_anchor(cfg, int(outer_seed))[1]
        for outer_seed in cfg.outer_seeds
    }
    acceptance = {
        "validation_only": True,
        "gaussian_projection_used": (
            data.reducer.manifest().get("kind") == "gaussian_random_projection"
        ),
        "private_upper_found": private_censor_status == "none",
        "anchor_upper_found": anchor_censor_status == "none",
        "private_aa_confirmation_passed": bool(aa_confirmation.get("passed")),
        "global_gallery_manifest_consistent": gallery_consistent,
        "centered_unit_isometric_anchor_validated": all(
            float(item["mean_residual_l2"]) <= 1.0e-10
            and float(item["gram_relative_error"]) <= 1.0e-10
            and abs(float(item["sigma_min"]) - 1.0) <= 1.0e-10
            and abs(float(item["sigma_max"]) - 1.0) <= 1.0e-10
            and abs(float(item["condition_number"]) - 1.0) <= 1.0e-10
            for item in pilot_anchor_manifests.values()
        ),
        "zero_noise_baselines_above_chance": all(
            bool(item.get("baseline_above_chance"))
            for item in [*private_evidence, *anchor_evidence]
        ),
    }
    acceptance["all_passed"] = all(acceptance.values())

    frozen = {
        "schema_version": PILOT_SCHEMA_VERSION,
        "calibration_passed": bool(acceptance["all_passed"]),
        "test_used": False,
        "projection": data.reducer.manifest(),
        "anchor_design": cfg.anchor_design,
        "anchors": pilot_anchor_manifests,
        "lower_bounds": {"private": 0.0, "anchor": 0.0},
        "private_lambda_upper": private_upper,
        "anchor_lambda_upper": anchor_upper,
        "private_censor_status": private_censor_status,
        "anchor_censor_status": anchor_censor_status,
        "private_upper_censored": private_censor_status != "none",
        "anchor_upper_censored": anchor_censor_status != "none",
        "selection_rule": (
            "first of two consecutive positive endpoints where retained utility <= 0.15 "
            "or strongest valid embedding 10-way top-1 <= 0.12 with its 95% "
            "source-record bootstrap CI containing 0.10"
        ),
        "private_evidence": private_evidence,
        "anchor_evidence": anchor_evidence,
        "private_aa_confirmation": aa_confirmation,
        "global_linkage_manifest_hashes": sorted(gallery_hashes),
        "outer_seeds": list(cfg.outer_seeds),
        "candidate_seed_manifest": manifest.to_dict(),
        "data_manifest_hash": mr.stable_hash(data.manifest, 32),
        "pilot_fingerprint": pilot_fingerprint,
        "implementation_sha256": identity["implementation_sha256"],
        "acceptance": acceptance,
        "module": "mnist_orchestrator.run_pilot_calibration",
    }
    diagnostic = {
        **frozen,
        "elapsed_seconds": time.perf_counter() - started,
        "frozen_manifest_written": bool(acceptance["all_passed"]),
    }
    mr._atomic_write_json(
        pilot_out / "pilot_calibration_diagnostic.json", diagnostic
    )

    manifest_path: Path | None = None
    manifest_hash: str | None = None
    if acceptance["all_passed"]:
        manifest_path = pilot_out / "frozen_pilot_manifest.json"
        _write_or_validate_frozen_json(
            manifest_path, frozen, label="successful frozen pilot manifest"
        )
        manifest_hash = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    return {
        "mode": "pilot",
        "status": {
            "private_lambda_upper": private_upper,
            "anchor_lambda_upper": anchor_upper,
            "private_censor_status": private_censor_status,
            "anchor_censor_status": anchor_censor_status,
            "elapsed_seconds": time.perf_counter() - started,
        },
        "config": cfg.to_dict(),
        "data_manifest": data.manifest,
        "auditor": auditor_meta,
        "records": store.records,
        "private_evidence": private_evidence,
        "anchor_evidence": anchor_evidence,
        "private_aa_confirmation": aa_confirmation,
        "pilot_manifest_path": None if manifest_path is None else str(manifest_path),
        "pilot_manifest_sha256": manifest_hash,
        "acceptance": acceptance,
        "required_report_keys": (
            "summary_path", "report_path", "plot_path", "reconstruction_grid"
        ),
    }


def run_experiment(config: Mapping[str, Any] | mr.ExperimentConfig, output_dir: str | Path | None = None):
    """Run a resumable all-method benchmark and return its report manifest."""

    cfg = mr.coerce_config(config)
    if cfg.mode == "pilot":
        return run_pilot_calibration(cfg, output_dir)
    if cfg.mode == "full":
        if not cfg.pilot_manifest_path or not cfg.pilot_manifest_sha256:
            raise ValueError("Full mode requires a frozen pilot manifest path and SHA-256")
        pilot_path = Path(cfg.pilot_manifest_path)
        if not pilot_path.is_file():
            raise FileNotFoundError(f"Frozen pilot manifest not found: {pilot_path}")
        actual_hash = hashlib.sha256(pilot_path.read_bytes()).hexdigest()
        if actual_hash.lower() != cfg.pilot_manifest_sha256.lower():
            raise ValueError("Frozen pilot manifest checksum mismatch")
        pilot = json.loads(pilot_path.read_text(encoding="utf-8"))
        if int(pilot.get("schema_version", 0)) < PILOT_SCHEMA_VERSION:
            raise ValueError(
                "Frozen pilot manifest predates the corrected resumable calibration schema"
            )
        if pilot.get("calibration_passed") is not True:
            raise ValueError("Frozen pilot manifest does not attest calibration_passed=true")
        if pilot.get("implementation_sha256") != _implementation_sha256():
            raise ValueError(
                "Full mode implementation differs from the code fingerprinted by the pilot"
            )
        censor_statuses = {
            "private": pilot.get("private_censor_status"),
            "anchor": pilot.get("anchor_censor_status"),
        }
        if any(status != "none" for status in censor_statuses.values()):
            raise ValueError(
                f"Full mode refuses censored pilot bounds: {censor_statuses}"
            )
        if not bool(
            (pilot.get("private_aa_confirmation") or {}).get("passed")
        ):
            raise ValueError("Frozen pilot manifest lacks a passing AA private-bound confirmation")
        if pilot.get("test_used") is not False:
            raise ValueError("Pilot manifest must attest test_used=false")
        pilot_projection = pilot.get("projection", {})
        pilot_h = pilot_projection.get(
            "output_dim",
            pilot_projection.get("reduced_dim", pilot_projection.get("paper_reduced_dimension")),
        )
        if pilot_h is None or int(pilot_h) != cfg.reduced_dim:
            raise ValueError(
                "Frozen pilot projection dimension does not match this run: "
                f"pilot h={pilot_h!r}, configured h={cfg.reduced_dim}. "
                "Rerun the validation-only pilot after changing cnn_side."
            )
        cfg = replace(
            cfg,
            private_lambda_upper=float(pilot["private_lambda_upper"]),
            anchor_lambda_upper=float(pilot["anchor_lambda_upper"]),
        )
    base_out = Path(output_dir or cfg.output_dir)
    out = base_out / cfg.mode
    cfg = replace(cfg, output_dir=str(out), protocols=tuple(m for m in cfg.protocols if m in METHODS))
    out.mkdir(parents=True, exist_ok=True)
    _write_or_validate_frozen_json(
        out / "config.json", cfg.to_dict(), label="resolved configuration"
    )
    data = mr.prepare_mnist_data(cfg, download=True)
    _write_or_validate_frozen_json(
        out / "data_manifest.json", data.manifest, label="data/projection manifest"
    )
    auditor, auditor_meta = mr.train_or_load_auditor(data, cfg, checkpoint_path=out / "clean_mnist_auditor.pt")
    mr._atomic_write_json(out / "auditor_metrics.json", auditor_meta)
    # Full mode reserves two extra named draw slots for deterministic zero and
    # calibrated-upper-bound endpoints; random draws remain 0..draws_per_seed-1.
    manifest_config = replace(
        cfg,
        draws_per_seed=cfg.draws_per_seed + (2 if cfg.mode == "full" else 0),
    )
    manifest = mr.make_noise_manifest(manifest_config)
    _write_or_validate_frozen_json(
        out / "noise_manifest.json", manifest.to_dict(), label="noise seed manifest"
    )
    store = mr.ResultStore(out)
    private_upper = cfg.private_lambda_upper if cfg.private_lambda_upper is not None else 0.15
    anchor_upper = cfg.anchor_lambda_upper if cfg.anchor_lambda_upper is not None else 0.15
    labels = {}
    base_blocks = {}
    for split in ("train", "val", "test"):
        base_blocks[split] = _blocks(data, split)
        labels[split] = torch.cat(base_blocks[split][1])
    target_participant = (cfg.reference_participant + 1) % cfg.n_participants
    test_z, test_y, test_indices = base_blocks["test"]
    frozen_linkage_manifest = mr.make_same_digit_linkage_manifest(
        test_y[target_participant],
        n_candidates=cfg.linkage_candidates,
        max_queries=cfg.linkage_max_queries,
        seed=mr.named_seed(cfg.seed, "frozen_linkage", target_participant),
    )
    _write_or_validate_frozen_json(
        out / "linkage_manifest.json",
        frozen_linkage_manifest.to_dict(),
        label="10-way linkage manifest",
    )
    grp_original = data.test.raw[test_indices[target_participant]]
    grp_floor = data.reducer.decode(test_z[target_participant].float()).cpu()
    grp_floor_image = grp_floor.clamp(0.0, 1.0)
    grp_floor_control = {
        "pixel": mr.compute_reconstruction_metrics(
            grp_original,
            grp_floor,
            ssim_max_images=cfg.ssim_max_images,
            seed=mr.named_seed(cfg.seed, "grp_floor"),
        ),
        "semantic": mr.auditor_semantic_metrics(
            auditor,
            grp_floor_image,
            test_y[target_participant],
            device=mr.resolve_device(cfg.device),
            batch_size=cfg.eval_batch_size,
        ),
    }
    mr._atomic_write_json(out / "grp_floor_control.json", grp_floor_control)
    config_hash = mr.stable_hash(cfg.to_dict())
    schedule: list[dict[str, Any]] = []
    outer_seeds = list(cfg.outer_seeds)
    anchor_dir = out / "anchors"
    anchor_dir.mkdir(parents=True, exist_ok=True)
    anchors: dict[int, torch.Tensor] = {}
    anchor_manifests: dict[int, dict[str, Any]] = {}
    for outer in outer_seeds:
        anchor, anchor_diagnostics = _make_centered_isometric_anchor(cfg, int(outer))
        anchors[int(outer)] = anchor
        anchor_manifests[int(outer)] = anchor_diagnostics
        _write_or_validate_frozen_json(
            anchor_dir / f"anchor_seed_{outer}.json",
            anchor_diagnostics,
            label="full anchor manifest",
        )
        anchor_tensor_path = anchor_dir / f"anchor_seed_{outer}.pt"
        if not anchor_tensor_path.exists():
            torch.save(anchor, anchor_tensor_path)
    if cfg.mode == "full":
        pilot_anchor_manifests = pilot.get("anchors", {})
        for outer, diagnostics in anchor_manifests.items():
            pilot_diagnostics = pilot_anchor_manifests.get(str(outer))
            if pilot_diagnostics is None or (
                pilot_diagnostics.get("sha256_float64_c_order")
                != diagnostics["sha256_float64_c_order"]
            ):
                raise ValueError("Full-run anchor differs from the frozen pilot anchor")
    noisy_methods = ["c_gdp_an", "aa_i_gdp_an", "pa_i_gdp"]
    clean_methods = ["c_gdp", "i_gdp", "aa_i_gdp"]
    for outer in outer_seeds:
        for draw in range(cfg.draws_per_seed):
            schedule.append({
                "outer_seed": outer,
                "draw": draw,
                "run_role": "random" if cfg.mode == "full" else "smoke",
                "private_lambda": manifest.lambda_value(
                    private_upper,
                    outer_seed=outer,
                    draw=draw,
                    role="private",
                    family="c_aa",
                ),
                "anchor_lambda": manifest.lambda_value(
                    anchor_upper,
                    outer_seed=outer,
                    draw=draw,
                    role="anchor",
                    family="pa",
                ),
                "methods": (
                    noisy_methods + clean_methods
                    if draw == 0 else noisy_methods
                ),
            })
        if cfg.mode == "full":
            schedule.extend([
                {
                    "outer_seed": outer,
                    "draw": cfg.draws_per_seed,
                    "run_role": "endpoint_zero",
                    "private_lambda": 0.0,
                    "anchor_lambda": 0.0,
                    "methods": noisy_methods,
                },
                {
                    "outer_seed": outer,
                    "draw": cfg.draws_per_seed + 1,
                    "run_role": "endpoint_upper",
                    "private_lambda": private_upper,
                    "anchor_lambda": anchor_upper,
                    "methods": noisy_methods,
                },
            ])

    planned_runs: list[dict[str, Any]] = []
    for spec in schedule:
        outer = int(spec["outer_seed"])
        draw = int(spec["draw"])
        run_role = str(spec["run_role"])
        lx = float(spec["private_lambda"])
        la = float(spec["anchor_lambda"])
        for method in spec["methods"]:
            if method in cfg.protocols:
                level = la if method == "pa_i_gdp" else (lx if method.endswith("_an") else 0.0)
                pair_family = (
                    "c_aa_an"
                    if method in {"c_gdp_an", "aa_i_gdp_an"}
                    else "c_aa"
                    if method in {"c_gdp", "aa_i_gdp"}
                    else method
                )
                model_draw = 0 if cfg.mode == "full" else draw
                model_family = (
                    "fixed_noise_sensitivity_model"
                    if cfg.mode == "full" else pair_family
                )
                planned_runs.append({
                    "run_id": mr.make_run_id(
                        config=config_hash,
                        anchor_sha256=anchor_manifests[outer]["sha256_float64_c_order"],
                        protocol=method,
                        outer_seed=outer,
                        draw=draw,
                        run_role=run_role,
                        level=level,
                    ),
                    "protocol": method,
                    "outer_seed": outer,
                    "draw": draw,
                    "run_role": run_role,
                    "private_lambda": lx,
                    "anchor_lambda": la,
                    "lambda": level,
                    "model_seed": manifest.model_seed(
                        outer, model_draw, model_family
                    ),
                })
    expected_full_fits = len(cfg.outer_seeds) * (
        3 + 3 * cfg.draws_per_seed + 6
    )
    if cfg.mode == "full" and len(planned_runs) != expected_full_fits:
        raise RuntimeError(
            f"Full manifest contains {len(planned_runs)} fits; expected "
            f"{expected_full_fits}"
        )
    run_manifest_path = out / ("full_run_manifest.json" if cfg.mode == "full" else "run_manifest.json")
    run_manifest_payload = {
        "schema_version": 3,
        "config_hash": config_hash,
        "anchor_design": cfg.anchor_design,
        "anchor_noise_parameterization": cfg.anchor_noise_parameterization,
        "anchors": {str(key): value for key, value in anchor_manifests.items()},
        "projection": data.reducer.manifest(),
        "allocation": {
            "noiseless_fits": 3 * len(cfg.outer_seeds) if cfg.mode == "full" else None,
            "private_random_paired_fits": 2 * len(cfg.outer_seeds) * cfg.draws_per_seed if cfg.mode == "full" else None,
            "pa_random_fits": len(cfg.outer_seeds) * cfg.draws_per_seed if cfg.mode == "full" else None,
            "endpoint_fits": 6 * len(cfg.outer_seeds) if cfg.mode == "full" else 0,
        },
        "planned_runs": planned_runs,
        "planned_run_count": len(planned_runs),
        "noise_manifest_hash": manifest.to_dict()["manifest_hash"],
    }
    _write_or_validate_frozen_json(
        run_manifest_path, run_manifest_payload, label="full run schedule manifest"
    )

    runtime_environment = {
        "device": str(mr.resolve_device(cfg.device)),
        "gpu_name": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        ),
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
    }
    runtime_session_id = mr.stable_hash(
        {"started_unix": time.time(), **runtime_environment}, 16
    )
    mr._atomic_write_json(
        out / "execution_sessions" / f"{runtime_session_id}.json",
        {"runtime_session_id": runtime_session_id, **runtime_environment},
    )
    started = time.perf_counter()
    for spec in schedule:
        outer = int(spec["outer_seed"])
        draw = int(spec["draw"])
        run_role = str(spec["run_role"])
        lx = float(spec["private_lambda"])
        la = float(spec["anchor_lambda"])
        anchor = anchors[outer]
        protocols, blocks, structural, sigma, v = _protocol_cell(
            cfg,
            data,
            manifest,
            anchor,
            outer,
            draw,
            lx,
            la,
            run_role,
            spec["methods"],
        )
        for method in spec["methods"]:
            if method in cfg.protocols:
                level = la if method == "pa_i_gdp" else (lx if method.endswith("_an") else 0.0)
                run_id = mr.make_run_id(
                    config=config_hash,
                    anchor_sha256=anchor_manifests[outer]["sha256_float64_c_order"],
                    protocol=method,
                    outer_seed=outer,
                    draw=draw,
                    run_role=run_role,
                    level=level,
                )
                if store.contains(run_id):
                    continue
                pair_family = "c_aa_an" if method in {"c_gdp_an", "aa_i_gdp_an"} else ("c_aa" if method in {"c_gdp", "aa_i_gdp"} else method)
                seed = manifest.model_seed(
                    outer,
                    0 if cfg.mode == "full" else draw,
                    "fixed_noise_sensitivity_model"
                    if cfg.mode == "full" else pair_family,
                )
                t0 = time.perf_counter()
                utility = _evaluate_utility(cfg, protocols[method], labels, run_id, out, seed)
                preview_path = None
                if cfg.mode == "smoke" or (
                    outer == cfg.outer_seeds[0]
                    and (
                        (
                            run_role == "random"
                            and draw == 0
                            and method in {"c_gdp", "i_gdp", "aa_i_gdp"}
                        )
                        or run_role.startswith("endpoint_")
                    )
                ):
                    preview_path = (
                        out / "previews" /
                        f"{method}_seed{outer}_{run_role}_draw{draw}.png"
                    )
                attacks, best, evaluation_split = _attack_results(
                    cfg, method, protocols[method], data, blocks, auditor, anchor,
                    mr.named_seed(cfg.seed, "attack_evaluation"),
                    preview_path,
                    frozen_linkage_manifest,
                ) if cfg.run_attacks else ([], None, "none")
                p_result = protocols[method]["train"]
                record = {
                    "run_id": run_id, "protocol": method, "protocol_name": p_result.name,
                    "seed": outer, "outer_seed": outer, "draw": draw,
                    "run_role": run_role,
                    "pair_id": f"{run_role}:{outer}:{draw}", "cnn_side": cfg.cnn_side,
                    "cnn_minimum_input_side": cfg.cnn_minimum_input_side,
                    "h": cfg.reduced_dim,
                    "anchor": anchor_manifests[outer],
                    "lambda": level, "private_lambda": lx, "anchor_lambda": la,
                    "noise_sigma": sigma if method.endswith("_an") else 0.0,
                    "anchor_noise_sigma": v if method == "pa_i_gdp" else 0.0,
                    "noise_bin": min(4, int(5 * level / max(private_upper if method != 'pa_i_gdp' else anchor_upper, 1e-15))),
                    "utility": utility, "attacks": attacks, "best_attack": best,
                    "evaluation_split": evaluation_split,
                    "structural": structural if method in {"c_gdp", "aa_i_gdp", "c_gdp_an", "aa_i_gdp_an"} else None,
                    "i_geometry_leakage": (
                        _i_geometry_leakage(
                            blocks[evaluation_split][0][target_participant],
                            protocols[method][evaluation_split].private_uploads[target_participant],
                        )
                        if method == "i_gdp" and evaluation_split in blocks else None
                    ),
                    "gpm": _diag(p_result, cfg.anchor_rows),
                    "alignment_theory": (
                        _pa_alignment_theory_diagnostics(cfg, p_result, v)
                        if method == "pa_i_gdp" else None
                    ),
                    "communication_bytes": sum(x.numel() * x.element_size() for x in p_result.private_uploads)
                    + sum(x.numel() * x.element_size() for x in (p_result.anchor_uploads or [])),
                    "runtime_seconds": time.perf_counter() - t0,
                    "runtime_session_id": runtime_session_id,
                    "runtime_environment": runtime_environment,
                }
                store.append(record)
                print(
                    f"completed {method} seed={outer} role={run_role} "
                    f"draw={draw} lambda={level:.5g}"
                )
    context = {
        "config": cfg.to_dict(), "data_manifest": data.manifest,
        "auditor": auditor_meta, "noise_manifest": manifest.to_dict(),
        "effective_private_lambda_upper": private_upper,
        "effective_anchor_lambda_upper": anchor_upper,
        "grp_floor_control": grp_floor_control,
        "run_manifest_path": str(run_manifest_path),
        "runtime_session_id": runtime_session_id,
        "runtime_environment": runtime_environment,
        "elapsed_seconds": time.perf_counter() - started,
    }
    return render_report({"records": store.records, "context": context}, out)


def _render_pilot_report(payload: Mapping[str, Any], out: Path) -> dict[str, Any]:
    """Render calibration curves and selected-endpoint attack grids."""

    out.mkdir(parents=True, exist_ok=True)
    summary_path = out / "pilot" / "pilot_calibration_summary.json"
    report_path = out / "pilot" / "pilot_calibration_report.md"
    plot_path = out / "pilot" / "pilot_calibration.png"
    grid_path = out / "pilot" / "pilot_selected_attack_grids.png"
    summary = {
        "mode": "pilot",
        "status": payload.get("status", {}),
        "acceptance": payload.get("acceptance", {}),
        "private_evidence": payload.get("private_evidence", []),
        "anchor_evidence": payload.get("anchor_evidence", []),
        "private_aa_confirmation": payload.get("private_aa_confirmation", {}),
        "pilot_manifest_path": payload.get("pilot_manifest_path"),
        "pilot_manifest_sha256": payload.get("pilot_manifest_sha256"),
        "data_manifest": payload.get("data_manifest", {}),
        "anchor_design": (payload.get("config", {}) or {}).get("anchor_design"),
        "anchors": {
            str(record.get("outer_seed")): record.get("anchor")
            for record in payload.get("records", [])
            if record.get("anchor") is not None
        },
    }
    mr._atomic_write_json(summary_path, summary)

    lines = [
        "# MNIST GRP upper-noise-bound calibration",
        "",
        "The official MNIST test split was not loaded. The lower bounds remain fixed at zero.",
        "",
        f"- Private-data upper bound: {payload.get('status', {}).get('private_lambda_upper')}",
        f"- Private-data censor status: {payload.get('status', {}).get('private_censor_status')}",
        f"- Anchor-noise v upper bound: {payload.get('status', {}).get('anchor_lambda_upper')}",
        f"- Anchor censor status: {payload.get('status', {}).get('anchor_censor_status')}",
        f"- Coupled AA confirmation passed: {payload.get('acceptance', {}).get('private_aa_confirmation_passed')}",
        f"- Gaussian random projection: {payload.get('acceptance', {}).get('gaussian_projection_used')}",
        f"- All calibration gates passed: {payload.get('acceptance', {}).get('all_passed')}",
        "",
        "## Candidate evidence",
        "",
        "| Family | noise level (lambda_x or v) | balanced accuracy | retained utility | strongest 10-way top-1 | 95% CI | trigger |",
        "|---|---:|---:|---:|---:|---:|:---:|",
    ]
    for family, key in (("private", "private_evidence"), ("anchor", "anchor_evidence")):
        for item in payload.get(key, []):
            ci = item.get("linkage_bootstrap_ci_95", [math.nan, math.nan])
            lines.append(
                f"| {family} | {item.get('lambda', math.nan):.6g} | "
                f"{item.get('validation_balanced_accuracy', math.nan):.4f} | "
                f"{item.get('retained_utility', math.nan):.4f} | "
                f"{item.get('strongest_valid_embedding_linkage_top1', math.nan):.4f} | "
                f"[{ci[0]:.4f}, {ci[1]:.4f}] | {item.get('trigger')} |"
            )
    report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    try:
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(2, 2, figsize=(13, 8), squeeze=False)
        for row, (family, key, upper_key) in enumerate((
            ("Private-data noise (C-GDP-AN)", "private_evidence", "private_lambda_upper"),
            ("Anchor noise (PA-I-GDP)", "anchor_evidence", "anchor_lambda_upper"),
        )):
            evidence = list(payload.get(key, []))
            x = np.asarray([float(item["lambda"]) for item in evidence])
            utility = np.asarray([float(item["retained_utility"]) for item in evidence])
            linkage = np.asarray([
                float(item["strongest_valid_embedding_linkage_top1"]) for item in evidence
            ])
            ci = np.asarray([item["linkage_bootstrap_ci_95"] for item in evidence], dtype=float)
            upper = float(payload.get("status", {}).get(upper_key, math.nan))
            axes[row, 0].plot(x, utility, marker="o")
            axes[row, 0].axhline(0.15, color="tab:red", linestyle="--", label="utility trigger")
            axes[row, 1].plot(x, linkage, marker="o")
            axes[row, 1].fill_between(x, ci[:, 0], ci[:, 1], alpha=0.2)
            axes[row, 1].axhline(0.10, color="black", linestyle=":", label="10-way chance")
            axes[row, 1].axhline(0.12, color="tab:red", linestyle="--", label="practical threshold")
            if math.isfinite(upper):
                for col in range(2):
                    axes[row, col].axvline(upper, color="tab:green", linestyle="-.", label="selected upper")
            smallest = min((value for value in x if value > 0), default=2.0**-8)
            for col in range(2):
                axes[row, col].set_xscale("symlog", base=2, linthresh=smallest / 2)
                axes[row, col].set_xlabel("normalized noise endpoint lambda")
                axes[row, col].grid(alpha=0.25)
            axes[row, 0].set_ylabel("normalized retained utility")
            axes[row, 1].set_ylabel("strongest valid embedding 10-way top-1")
            axes[row, 0].set_title(f"{family}: utility")
            axes[row, 1].set_title(f"{family}: attack linkage")
            axes[row, 0].legend(fontsize=8)
            axes[row, 1].legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(plot_path, dpi=180, bbox_inches="tight")
        plt.close(fig)

        records = list(payload.get("records", []))
        selected_paths: list[tuple[str, str]] = []
        for family, upper_key in (("private", "private_lambda_upper"), ("anchor", "anchor_lambda_upper")):
            upper = float(payload.get("status", {}).get(upper_key, math.nan))
            candidates = [
                record for record in records
                if record.get("pilot_family") == family
                and math.isclose(float(record.get("lambda", math.nan)), upper)
                and (record.get("best_attack") or {}).get("preview_path")
            ]
            if candidates:
                candidates.sort(key=lambda record: int(record.get("outer_seed", 0)))
                selected_paths.append((
                    f"{family} selected lambda={upper:.6g}",
                    str(candidates[0]["best_attack"]["preview_path"]),
                ))
        if selected_paths:
            fig, axes = plt.subplots(
                len(selected_paths), 1,
                figsize=(18, 7.5 * len(selected_paths)),
                squeeze=False,
            )
            for row, (title, path) in enumerate(selected_paths):
                axes[row, 0].imshow(plt.imread(path))
                axes[row, 0].set_title(title)
                axes[row, 0].axis("off")
            fig.tight_layout()
            fig.savefig(grid_path, dpi=140, bbox_inches="tight")
            plt.close(fig)
    except Exception as exc:
        summary["plot_warning"] = repr(exc)
        mr._atomic_write_json(summary_path, summary)

    return {
        "output_dir": str(out),
        "summary_path": str(summary_path),
        "report_path": str(report_path),
        "plot_path": str(plot_path),
        "reconstruction_grid": str(grid_path),
        **summary,
    }


def _nested_finite_float(record: Mapping[str, Any], dotted_key: str) -> float | None:
    """Read a finite scalar from a nested result record."""

    value: Any = record
    for part in dotted_key.split("."):
        if not isinstance(value, Mapping) or part not in value:
            return None
        value = value[part]
    try:
        scalar = float(value)
    except (TypeError, ValueError):
        return None
    return scalar if math.isfinite(scalar) else None


def _fixed_cubic_spline_design(normalized_x: np.ndarray) -> np.ndarray:
    """Fixed nine-column cubic regression-spline basis on ``[0, 1]``.

    Five equally spaced interior knots are fixed before seeing any response.
    The truncated-power construction is piecewise cubic and has continuous
    first and second derivatives at every knot.  Keeping this basis fixed in
    every bootstrap resample prevents response-dependent model selection.
    """

    x = np.asarray(normalized_x, dtype=np.float64).reshape(-1)
    interior_knots = np.linspace(0.0, 1.0, 7, dtype=np.float64)[1:-1]
    columns = [np.ones_like(x), x, np.square(x), x**3]
    columns.extend(np.maximum(x - knot, 0.0) ** 3 for knot in interior_knots)
    return np.column_stack(columns)


def _conditional_curve_fit(
    x: np.ndarray,
    y: np.ndarray,
    *,
    upper: float,
    scale: str,
    n_resamples: int,
    seed: int,
    grid_size: int = 201,
) -> dict[str, Any]:
    """Fit one fixed cubic spline and inner-draw percentile bootstrap band.

    Only the random inner draws enter the fit and bootstrap.  The deterministic
    zero/upper endpoint controls remain available as raw points, but do not
    receive sampling weight.  For positive solver diagnostics, fitting on the
    log scale avoids impossible negative fitted residuals or condition numbers.
    """

    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    valid = np.isfinite(x) & np.isfinite(y) & (x >= 0.0) & (x <= upper)
    if scale == "log":
        valid &= y > 0.0
    x, y = x[valid], y[valid]
    if len(x) < 8 or len(np.unique(x)) < 4:
        raise ValueError(
            "a conditional spline requires at least eight finite draws at "
            "four distinct noise levels"
        )

    normalized_x = np.clip(x / upper, 0.0, 1.0)
    grid_normalized = np.linspace(0.0, 1.0, grid_size, dtype=np.float64)
    grid_lambda = grid_normalized * upper
    design = _fixed_cubic_spline_design(normalized_x)
    grid_design = _fixed_cubic_spline_design(grid_normalized)
    transformed_y = np.log(y) if scale == "log" else y

    gram = design.T @ design
    # The small, prespecified ridge is numerical regularization, not a tuned
    # smoothing parameter.  The intercept is intentionally left unpenalized.
    ridge = 1.0e-8 * max(float(np.trace(gram) / design.shape[1]), 1.0)
    penalty = np.eye(design.shape[1], dtype=np.float64) * ridge
    penalty[0, 0] = 0.0
    coefficients = np.linalg.solve(gram + penalty, design.T @ transformed_y)
    estimate_transformed = grid_design @ coefficients

    rng = np.random.default_rng(int(seed))
    n_resamples = int(n_resamples)
    if n_resamples < 1:
        raise ValueError("n_resamples must be positive")
    bootstrap_predictions = np.empty(
        (n_resamples, grid_size), dtype=np.float64
    )
    probability = np.full(len(x), 1.0 / len(x), dtype=np.float64)
    chunk_size = 256
    for start in range(0, n_resamples, chunk_size):
        stop = min(start + chunk_size, n_resamples)
        counts = rng.multinomial(
            len(x), probability, size=stop - start
        ).astype(np.float64, copy=False)
        boot_gram = np.einsum(
            "bn,np,nq->bpq", counts, design, design, optimize=True
        )
        boot_rhs = np.einsum(
            "bn,np,n->bp", counts, design, transformed_y, optimize=True
        )
        boot_coefficients = np.linalg.solve(
            boot_gram + penalty[None, :, :], boot_rhs[..., None]
        )[..., 0]
        bootstrap_predictions[start:stop] = boot_coefficients @ grid_design.T

    lower_transformed, upper_transformed = np.quantile(
        bootstrap_predictions, [0.025, 0.975], axis=0
    )
    if scale == "log":
        estimate = np.exp(estimate_transformed)
        lower = np.exp(lower_transformed)
        upper_band = np.exp(upper_transformed)
    else:
        estimate = estimate_transformed
        lower = lower_transformed
        upper_band = upper_transformed
    return {
        "lambda": grid_lambda,
        "normalized_lambda": grid_normalized,
        "estimate": estimate,
        "ci_lower": lower,
        "ci_upper": upper_band,
        "n_inner_draws": int(len(x)),
        "bootstrap_resamples": n_resamples,
        "ridge": ridge,
        "response_scale": scale,
    }


def _write_rows_csv(path: Path, rows: Sequence[Mapping[str, Any]], fields: Sequence[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(fields), extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def _five_bin_summary(raw_points: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Return the prespecified five equal-width conditional-noise summaries."""

    grouped: dict[tuple[str, str, str, str], list[float]] = {}
    for row in raw_points:
        if not row.get("included_in_spline") or row.get("value") is None:
            continue
        normalized = float(row["normalized_lambda"])
        bin_index = min(4, max(0, int(5.0 * normalized)))
        key = (
            str(row["curve_kind"]), str(row["method"]),
            str(row.get("comparison", "")), str(row["metric"]),
        )
        grouped.setdefault((*key, str(bin_index)), []).append(float(row["value"]))

    summaries: list[dict[str, Any]] = []
    for key, values in sorted(grouped.items()):
        curve_kind, method, comparison, metric, bin_text = key
        bin_index = int(bin_text)
        array = np.asarray(values, dtype=np.float64)
        summaries.append({
            "curve_kind": curve_kind,
            "method": method,
            "comparison": comparison,
            "metric": metric,
            "bin": bin_index,
            "normalized_lambda_lower": bin_index / 5.0,
            "normalized_lambda_upper": (bin_index + 1) / 5.0,
            "n": int(len(array)),
            "mean": float(np.mean(array)),
            "standard_deviation": (
                float(np.std(array, ddof=1)) if len(array) > 1 else 0.0
            ),
            "median": float(np.median(array)),
            "quartile_25": float(np.quantile(array, 0.25)),
            "quartile_75": float(np.quantile(array, 0.75)),
        })
    return summaries


def _plot_conditional_family(
    curve_rows: Sequence[Mapping[str, Any]],
    raw_points: Sequence[Mapping[str, Any]],
    *,
    metrics: Sequence[str],
    metric_meta: Mapping[str, Mapping[str, str]],
    path: Path,
    curve_kind: str,
) -> bool:
    import matplotlib.pyplot as plt

    available = [
        metric for metric in metrics
        if any(
            row["curve_kind"] == curve_kind and row["metric"] == metric
            for row in curve_rows
        )
    ]
    if not available:
        return False
    fig, axes = plt.subplots(
        1, len(available), figsize=(5.6 * len(available), 4.5), squeeze=False
    )
    colors = {
        "c_gdp_an": "#1f77b4",
        "aa_i_gdp_an": "#ff7f0e",
        "pa_i_gdp": "#2ca02c",
        "c_gdp_an_minus_aa_i_gdp_an": "#7f3c8d",
    }
    for column, metric in enumerate(available):
        axis = axes[0, column]
        methods = sorted({
            str(row["method"]) for row in curve_rows
            if row["curve_kind"] == curve_kind and row["metric"] == metric
        })
        for method in methods:
            curve = sorted(
                (
                    row for row in curve_rows
                    if row["curve_kind"] == curve_kind
                    and row["metric"] == metric
                    and row["method"] == method
                ),
                key=lambda row: float(row["lambda"]),
            )
            raw = [
                row for row in raw_points
                if row["curve_kind"] == curve_kind
                and row["metric"] == metric
                and row["method"] == method
                and row.get("value") is not None
            ]
            color = colors.get(method)
            random_raw = [row for row in raw if row.get("included_in_spline")]
            endpoint_raw = [row for row in raw if not row.get("included_in_spline")]
            axis.scatter(
                [row["lambda"] for row in random_raw],
                [row["value"] for row in random_raw],
                color=color, alpha=0.22, s=15, linewidths=0,
            )
            if endpoint_raw:
                axis.scatter(
                    [row["lambda"] for row in endpoint_raw],
                    [row["value"] for row in endpoint_raw],
                    color=color, marker="X", s=48, linewidths=0.8,
                )
            axis.plot(
                [row["lambda"] for row in curve],
                [row["estimate"] for row in curve],
                color=color, linewidth=2.0, label=method,
            )
            axis.fill_between(
                [row["lambda"] for row in curve],
                [row["ci_lower"] for row in curve],
                [row["ci_upper"] for row in curve],
                color=color, alpha=0.16, linewidth=0,
            )
        if "linkage_" in metric:
            axis.axhline(0.10, color="black", linestyle="--", linewidth=1.0, alpha=0.7)
        if curve_kind == "paired_difference":
            axis.axhline(0.0, color="black", linestyle="--", linewidth=1.0, alpha=0.7)
        axis.set_xlabel("normalized noise level λ")
        axis.set_ylabel(metric_meta[metric]["label"])
        axis.set_title(metric_meta[metric]["label"])
        if metric_meta[metric]["scale"] == "log":
            axis.set_yscale("log")
        axis.grid(alpha=0.18)
        if column == 0:
            axis.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return True


def _render_conditional_noise_response(
    records: Sequence[Mapping[str, Any]],
    context: Mapping[str, Any],
    out: Path,
) -> dict[str, Any]:
    """Persist one-deployment spline curves and inner-draw bootstrap bands."""

    config = context.get("config", {}) if isinstance(context, Mapping) else {}
    n_resamples = int(config.get("bootstrap_resamples", 10_000))
    base_seed = int(config.get("seed", 0))
    metric_meta = {item["key"]: item for item in _CONDITIONAL_METRICS}
    noisy_methods = ("c_gdp_an", "aa_i_gdp_an", "pa_i_gdp")
    upper_by_method = {
        "c_gdp_an": float(context.get("effective_private_lambda_upper", math.nan)),
        "aa_i_gdp_an": float(context.get("effective_private_lambda_upper", math.nan)),
        "pa_i_gdp": float(context.get("effective_anchor_lambda_upper", math.nan)),
    }
    for method in noisy_methods:
        if not math.isfinite(upper_by_method[method]) or upper_by_method[method] <= 0.0:
            available = [
                float(row.get("lambda", math.nan)) for row in records
                if row.get("protocol") == method
                and math.isfinite(float(row.get("lambda", math.nan)))
            ]
            upper_by_method[method] = max(available, default=math.nan)

    raw_points: list[dict[str, Any]] = []
    curve_rows: list[dict[str, Any]] = []
    curve_models: list[dict[str, Any]] = []
    warnings: list[str] = []

    for method in noisy_methods:
        method_rows = [row for row in records if row.get("protocol") == method]
        upper = upper_by_method[method]
        if not method_rows or not math.isfinite(upper) or upper <= 0.0:
            warnings.append(f"{method}: missing a positive calibrated upper bound")
            continue
        metric_items = [
            item for item in _CONDITIONAL_METRICS
            if item["family"] != "gpm" or method == "pa_i_gdp"
        ]
        for item in metric_items:
            metric = item["key"]
            for row in method_rows:
                role = str(row.get("run_role", ""))
                if role not in {"random", "endpoint_zero", "endpoint_upper"}:
                    continue
                level = float(row.get("lambda", math.nan))
                value = _nested_finite_float(row, metric)
                raw_points.append({
                    "curve_kind": "absolute",
                    "method": method,
                    "comparison": "",
                    "metric": metric,
                    "outer_seed": int(row.get("outer_seed", 0)),
                    "draw": int(row.get("draw", 0)),
                    "run_role": role,
                    "lambda": level,
                    "normalized_lambda": level / upper,
                    "value": value,
                    "included_in_spline": role == "random" and value is not None,
                })
            fit_points = [
                row for row in raw_points
                if row["curve_kind"] == "absolute"
                and row["method"] == method
                and row["metric"] == metric
                and row["included_in_spline"]
            ]
            if len(fit_points) < 8:
                warnings.append(
                    f"{method}/{metric}: only {len(fit_points)} finite random draws; curve omitted"
                )
                continue
            try:
                fitted = _conditional_curve_fit(
                    np.asarray([row["lambda"] for row in fit_points]),
                    np.asarray([row["value"] for row in fit_points]),
                    upper=upper,
                    scale=item["scale"],
                    n_resamples=n_resamples,
                    seed=mr.named_seed(base_seed, "conditional_curve", method, metric),
                )
            except Exception as exc:
                warnings.append(f"{method}/{metric}: {exc!r}")
                continue
            curve_models.append({
                "curve_kind": "absolute",
                "method": method,
                "comparison": "",
                "metric": metric,
                "upper_bound": upper,
                "n_inner_draws": fitted["n_inner_draws"],
                "bootstrap_resamples": fitted["bootstrap_resamples"],
                "response_scale": fitted["response_scale"],
                "ridge": fitted["ridge"],
            })
            for index in range(len(fitted["lambda"])):
                curve_rows.append({
                    "curve_kind": "absolute",
                    "method": method,
                    "comparison": "",
                    "metric": metric,
                    "lambda": float(fitted["lambda"][index]),
                    "normalized_lambda": float(fitted["normalized_lambda"][index]),
                    "estimate": float(fitted["estimate"][index]),
                    "ci_lower": float(fitted["ci_lower"][index]),
                    "ci_upper": float(fitted["ci_upper"][index]),
                    "n_inner_draws": fitted["n_inner_draws"],
                    "bootstrap_resamples": fitted["bootstrap_resamples"],
                    "response_scale": fitted["response_scale"],
                    "ridge": fitted["ridge"],
                })

    # C-GDP-AN and AA-I-GDP-AN share draw IDs and noise levels.  Resampling
    # their within-draw difference preserves that pairing exactly.
    c_rows = {
        (int(row.get("outer_seed", 0)), int(row.get("draw", 0)), str(row.get("run_role", ""))): row
        for row in records if row.get("protocol") == "c_gdp_an"
    }
    aa_rows = {
        (int(row.get("outer_seed", 0)), int(row.get("draw", 0)), str(row.get("run_role", ""))): row
        for row in records if row.get("protocol") == "aa_i_gdp_an"
    }
    paired_upper = upper_by_method["c_gdp_an"]
    if math.isfinite(paired_upper) and paired_upper > 0.0:
        for item in (entry for entry in _CONDITIONAL_METRICS if entry["family"] != "gpm"):
            metric = item["key"]
            for key in sorted(set(c_rows).intersection(aa_rows)):
                c_row, aa_row = c_rows[key], aa_rows[key]
                role = key[2]
                if role not in {"random", "endpoint_zero", "endpoint_upper"}:
                    continue
                c_level = float(c_row.get("lambda", math.nan))
                aa_level = float(aa_row.get("lambda", math.nan))
                if not math.isclose(c_level, aa_level, rel_tol=0.0, abs_tol=1.0e-12):
                    warnings.append(
                        f"unpaired lambda for C/AA draw {key[1]}: {c_level} versus {aa_level}"
                    )
                    continue
                c_value = _nested_finite_float(c_row, metric)
                aa_value = _nested_finite_float(aa_row, metric)
                difference = (
                    c_value - aa_value
                    if c_value is not None and aa_value is not None else None
                )
                raw_points.append({
                    "curve_kind": "paired_difference",
                    "method": "c_gdp_an_minus_aa_i_gdp_an",
                    "comparison": "C-GDP-AN minus AA-I-GDP-AN",
                    "metric": metric,
                    "outer_seed": key[0],
                    "draw": key[1],
                    "run_role": role,
                    "lambda": c_level,
                    "normalized_lambda": c_level / paired_upper,
                    "value": difference,
                    "included_in_spline": role == "random" and difference is not None,
                })
            fit_points = [
                row for row in raw_points
                if row["curve_kind"] == "paired_difference"
                and row["metric"] == metric
                and row["included_in_spline"]
            ]
            if len(fit_points) < 8:
                continue
            try:
                fitted = _conditional_curve_fit(
                    np.asarray([row["lambda"] for row in fit_points]),
                    np.asarray([row["value"] for row in fit_points]),
                    upper=paired_upper,
                    scale="linear",
                    n_resamples=n_resamples,
                    seed=mr.named_seed(base_seed, "paired_conditional_curve", metric),
                )
            except Exception as exc:
                warnings.append(f"paired C/AA {metric}: {exc!r}")
                continue
            curve_models.append({
                "curve_kind": "paired_difference",
                "method": "c_gdp_an_minus_aa_i_gdp_an",
                "comparison": "C-GDP-AN minus AA-I-GDP-AN",
                "metric": metric,
                "upper_bound": paired_upper,
                "n_inner_draws": fitted["n_inner_draws"],
                "bootstrap_resamples": fitted["bootstrap_resamples"],
                "response_scale": "linear",
                "ridge": fitted["ridge"],
            })
            for index in range(len(fitted["lambda"])):
                curve_rows.append({
                    "curve_kind": "paired_difference",
                    "method": "c_gdp_an_minus_aa_i_gdp_an",
                    "comparison": "C-GDP-AN minus AA-I-GDP-AN",
                    "metric": metric,
                    "lambda": float(fitted["lambda"][index]),
                    "normalized_lambda": float(fitted["normalized_lambda"][index]),
                    "estimate": float(fitted["estimate"][index]),
                    "ci_lower": float(fitted["ci_lower"][index]),
                    "ci_upper": float(fitted["ci_upper"][index]),
                    "n_inner_draws": fitted["n_inner_draws"],
                    "bootstrap_resamples": fitted["bootstrap_resamples"],
                    "response_scale": "linear",
                    "ridge": fitted["ridge"],
                })

    bin_rows = _five_bin_summary(raw_points)
    raw_path = out / "mnist_conditional_raw_points.csv"
    curve_path = out / "mnist_conditional_curves.csv"
    curve_json_path = out / "mnist_conditional_curves.json"
    bin_path = out / "mnist_conditional_five_bin_summary.csv"
    bin_json_path = out / "mnist_conditional_five_bin_summary.json"
    _write_rows_csv(raw_path, raw_points, (
        "curve_kind", "method", "comparison", "metric", "outer_seed", "draw",
        "run_role", "lambda", "normalized_lambda", "value", "included_in_spline",
    ))
    _write_rows_csv(curve_path, curve_rows, (
        "curve_kind", "method", "comparison", "metric", "lambda",
        "normalized_lambda", "estimate", "ci_lower", "ci_upper",
        "n_inner_draws", "bootstrap_resamples", "response_scale", "ridge",
    ))
    _write_rows_csv(bin_path, bin_rows, (
        "curve_kind", "method", "comparison", "metric", "bin",
        "normalized_lambda_lower", "normalized_lambda_upper", "n", "mean",
        "standard_deviation", "median", "quartile_25", "quartile_75",
    ))
    curve_payload = {
        "schema_version": 1,
        "inference_scope": (
            "conditional on one fixed deployment; the 95% bands resample the "
            "100 inner noise draws and do not quantify deployment-level uncertainty"
        ),
        "spline": {
            "degree": 3,
            "basis": "truncated-power cubic regression spline",
            "interior_knots_normalized": np.linspace(0.0, 1.0, 7)[1:-1].tolist(),
            "coefficient_count": 9,
            "ridge_rule": "1e-8 * max(trace(X'X)/9, 1); intercept unpenalized",
            "grid_points": 201,
            "endpoint_controls_in_fit": False,
        },
        "bootstrap": {
            "unit": "inner noise draw; paired draw for C-GDP-AN minus AA-I-GDP-AN",
            "method": "ordinary nonparametric percentile bootstrap",
            "confidence_level": 0.95,
            "resamples": n_resamples,
            "seed": base_seed,
        },
        "curve_models": curve_models,
        "curve_rows": curve_rows,
        "raw_points": raw_points,
        "warnings": warnings,
    }
    mr._atomic_write_json(curve_json_path, curve_payload)
    mr._atomic_write_json(bin_json_path, {"schema_version": 1, "rows": bin_rows})

    figures: dict[str, str] = {}
    figure_specs = {
        "utility": (
            [item["key"] for item in _CONDITIONAL_METRICS if item["family"] == "utility"],
            out / "mnist_conditional_utility_curves.png",
            "absolute",
        ),
        "privacy": (
            [item["key"] for item in _CONDITIONAL_METRICS if item["family"] == "privacy"],
            out / "mnist_conditional_privacy_curves.png",
            "absolute",
        ),
        "gpm": (
            [item["key"] for item in _CONDITIONAL_METRICS if item["family"] == "gpm"],
            out / "mnist_conditional_gpm_curves.png",
            "absolute",
        ),
        "paired_c_aa": (
            [
                "utility.test_balanced_accuracy",
                "best_attack.metrics.linkage_embedding_top1",
                "best_attack.metrics.pixel_nrmse",
            ],
            out / "mnist_conditional_c_aa_paired_differences.png",
            "paired_difference",
        ),
    }
    try:
        for name, (metrics, path, curve_kind) in figure_specs.items():
            if _plot_conditional_family(
                curve_rows, raw_points, metrics=metrics, metric_meta=metric_meta,
                path=path, curve_kind=curve_kind,
            ):
                figures[name] = str(path)
    except Exception as exc:
        warnings.append(f"conditional figure rendering: {exc!r}")
        curve_payload["warnings"] = warnings
        mr._atomic_write_json(curve_json_path, curve_payload)

    return {
        "status": "computed" if curve_rows else "insufficient_data",
        "inference_scope": curve_payload["inference_scope"],
        "inner_draw_bootstrap_resamples": n_resamples,
        "curve_model_count": len(curve_models),
        "curve_csv": str(curve_path),
        "curve_json": str(curve_json_path),
        "raw_points_csv": str(raw_path),
        "five_bin_summary_csv": str(bin_path),
        "five_bin_summary_json": str(bin_json_path),
        "figures": figures,
        "warnings": warnings,
    }


def render_report(results: Mapping[str, Any] | Sequence[Mapping[str, Any]] | str | Path, output_dir: str | Path | None = None):
    """Write summary JSON/Markdown and a compact PNG dashboard."""

    if isinstance(results, Mapping) and results.get("mode") == "pilot":
        pilot_out = Path(output_dir or results.get("config", {}).get("output_dir", "."))
        return _render_pilot_report(results, pilot_out)

    if isinstance(results, (str, Path)):
        path = Path(results)
        payload = json.loads(path.read_text(encoding="utf-8"))
        out = Path(output_dir or path.parent)
    elif isinstance(results, Mapping):
        payload = dict(results)
        out = Path(output_dir or payload.get("context", {}).get("config", {}).get("output_dir", "."))
    else:
        payload, out = {"records": list(results)}, Path(output_dir or ".")
    out.mkdir(parents=True, exist_ok=True)
    records = list(payload.get("records", []))
    methods = sorted({r.get("protocol") for r in records})
    unique_outer_seeds = {r.get("outer_seed") for r in records}
    structural_values = [r.get("structural", {}).get("passed") for r in records if r.get("structural")]
    acceptance = {
        "all_six_methods_completed": set(METHODS).issubset(methods),
        "structural_coupling_passed": bool(structural_values) and all(structural_values),
        "statistical_equivalence_status": (
            "not applicable: one fixed deployment; report structural equality and paired noise-response differences"
            if len(unique_outer_seeds) == 1
            else "requires_at_least_two_paired_outer_seeds"
        ),
    }
    acceptance["all_passed"] = bool(
        acceptance["all_six_methods_completed"]
        and acceptance["structural_coupling_passed"]
    )
    summary = {"schema_version": 1, "records": records, "methods": methods, "acceptance": acceptance, "context": payload.get("context", {})}
    if len(unique_outer_seeds) >= 2:
        margins = payload.get("context", {}).get("config", {}).get("equivalence_margins", {})
        clean_records = [
            row for row in records
            if row.get("run_role") in {"random", "smoke"}
            and int(row.get("draw", 0)) == 0
        ]
        private_random_records = [
            row for row in records
            if row.get("run_role") in {"random", "smoke"}
        ]
        summary["equivalence"] = {
            "clean": mr.evaluate_equivalence_cells(clean_records, "c_gdp", "aa_i_gdp", margins, group_keys=("cnn_side",), seed_key="pair_id"),
            "noisy": mr.evaluate_equivalence_cells(private_random_records, "c_gdp_an", "aa_i_gdp_an", margins, group_keys=("cnn_side", "noise_bin"), seed_key="pair_id"),
        }
        acceptance["statistical_equivalence_status"] = (
            "computed on the prespecified paired random levels; inspect every cell"
        )
    random_counts = {
        method: sum(
            row.get("protocol") == method and row.get("run_role") == "random"
            for row in records
        )
        for method in ("c_gdp_an", "aa_i_gdp_an", "pa_i_gdp")
    }
    if len(unique_outer_seeds) == 1 and max(random_counts.values(), default=0) >= 8:
        try:
            summary["conditional_noise_response"] = _render_conditional_noise_response(
                records, payload.get("context", {}), out
            )
        except Exception as exc:
            summary["conditional_noise_response"] = {
                "status": "failed",
                "error": repr(exc),
            }
    else:
        summary["conditional_noise_response"] = {
            "status": "not_applicable",
            "reason": (
                "requires one outer deployment and at least eight random inner "
                "draws for a noisy method"
            ),
            "random_draw_counts": random_counts,
        }
    mr._atomic_write_json(out / "mnist_summary.json", summary)
    lines = ["# MNIST I-GDP experiment report", "", f"Runs completed: {len(records)}", f"Methods: {', '.join(methods)}", "", "## Acceptance", ""]
    lines += [f"- {k}: {v}" for k, v in acceptance.items()]
    conditional = summary["conditional_noise_response"]
    lines += [
        "",
        "## Conditional noise response",
        "",
        f"- status: {conditional.get('status')}",
    ]
    if conditional.get("status") == "computed":
        lines += [
            f"- inference scope: {conditional.get('inference_scope')}",
            f"- inner-draw bootstrap resamples: {conditional.get('inner_draw_bootstrap_resamples')}",
            "- spline: fixed cubic regression spline with five equally spaced interior knots",
            "- endpoint controls: plotted and saved as raw points, but excluded from fitting and resampling",
            f"- curve data: {conditional.get('curve_csv')}",
            f"- raw points: {conditional.get('raw_points_csv')}",
            f"- five-bin summary: {conditional.get('five_bin_summary_csv')}",
        ]
    elif conditional.get("error"):
        lines.append(f"- error: {conditional.get('error')}")
    (out / "mnist_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
        for method in methods:
            rows = [r for r in records if r.get("protocol") == method]
            x = [r.get("lambda", 0.0) for r in rows]
            axes[0].scatter(x, [r.get("utility", {}).get("test_accuracy", np.nan) for r in rows], label=method, s=24)
            axes[1].scatter(x, [r.get("best_attack", {}).get("metrics", {}).get("pixel_nrmse", np.nan) if r.get("best_attack") else np.nan for r in rows], s=24)
        pa = [r for r in records if r.get("gpm")]
        axes[2].scatter([r["anchor_lambda"] for r in pa], [r["gpm"]["relative_gopp_residual"] for r in pa], s=28)
        axes[0].set(xlabel="normalized noise", ylabel="test accuracy", title="Utility")
        axes[1].set(xlabel="normalized noise", ylabel="best attack pixel NRMSE", title="Reconstruction")
        axes[2].set(xlabel="anchor noise", ylabel="relative GOPP residual", title="GPM diagnostic", yscale="log")
        axes[0].legend(fontsize=7)
        fig.tight_layout(); fig.savefig(out / "mnist_report.png", dpi=180); plt.close(fig)

        first_preview_by_method: dict[str, str] = {}
        for record in records:
            preview = (record.get("best_attack") or {}).get("preview_path")
            if preview and record.get("protocol") not in first_preview_by_method:
                first_preview_by_method[str(record.get("protocol"))] = str(preview)
        if first_preview_by_method:
            preview_items = sorted(first_preview_by_method.items())
            fig, axes = plt.subplots(
                len(preview_items), 1, figsize=(18, 5.3 * len(preview_items)), squeeze=False
            )
            for row, (method, path) in enumerate(preview_items):
                axes[row, 0].imshow(plt.imread(path))
                axes[row, 0].set_title(method)
                axes[row, 0].axis("off")
            fig.tight_layout()
            fig.savefig(out / "mnist_reconstruction_grid.png", dpi=130, bbox_inches="tight")
            plt.close(fig)
    except Exception as exc:
        summary["plot_warning"] = repr(exc)
        mr._atomic_write_json(out / "mnist_summary.json", summary)
    conditional = summary.get("conditional_noise_response", {})
    return {
        "output_dir": str(out),
        "summary_path": str(out / "mnist_summary.json"),
        "report_path": str(out / "mnist_report.md"),
        "plot_path": str(out / "mnist_report.png"),
        "reconstruction_grid": str(out / "mnist_reconstruction_grid.png"),
        "conditional_curve_csv": conditional.get("curve_csv"),
        "conditional_curve_json": conditional.get("curve_json"),
        "conditional_raw_points_csv": conditional.get("raw_points_csv"),
        "conditional_five_bin_summary_csv": conditional.get("five_bin_summary_csv"),
        "conditional_figures": conditional.get("figures", {}),
        "required_report_keys": [
            "summary_path", "report_path", "plot_path", "reconstruction_grid",
            "conditional_curve_csv", "conditional_curve_json",
            "conditional_raw_points_csv", "conditional_five_bin_summary_csv",
            "conditional_figures",
        ],
        **summary,
    }


__all__ = [
    "print_config",
    "run_experiment",
    "run_pilot_calibration",
    "render_report",
]


In [ ]:
# 4. Import implementation and capture hardware metadata
import hashlib
import importlib
import json
import platform
import time
from pathlib import Path
import torch

import core_protocols
import mnist_runner
import mnist_orchestrator
importlib.reload(core_protocols)
importlib.reload(mnist_runner)
importlib.reload(mnist_orchestrator)

MODULE_HASHES = {
    filename: hashlib.sha256(Path(filename).read_bytes()).hexdigest()
    for filename in ("core_protocols.py", "mnist_runner.py", "mnist_orchestrator.py")
}
required_core = ("run_self_tests",)
required_runner = ("make_config", "print_config", "run_experiment", "render_report")
required_orchestrator = (
    "print_config", "run_experiment", "run_pilot_calibration", "render_report"
)
missing_api = [f"core_protocols.{name}" for name in required_core
               if not hasattr(core_protocols, name)]
missing_api += [f"mnist_runner.{name}" for name in required_runner
                if not hasattr(mnist_runner, name)]
missing_api += [f"mnist_orchestrator.{name}" for name in required_orchestrator
                if not hasattr(mnist_orchestrator, name)]
if missing_api:
    raise AttributeError(f"Notebook implementation API is incomplete: {missing_api}")

# One reviewed orchestration path owns smoke, validation-only calibration, and
# the full run so that every mode persists the same attacks and diagnostics.
experiment_api = mnist_orchestrator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if RUN_MODE in {"pilot", "full"} and DEVICE != "cuda":
    raise RuntimeError("Pilot/full mode requires a Colab GPU runtime.")
try:
    import psutil
    ram_gib = psutil.virtual_memory().total / (1024 ** 3)
except ImportError:
    ram_gib = None
hardware = {
    "device": DEVICE,
    "torch": torch.__version__,
    "torchvision": __import__("torchvision").__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gib": (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        if torch.cuda.is_available() else None
    ),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "cpu": platform.processor(),
    "ram_gib": ram_gib,
    "module_sha256": MODULE_HASHES,
    "preferred_gpu": "A100" if PREFER_A100 else None,
}
hardware["preferred_gpu_available"] = bool(
    not PREFER_A100 or (hardware["gpu"] and "A100" in hardware["gpu"])
)
if PREFER_A100 and not hardware["preferred_gpu_available"]:
    message = (
        f"A100 was requested but this runtime supplied {hardware['gpu']!r}. "
        "You may continue and resume later, or reconnect after selecting a premium GPU."
    )
    if REQUIRE_A100:
        raise RuntimeError(message)
    print("WARNING:", message)
print(json.dumps(hardware, indent=2))
(OUTPUT_DIR / "execution_metadata.json").write_text(
    json.dumps(hardware, indent=2, sort_keys=True), encoding="utf-8"
)
session_dir = OUTPUT_DIR / "execution_sessions"
session_dir.mkdir(parents=True, exist_ok=True)
session_name = f"notebook_session_{int(time.time())}_{hardware['gpu'] or 'cpu'}.json"
(session_dir / session_name.replace(" ", "_")).write_text(
    json.dumps(hardware, indent=2, sort_keys=True), encoding="utf-8"
)


## 5. Algebraic acceptance tests

These tests run before downloading MNIST or training a model. A failure stops the notebook so a sign, centering, orientation, or attack error cannot silently enter the results.

In [ ]:
# Run deterministic linear-algebra and planted-solution tests
import math

self_test_report = core_protocols.run_self_tests(device=DEVICE)
rotation_cpu = torch.eye(4, dtype=torch.float64, device="cpu")
if torch.cuda.is_available():
    rotation_cuda = rotation_cpu.to("cuda")
    cpu_left_error = mnist_runner.relative_frobenius_error(
        rotation_cpu, rotation_cuda
    )
    cuda_left_error = mnist_runner.relative_frobenius_error(
        rotation_cuda, rotation_cpu
    )
    perturbed_cuda = rotation_cuda.clone()
    perturbed_cuda[0, 0] = 2.0
    nonzero_error = mnist_runner.relative_frobenius_error(
        rotation_cpu, perturbed_cuda
    )
    expected_nonzero_error = 1.0 / math.sqrt(7.0)
    mixed_device_passed = bool(
        cpu_left_error == 0.0
        and cuda_left_error == 0.0
        and abs(nonzero_error - expected_nonzero_error) <= 1.0e-12
    )
    self_test_report["mixed_device_frobenius"] = {
        "cpu_left_zero_error": cpu_left_error,
        "cuda_left_zero_error": cuda_left_error,
        "nonzero_error": nonzero_error,
        "expected_nonzero_error": expected_nonzero_error,
        "passed": mixed_device_passed,
    }
else:
    mixed_device_passed = True
    self_test_report["mixed_device_frobenius"] = {
        "passed": True,
        "status": "skipped_without_cuda",
    }
self_test_report["all_passed"] = bool(
    self_test_report.get("all_passed", False)
    and mixed_device_passed
)
print(json.dumps(self_test_report, indent=2, default=float))
(OUTPUT_DIR / "algebraic_self_tests.json").write_text(
    json.dumps(self_test_report, indent=2, default=float), encoding="utf-8"
)
assert self_test_report.get("all_passed", False), self_test_report


## 6. Freeze this run's configuration

`smoke` touches all six methods with reduced sample sizes. `pilot` uses validation data only to calibrate the two upper noise bounds. `full` consumes the frozen pilot manifest and writes restart-safe per-run artifacts.

In [ ]:
config = mnist_runner.make_config(RUN_MODE)
config["device"] = DEVICE
config["output_dir"] = str(OUTPUT_DIR)
assert config["cnn_side"] == 10
if RUN_MODE == "full":
    pilot_manifest = Path(PILOT_MANIFEST_PATH)
    if not pilot_manifest.is_file():
        raise FileNotFoundError(
            "Full mode requires the completed frozen pilot manifest: "
            f"{pilot_manifest}"
        )
    config["pilot_manifest_path"] = str(pilot_manifest)
    config["pilot_manifest_sha256"] = hashlib.sha256(
        pilot_manifest.read_bytes()
    ).hexdigest()
mnist_runner.print_config(config)


## 7. Execute

The runner checkpoints its manifest and result rows after each completed unit. Re-running the cell resumes compatible work rather than duplicating it.

In [ ]:
result = experiment_api.run_experiment(config=config, output_dir=OUTPUT_DIR)
print(json.dumps(result.get("status", {}), indent=2, default=str))


## 8. Report and visual checks

The generated report distinguishes the GRP reconstruction floor from protocol/attack error. Each grid has fixed digit columns and labeled rows for the original, GRP floor, and every recovery attack. Attack rows state embedding (E) and pixel (P) 10-way top-1 linkage; chance is 10%.

In [ ]:
report = (
    experiment_api.render_report(result, OUTPUT_DIR)
    if result.get("mode") == "pilot"
    else result
)
print("Report artifacts:")
for name in (
    "summary_path", "report_path", "plot_path", "reconstruction_grid",
    "json_path", "markdown_path",
):
    if name in report:
        print(f"  {name}: {report[name]}")

required_report_keys = tuple(result.get("required_report_keys", ()))
missing_reports = [
    key for key in required_report_keys
    if key not in report or not Path(report[key]).is_file()
    or Path(report[key]).stat().st_size == 0
]
if missing_reports:
    raise RuntimeError(f"Required report artifacts are missing/empty: {missing_reports}")

try:
    from IPython.display import display, Image
except ImportError:
    display = Image = None
if display is not None:
    if "summary_table" in result:
        display(result["summary_table"])
    for key in ("reconstruction_grid", "utility_plot", "privacy_utility_plot"):
        path = report.get(key)
        if path and Path(path).exists():
            display(Image(filename=str(path)))

acceptance = result.get("acceptance", {})
print("\nACCEPTANCE SUMMARY")
print(json.dumps(acceptance, indent=2, default=str))
if acceptance and not acceptance.get("all_passed", False):
    raise RuntimeError("Experiment acceptance checks did not all pass.")


## Interpretation order

1. Confirm all algebraic tests pass.
2. Confirm the smoke run contains every declared method, attack, and GPM diagnostic.
3. Inspect the labeled attack-by-query grids and clean-auditor/GRP-floor controls.
4. Inspect the validation-only pilot's selected upper-bound evidence before freezing the full manifest.
5. Only then run `full`. It uses one frozen outer deployment, 100 paired private-noise levels, 100 independent PA anchor-noise levels, and deterministic zero and upper endpoints.

The paper text is written only after these executed artifacts have been analyzed.
